# Phase 3 — Analyze
## 01 — Sales Analysis

### Objective
Analyze governed sales data to identify:

- Revenue and profit drivers
- Product and category performance
- Sales-volume patterns
- Margin performance
- Pareto 80/20 contribution
- ABC classification
- High-performing and weak-performing products
- Business insights required for later cross-functional analysis

> Data Source:
> Governed Phase 2 sales dataset. Raw Excel files are not used directly in Phase 3.

In [1]:
## Import libraries
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Libraries imported successfully.")

Libraries imported successfully.


In [4]:
## Load the governed sales dataset
PROJECT_ROOT = Path.cwd().parents[1]

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)
print("Processed data directory:", PROCESSED_DATA_DIR)

if PROCESSED_DATA_DIR.exists():

    files = [
        file.name
        for file in PROCESSED_DATA_DIR.iterdir()
        if file.is_file()
    ]

    print("Processed files found:")
    
    for file in sorted(files):
        print(" -", file)

else:
    print("Processed data directory does not exist.")

Project root: d:\STUDY\Data_Science_Courses\PROJECTS\12.Smart_AI-Retail_System\Smart_AI_Retail_System
Processed data directory: d:\STUDY\Data_Science_Courses\PROJECTS\12.Smart_AI-Retail_System\Smart_AI_Retail_System\data\processed
Processed files found:
 - dim_product.csv
 - fact_sales.csv
 - fact_stock.csv


In [8]:
# ============================================================
# LOAD GOVERNED SALES DATA
# ============================================================

SALES_PATH = PROCESSED_DATA_DIR / "fact_sales.csv"

if not SALES_PATH.exists():
    raise FileNotFoundError(f"fact_sales.csv not found at: {SALES_PATH}")

sales = pd.read_csv(SALES_PATH)

print("Governed sales dataset loaded successfully.")
print(f"Rows    : {sales.shape[0]:,}")
print(f"Columns : {sales.shape[1]}")
print("\nColumns:")
print(sales.columns.tolist())

Governed sales dataset loaded successfully.
Rows    : 966
Columns : 18

Columns:
['Sales_Record_ID', 'Product_ID', 'Product_Key', 'Source_Month', 'Category', 'Stock Code', 'Description', 'Record_Type', 'Level', 'Sold Period', 'Transaction_Status', 'Unit Cost', 'Unit Price', 'Cost Sales', 'Sales Value', 'Profit', 'Profit %', 'Cost_Sales_Reconciliation_Flag']


In [9]:
# ============================================================
# BASIC STRUCTURE CHECK
# ============================================================

display(sales.head())

print("\nDataset shape:")
print(sales.shape)

print("\nData types:")
display(
    sales.dtypes
         .rename("dtype")
         .to_frame()
)

,Sales_Record_ID,Product_ID,Product_Key,Source_Month,Category,Stock Code,Description,Record_Type,Level,Sold Period,Transaction_Status,Unit Cost,Unit Price,Cost Sales,Sales Value,Profit,Profit %,Cost_Sales_Reconciliation_Flag
0,1,1223,TLS169BOXE,Nov,ACCESSORIES,TLS169BOXE,Boxed Uni Floor Tool 30-38MM,PRODUCT,0,1,POSITIVE_SALES_ACTIVITY,9.31,29.99,9.31,24.99,15.68,62.75,MATCH
1,2,17,112.204,Nov,ACCESSORIES,112.204,TV Arial Lead 4.0m,PRODUCT,3,1,POSITIVE_SALES_ACTIVITY,1.28,2.99,1.28,2.49,1.21,48.59,MATCH
2,3,385,DLSC500,Nov,ACCESSORIES,DLSC500,Delonghi Descaler,PRODUCT,5,1,POSITIVE_SALES_ACTIVITY,7.77,14.49,7.77,7.50,-0.27,-3.60,MATCH
3,4,246,AF01,Nov,ACCESSORIES,AF01,VACUUM FRESHENERS AF101,PRODUCT,3,2,POSITIVE_SALES_ACTIVITY,2.30,3.49,4.60,3.32,-1.28,-38.55,MATCH
4,5,1076,SES007NEU0,Nov,ACCESSORIES,SES007NEU0,Sage Descaler (pack of 4),PRODUCT,6,2,POSITIVE_SALES_ACTIVITY,9.39,14.49,18.78,23.32,4.54,19.47,MATCH



Dataset shape:
(966, 18)

Data types:


,dtype
Sales_Record_ID,int64
Product_ID,int64
Product_Key,object
Source_Month,object
Category,object
Stock Code,object
Description,object
Record_Type,object
Level,int64
Sold Period,int64


In [10]:
# ============================================================
#  GOVERNED INPUT VALIDATION
# ============================================================

validation = pd.DataFrame({
    "dtype": sales.dtypes.astype(str),
    "null_count": sales.isna().sum(),
    "null_pct": (sales.isna().mean() * 100).round(2),
    "unique_values": sales.nunique(dropna=False)
})

display(validation)

,dtype,null_count,null_pct,unique_values
Sales_Record_ID,int64,0,0.00,966
Product_ID,int64,0,0.00,767
Product_Key,object,0,0.00,767
Source_Month,object,0,0.00,3
Category,object,0,0.00,88
Stock Code,object,0,0.00,768
Description,object,0,0.00,749
Record_Type,object,0,0.00,2
Level,int64,0,0.00,27
Sold Period,int64,0,0.00,13


In [11]:
print("Duplicate full rows:", sales.duplicated().sum())

Duplicate full rows: 0


### Step 1 — Validate analytical dimensions

In [12]:
# ============================================================
#  ANALYTICAL DIMENSION VALIDATION
# ============================================================

print("Source months:")
print(sales["Source_Month"].value_counts(dropna=False))

print("\nRecord types:")
print(sales["Record_Type"].value_counts(dropna=False))

print("\nTransaction statuses:")
print(sales["Transaction_Status"].value_counts(dropna=False))

print("\nCost Sales reconciliation flags:")
print(sales["Cost_Sales_Reconciliation_Flag"].value_counts(dropna=False))

Source months:
Source_Month
Nov    381
Dec    323
Jan    262
Name: count, dtype: int64

Record types:
Record_Type
PRODUCT           960
SERVICE_CHARGE      6
Name: count, dtype: int64

Transaction statuses:
Transaction_Status
POSITIVE_SALES_ACTIVITY           941
NEGATIVE_SALES_ACTIVITY            15
ZERO_UNIT_FINANCIAL_ADJUSTMENT     10
Name: count, dtype: int64

Cost Sales reconciliation flags:
Cost_Sales_Reconciliation_Flag
MATCH                931
SOURCE_ADJUSTMENT     35
Name: count, dtype: int64


In [13]:
# Category distribution

category_counts = (
    sales["Category"]
    .value_counts()
    .rename_axis("Category")
    .reset_index(name="Record_Count")
)

print(f"Total categories: {len(category_counts)}")

display(category_counts)

Total categories: 88


,Category,Record_Count
0,FOOD PREP,51
1,KETTLES,40
2,WASHING MACHINES,40
3,IT ACCESSORIES,32
4,CABLES,30
...,...,...
83,U/C FREEZER,1
84,BUILT UNDER DBL OVEN,1
85,COOLING FANS,1
86,TV,1


In [14]:
## Validate sales measures
sales_measures = [
    "Level",
    "Sold Period",
    "Unit Cost",
    "Unit Price",
    "Cost Sales",
    "Sales Value",
    "Profit",
    "Profit %"
]

display(
    sales[sales_measures]
    .describe()
    .T
)

,count,mean,std,min,25%,50%,75%,max
Level,966.00,1.08,4.42,-53.00,0.00,1.00,2.00,29.00
Sold Period,966.00,1.22,0.89,-5.00,1.00,1.00,1.00,12.00
Unit Cost,966.00,254.39,371.84,0.00,25.91,120.40,358.08,"3,774.00"
Unit Price,966.00,405.04,593.72,0.49,42.99,192.74,552.37,"5,804.99"
Cost Sales,966.00,283.30,431.07,-721.50,25.91,122.76,377.08,"3,774.00"
Sales Value,966.00,289.18,454.54,"-4,104.19",33.33,138.75,399.79,"4,000.00"
Profit,966.00,5.88,177.31,"-4,104.23",-5.00,6.51,27.84,815.90
Profit %,966.00,11.75,30.48,-201.82,-3.20,12.79,25.22,100.00


In [15]:
sign_check = pd.DataFrame({
    "Negative": [(sales[col] < 0).sum() for col in sales_measures],
    "Zero": [(sales[col] == 0).sum() for col in sales_measures],
    "Positive": [(sales[col] > 0).sum() for col in sales_measures]
}, index=sales_measures)

display(sign_check)


,Negative,Zero,Positive
Level,72,282,612
Sold Period,15,10,941
Unit Cost,0,16,950
Unit Price,0,0,966
Cost Sales,16,24,926
Sales Value,21,0,945
Profit,298,8,660
Profit %,281,8,677


In [16]:
# ============================================================
# DEFINE ANALYTICAL POPULATIONS
# ============================================================

# Governed source remains unchanged
sales_all = sales.copy()

# Product records only
sales_products = sales_all[
    sales_all["Record_Type"] == "PRODUCT"
].copy()

# Normal positive sales activity
sales_positive = sales_products[
    sales_products["Transaction_Status"] == "POSITIVE_SALES_ACTIVITY"
].copy()

# Negative activity / returns / reversals
sales_negative = sales_products[
    sales_products["Transaction_Status"] == "NEGATIVE_SALES_ACTIVITY"
].copy()

# Zero-unit financial adjustments
sales_adjustments = sales_products[
    sales_products["Transaction_Status"] == "ZERO_UNIT_FINANCIAL_ADJUSTMENT"
].copy()


print("ANALYTICAL POPULATIONS")
print("=" * 50)

print(f"All governed records       : {len(sales_all):,}")
print(f"Product records            : {len(sales_products):,}")
print(f"Positive sales activity    : {len(sales_positive):,}")
print(f"Negative sales activity    : {len(sales_negative):,}")
print(f"Zero-unit adjustments      : {len(sales_adjustments):,}")

ANALYTICAL POPULATIONS
All governed records       : 966
Product records            : 960
Positive sales activity    : 936
Negative sales activity    : 15
Zero-unit adjustments      : 9


In [17]:
# ============================================================
#  ANALYTICAL POPULATION RECONCILIATION
# ============================================================

population_check = pd.DataFrame({
    "Population": [
        "All governed records",
        "Product records",
        "Service charges",
        "Positive product sales",
        "Negative product activity",
        "Zero-unit product adjustments"
    ],
    "Records": [
        len(sales_all),
        len(sales_products),
        (sales_all["Record_Type"] == "SERVICE_CHARGE").sum(),
        len(sales_positive),
        len(sales_negative),
        len(sales_adjustments)
    ]
})

display(population_check)

product_reconciliation = (
    len(sales_positive)
    + len(sales_negative)
    + len(sales_adjustments)
)

print(
    "\nProduct population reconciles:",
    product_reconciliation == len(sales_products)
)

,Population,Records
0,All governed records,966
1,Product records,960
2,Service charges,6
3,Positive product sales,936
4,Negative product activity,15
5,Zero-unit product adjustments,9



Product population reconciles: True


###  Step 2 — Sales Exploratory & Performance Analysis

####  Objective

Establish the overall commercial performance of the governed sales dataset before performing deeper product, category, Pareto and margin analysis.

This section evaluates:

- Net sales activity
- Units sold
- Revenue
- Cost of sales
- Gross profit
- Gross margin
- Monthly performance
- Transaction adjustments and reversals

Product sales, negative activity and financial adjustments are retained according to their governed transaction classifications.

In [18]:
# ============================================================
#  OVERALL SALES KPI BASELINE
# ============================================================

total_units = sales_products["Sold Period"].sum()
total_revenue = sales_products["Sales Value"].sum()
total_cost_sales = sales_products["Cost Sales"].sum()
total_profit = sales_products["Profit"].sum()

gross_margin_pct = (
    total_profit / total_revenue * 100
    if total_revenue != 0
    else np.nan
)

unique_products = sales_products["Product_ID"].nunique()
active_categories = sales_products["Category"].nunique()

sales_kpis = pd.DataFrame({
    "KPI": [
        "Net Units",
        "Net Revenue",
        "Cost of Sales",
        "Gross Profit",
        "Gross Margin %",
        "Unique Products",
        "Categories"
    ],
    "Value": [
        total_units,
        total_revenue,
        total_cost_sales,
        total_profit,
        gross_margin_pct,
        unique_products,
        active_categories
    ]
})

display(sales_kpis)

,KPI,Value
0,Net Units,"1,163.00"
1,Net Revenue,"279,248.37"
2,Cost of Sales,"273,671.73"
3,Gross Profit,"5,576.64"
4,Gross Margin %,2.00
5,Unique Products,766.00
6,Categories,88.00


#####  Positive vs Negative vs Adjustment Contribution

In [19]:
# ============================================================
# TRANSACTION TYPE CONTRIBUTION
# ============================================================

transaction_summary = (
    sales_products
    .groupby("Transaction_Status", as_index=False)
    .agg(
        Records=("Sales_Record_ID", "count"),
        Units=("Sold Period", "sum"),
        Revenue=("Sales Value", "sum"),
        Cost_Sales=("Cost Sales", "sum"),
        Profit=("Profit", "sum")
    )
)

transaction_summary["Revenue_Share_%"] = (
    transaction_summary["Revenue"] /
    sales_products["Sales Value"].sum() * 100
)

display(transaction_summary)

,Transaction_Status,Records,Units,Revenue,Cost_Sales,Profit,Revenue_Share_%
0,NEGATIVE_SALES_ACTIVITY,15,-19,"-3,548.22","-3,135.80",-412.42,-1.27
1,POSITIVE_SALES_ACTIVITY,936,1182,"282,849.08","276,813.19","6,035.89",101.29
2,ZERO_UNIT_FINANCIAL_ADJUSTMENT,9,0,-52.49,-5.66,-46.83,-0.02


In [20]:
# ============================================================
#  MONTHLY SALES PERFORMANCE
# ============================================================

month_order = ["Nov", "Dec", "Jan"]

monthly_sales = (
    sales_products
    .groupby("Source_Month", as_index=False)
    .agg(
        Records=("Sales_Record_ID", "count"),
        Unique_Products=("Product_ID", "nunique"),
        Net_Units=("Sold Period", "sum"),
        Revenue=("Sales Value", "sum"),
        Cost_Sales=("Cost Sales", "sum"),
        Profit=("Profit", "sum")
    )
)

monthly_sales["Gross_Margin_%"] = np.where(
    monthly_sales["Revenue"] != 0,
    monthly_sales["Profit"] / monthly_sales["Revenue"] * 100,
    np.nan
)

monthly_sales["Source_Month"] = pd.Categorical(
    monthly_sales["Source_Month"],
    categories=month_order,
    ordered=True
)

monthly_sales = (
    monthly_sales
    .sort_values("Source_Month")
    .reset_index(drop=True)
)

display(monthly_sales)

,Source_Month,Records,Unique_Products,Net_Units,Revenue,Cost_Sales,Profit,Gross_Margin_%
0,Nov,379,379,461,"126,996.60","124,602.42","2,394.18",1.89
1,Dec,321,317,385,"89,628.39","85,107.76","4,520.63",5.04
2,Jan,260,260,317,"62,623.38","63,961.55","-1,338.17",-2.14


In [21]:
# ============================================================
# MONTH-OVER-MONTH CHANGE
# ============================================================

monthly_sales["Revenue_MoM_%"] = (
    monthly_sales["Revenue"].pct_change() * 100
)

monthly_sales["Profit_MoM_%"] = (
    monthly_sales["Profit"].pct_change() * 100
)

monthly_sales["Units_MoM_%"] = (
    monthly_sales["Net_Units"].pct_change() * 100
)

display(
    monthly_sales[
        [
            "Source_Month",
            "Net_Units",
            "Revenue",
            "Revenue_MoM_%",
            "Profit",
            "Profit_MoM_%",
            "Gross_Margin_%"
        ]
    ]
)

,Source_Month,Net_Units,Revenue,Revenue_MoM_%,Profit,Profit_MoM_%,Gross_Margin_%
0,Nov,461,"126,996.60",NaN,"2,394.18",NaN,1.89
1,Dec,385,"89,628.39",-29.42,"4,520.63",88.82,5.04
2,Jan,317,"62,623.38",-30.13,"-1,338.17",-129.60,-2.14


### Step 3 — Category Sales Performance

#### Objective

Evaluate sales performance across product categories to determine:

- Which categories generate the most revenue
- Which categories generate the most gross profit
- Which categories operate at weak or negative margins
- Which categories contribute to monthly performance changes
- Which categories require deeper SKU-level investigation

Category performance is calculated from governed product transactions, including
negative activity and financial adjustments where applicable, so financial results
represent net commercial performance.

In [22]:
# ============================================================
# CATEGORY KPI SUMMARY
# ============================================================

category_sales = (
    sales_products
    .groupby("Category", as_index=False)
    .agg(
        Records=("Sales_Record_ID", "count"),
        Unique_Products=("Product_ID", "nunique"),
        Net_Units=("Sold Period", "sum"),
        Revenue=("Sales Value", "sum"),
        Cost_Sales=("Cost Sales", "sum"),
        Profit=("Profit", "sum")
    )
)

category_sales["Gross_Margin_%"] = np.where(
    category_sales["Revenue"] != 0,
    category_sales["Profit"] / category_sales["Revenue"] * 100,
    np.nan
)

category_sales["Revenue_Share_%"] = (
    category_sales["Revenue"] /
    category_sales["Revenue"].sum() * 100
)

category_sales["Profit_Share_%"] = (
    category_sales["Profit"] /
    category_sales["Profit"].sum() * 100
)

category_sales = (
    category_sales
    .sort_values("Revenue", ascending=False)
    .reset_index(drop=True)
)

display(category_sales.head(20))

,Category,Records,Unique_Products,Net_Units,Revenue,Cost_Sales,Profit,Gross_Margin_%,Revenue_Share_%,Profit_Share_%
0,WASHING MACHINES,40,30,50,"22,279.54","21,018.49","1,261.05",5.66,7.98,22.61
1,TV 51 - 59,25,18,33,"17,919.17","21,413.07","-3,493.90",-19.50,6.42,-62.65
2,USA F/F,18,12,19,"17,525.42","17,897.31",-371.89,-2.12,6.28,-6.67
3,SINGLE OVENS,29,27,29,"17,204.18","15,525.43","1,678.75",9.76,6.16,30.10
4,TUMBLE DRYERS,24,17,28,"15,132.28","12,962.90","2,169.38",14.34,5.42,38.90
5,RANGE COOKERS,9,9,9,"13,959.16","14,968.03","-1,008.87",-7.23,5.00,-18.09
6,COFFEE MAKERS,27,23,28,"13,252.44","11,575.08","1,677.36",12.66,4.75,30.08
7,TV 60 - 70,13,9,13,"10,961.66","12,900.00","-1,938.34",-17.68,3.93,-34.76
8,INT DISHWASHERS,14,12,17,"9,777.52","8,446.67","1,330.85",13.61,3.50,23.86
9,FRIDGE FREEZERS,16,13,16,"8,515.84","7,865.64",650.20,7.64,3.05,11.66


In [23]:
# ============================================================
#  TOP REVENUE CATEGORIES
# ============================================================

top_revenue_categories = (
    category_sales[
        [
            "Category",
            "Net_Units",
            "Revenue",
            "Revenue_Share_%",
            "Profit",
            "Gross_Margin_%"
        ]
    ]
    .sort_values("Revenue", ascending=False)
    .head(15)
)

display(top_revenue_categories)

,Category,Net_Units,Revenue,Revenue_Share_%,Profit,Gross_Margin_%
0,WASHING MACHINES,50,"22,279.54",7.98,"1,261.05",5.66
1,TV 51 - 59,33,"17,919.17",6.42,"-3,493.90",-19.50
2,USA F/F,19,"17,525.42",6.28,-371.89,-2.12
3,SINGLE OVENS,29,"17,204.18",6.16,"1,678.75",9.76
4,TUMBLE DRYERS,28,"15,132.28",5.42,"2,169.38",14.34
5,RANGE COOKERS,9,"13,959.16",5.00,"-1,008.87",-7.23
6,COFFEE MAKERS,28,"13,252.44",4.75,"1,677.36",12.66
7,TV 60 - 70,13,"10,961.66",3.93,"-1,938.34",-17.68
8,INT DISHWASHERS,17,"9,777.52",3.50,"1,330.85",13.61
9,FRIDGE FREEZERS,16,"8,515.84",3.05,650.20,7.64


In [24]:
# ============================================================
#  PROFIT PERFORMANCE BY CATEGORY
# ============================================================

top_profit_categories = (
    category_sales
    .sort_values("Profit", ascending=False)
    .head(15)
)

loss_categories = (
    category_sales[category_sales["Profit"] < 0]
    .sort_values("Profit")
)

print("TOP PROFIT-GENERATING CATEGORIES")
display(
    top_profit_categories[
        ["Category", "Revenue", "Profit", "Gross_Margin_%"]
    ]
)

print("\nLOSS-MAKING CATEGORIES")
display(
    loss_categories[
        ["Category", "Revenue", "Profit", "Gross_Margin_%"]
    ]
)

TOP PROFIT-GENERATING CATEGORIES


,Category,Revenue,Profit,Gross_Margin_%
4,TUMBLE DRYERS,"15,132.28","2,169.38",14.34
3,SINGLE OVENS,"17,204.18","1,678.75",9.76
6,COFFEE MAKERS,"13,252.44","1,677.36",12.66
12,DOWNDRAFT HOBS,"7,731.67","1,424.35",18.42
43,INSTALLATION,"1,412.52","1,412.23",99.98
8,INT DISHWASHERS,"9,777.52","1,330.85",13.61
0,WASHING MACHINES,"22,279.54","1,261.05",5.66
18,TAPS,"4,220.00","1,024.78",24.28
11,ROBOT CLEANING,"8,066.64",802.11,9.94
19,SPEAKERS,"3,965.79",736.29,18.57



LOSS-MAKING CATEGORIES


,Category,Revenue,Profit,Gross_Margin_%
87,MISC,"-3,954.19","-3,954.31",100.00
1,TV 51 - 59,"17,919.17","-3,493.90",-19.50
7,TV 60 - 70,"10,961.66","-1,938.34",-17.68
25,TV 75+,"2,828.34","-1,538.13",-54.38
10,STICK VACS,"8,297.41","-1,486.14",-17.91
13,TV 33 - 43,"6,756.64","-1,128.03",-16.70
5,RANGE COOKERS,"13,959.16","-1,008.87",-7.23
26,TV 44 - 50,"2,805.83",-894.93,-31.90
33,TV 60+,"2,234.99",-693.85,-31.04
24,FRYERS,"3,172.47",-517.45,-16.31


In [25]:
# ============================================================
#  CATEGORY PROFITABILITY DIAGNOSTICS
# ============================================================

category_profitability = pd.DataFrame({
    "Metric": [
        "Total Categories",
        "Profitable Categories",
        "Break-even Categories",
        "Loss-making Categories"
    ],
    "Count": [
        len(category_sales),
        (category_sales["Profit"] > 0).sum(),
        (category_sales["Profit"] == 0).sum(),
        (category_sales["Profit"] < 0).sum()
    ]
})

category_profitability["Share_%"] = (
    category_profitability["Count"] /
    len(category_sales) * 100
)

display(category_profitability)

,Metric,Count,Share_%
0,Total Categories,88,100.00
1,Profitable Categories,68,77.27
2,Break-even Categories,0,0.00
3,Loss-making Categories,20,22.73


In [26]:
# ============================================================
#  CATEGORY × MONTH PERFORMANCE
# ============================================================

category_month = (
    sales_products
    .groupby(["Source_Month", "Category"], as_index=False)
    .agg(
        Net_Units=("Sold Period", "sum"),
        Revenue=("Sales Value", "sum"),
        Cost_Sales=("Cost Sales", "sum"),
        Profit=("Profit", "sum")
    )
)

category_month["Gross_Margin_%"] = np.where(
    category_month["Revenue"] > 0,
    category_month["Profit"] / category_month["Revenue"] * 100,
    np.nan
)

category_month["Source_Month"] = pd.Categorical(
    category_month["Source_Month"],
    categories=["Nov", "Dec", "Jan"],
    ordered=True
)

category_month = category_month.sort_values(
    ["Source_Month", "Revenue"],
    ascending=[True, False]
)

display(category_month.head(20))

,Source_Month,Category,Net_Units,Revenue,Cost_Sales,Profit,Gross_Margin_%
208,Nov,USA F/F,10,"10,175.84","9,988.83",187.01,1.84
202,Nov,TV 51 - 59,17,"9,520.84","11,735.64","-2,214.80",-23.26
188,Nov,SINGLE OVENS,15,"7,999.18","7,233.75",765.43,9.57
198,Nov,TUMBLE DRYERS,15,"7,931.46","6,866.50","1,064.96",13.43
213,Nov,WASHING MACHINES,16,"7,521.69","7,212.30",309.39,4.11
203,Nov,TV 60 - 70,7,"6,052.50","7,344.18","-1,291.68",-21.34
168,Nov,INT DISHWASHERS,9,"4,873.33","4,371.63",501.70,10.29
200,Nov,TV 33 - 43,15,"4,215.81","4,967.90",-752.09,-17.84
144,Nov,COFFEE MAKERS,11,"4,012.47","3,333.43",679.04,16.92
184,Nov,ROBOT CLEANING,9,"3,926.64","3,591.18",335.46,8.54


In [27]:
# ============================================================
# JANUARY LOSS DECOMPOSITION
# ============================================================

jan_category = (
    category_month[
        category_month["Source_Month"] == "Jan"
    ]
    .copy()
)

jan_category = jan_category.sort_values("Profit")

print("JANUARY — LARGEST LOSS-MAKING CATEGORIES")

display(
    jan_category[
        jan_category["Profit"] < 0
    ][
        [
            "Category",
            "Net_Units",
            "Revenue",
            "Cost_Sales",
            "Profit",
            "Gross_Margin_%"
        ]
    ]
)

JANUARY — LARGEST LOSS-MAKING CATEGORIES


,Category,Net_Units,Revenue,Cost_Sales,Profit,Gross_Margin_%
103,MISC,4,"-4,104.19",0.04,"-4,104.23",NaN
106,RANGE COOKERS,2,"2,666.67","3,314.88",-648.21,-24.31
116,STICK VACS,13,"2,897.85","3,450.30",-552.45,-19.06
124,TV 51 - 59,9,"4,349.16","4,870.64",-521.48,-11.99
125,TV 60 - 70,3,"1,702.50","2,134.98",-432.48,-25.40
110,SECURITY,4,403.33,696.27,-292.94,-72.63
130,USA F/F,3,"2,330.83","2,538.33",-207.50,-8.90
86,FRYERS,9,"1,110.83","1,282.06",-171.23,-15.41
126,TV 75+,1,495.84,592.47,-96.63,-19.49
122,TV 33 - 43,2,363.33,418.77,-55.44,-15.26


In [28]:
jan_loss_categories = jan_category[
    jan_category["Profit"] < 0
].copy()

total_jan_category_losses = abs(
    jan_loss_categories["Profit"].sum()
)

jan_loss_categories["Loss_Contribution_%"] = (
    abs(jan_loss_categories["Profit"]) /
    total_jan_category_losses * 100
)

jan_loss_categories["Cumulative_Loss_%"] = (
    jan_loss_categories["Loss_Contribution_%"].cumsum()
)

display(
    jan_loss_categories[
        [
            "Category",
            "Revenue",
            "Profit",
            "Loss_Contribution_%",
            "Cumulative_Loss_%"
        ]
    ]
)

,Category,Revenue,Profit,Loss_Contribution_%,Cumulative_Loss_%
103,MISC,"-4,104.19","-4,104.23",56.78,56.78
106,RANGE COOKERS,"2,666.67",-648.21,8.97,65.74
116,STICK VACS,"2,897.85",-552.45,7.64,73.39
124,TV 51 - 59,"4,349.16",-521.48,7.21,80.60
125,TV 60 - 70,"1,702.50",-432.48,5.98,86.58
110,SECURITY,403.33,-292.94,4.05,90.63
130,USA F/F,"2,330.83",-207.50,2.87,93.51
86,FRYERS,"1,110.83",-171.23,2.37,95.87
126,TV 75+,495.84,-96.63,1.34,97.21
122,TV 33 - 43,363.33,-55.44,0.77,97.98


In [29]:
# ============================================================
#  CATEGORY PROFIT BY MONTH
# ============================================================

category_profit_pivot = (
    category_month
    .pivot(
        index="Category",
        columns="Source_Month",
        values="Profit"
    )
    .fillna(0)
)

category_profit_pivot["Total_Profit"] = (
    category_profit_pivot.sum(axis=1)
)

category_profit_pivot = (
    category_profit_pivot
    .sort_values("Total_Profit")
)

display(category_profit_pivot)

Source_Month,Nov,Dec,Jan,Total_Profit
Category,,,,
MISC,0.00,149.92,"-4,104.23","-3,954.31"
TV 51 - 59,"-2,214.80",-757.62,-521.48,"-3,493.90"
TV 60 - 70,"-1,291.68",-214.18,-432.48,"-1,938.34"
TV 75+,"-1,441.50",0.00,-96.63,"-1,538.13"
STICK VACS,-303.27,-630.42,-552.45,"-1,486.14"
...,...,...,...,...
INSTALLATION,308.26,695.70,408.27,"1,412.23"
DOWNDRAFT HOBS,608.45,0.00,815.90,"1,424.35"
COFFEE MAKERS,679.04,952.79,45.53,"1,677.36"


In [30]:
# ============================================================
#  INVESTIGATE MISC FINANCIAL ANOMALY
# ============================================================

misc_records = (
    sales_products[
        sales_products["Category"] == "MISC"
    ]
    .sort_values(["Source_Month", "Sales Value"])
)

display(
    misc_records[
        [
            "Sales_Record_ID",
            "Source_Month",
            "Product_ID",
            "Product_Key",
            "Stock Code",
            "Description",
            "Level",
            "Sold Period",
            "Transaction_Status",
            "Unit Cost",
            "Unit Price",
            "Cost Sales",
            "Sales Value",
            "Profit",
            "Profit %",
            "Cost_Sales_Reconciliation_Flag"
        ]
    ]
)

,Sales_Record_ID,Source_Month,Product_ID,Product_Key,Stock Code,Description,Level,Sold Period,Transaction_Status,Unit Cost,Unit Price,Cost Sales,Sales Value,Profit,Profit %,Cost_Sales_Reconciliation_Flag
570,571,Dec,767,M,M,MISCELANEOUS,-22,8,POSITIVE_SALES_ACTIVITY,0.01,0.49,0.08,150.00,149.92,99.95,MATCH
856,857,Jan,767,M,M,MISCELANEOUS,-22,4,POSITIVE_SALES_ACTIVITY,0.01,0.49,0.04,"-4,104.19","-4,104.23",100.00,MATCH


In [31]:
# MISC summary by month

misc_month_summary = (
    misc_records
    .groupby("Source_Month", as_index=False)
    .agg(
        Records=("Sales_Record_ID", "count"),
        Units=("Sold Period", "sum"),
        Revenue=("Sales Value", "sum"),
        Cost_Sales=("Cost Sales", "sum"),
        Profit=("Profit", "sum")
    )
)

display(misc_month_summary)

,Source_Month,Records,Units,Revenue,Cost_Sales,Profit
0,Dec,1,8,150.00,0.08,149.92
1,Jan,1,4,"-4,104.19",0.04,"-4,104.23"


In [32]:
# ============================================================
# PERSISTENT CATEGORY PROFITABILITY
# ============================================================

profit_by_month = (
    category_month
    .pivot(
        index="Category",
        columns="Source_Month",
        values="Profit"
    )
    .fillna(0)
)

# Whether category actually appeared in each month
presence_by_month = (
    category_month
    .assign(Present=1)
    .pivot(
        index="Category",
        columns="Source_Month",
        values="Present"
    )
    .fillna(0)
)

for month in ["Nov", "Dec", "Jan"]:
    if month not in profit_by_month.columns:
        profit_by_month[month] = 0

    if month not in presence_by_month.columns:
        presence_by_month[month] = 0


persistent_analysis = pd.DataFrame(index=profit_by_month.index)

persistent_analysis["Months_Present"] = (
    presence_by_month[["Nov", "Dec", "Jan"]].sum(axis=1)
)

persistent_analysis["Loss_Months"] = (
    (
        (profit_by_month[["Nov", "Dec", "Jan"]] < 0)
        & (presence_by_month[["Nov", "Dec", "Jan"]] == 1)
    )
    .sum(axis=1)
)

persistent_analysis["Total_Profit"] = (
    profit_by_month[["Nov", "Dec", "Jan"]].sum(axis=1)
)

persistent_analysis = (
    persistent_analysis
    .reset_index()
    .sort_values(
        ["Loss_Months", "Total_Profit"],
        ascending=[False, True]
    )
)

display(persistent_analysis.head(30))

,Category,Months_Present,Loss_Months,Total_Profit
72,TV 51 - 59,3.00,3,"-3,493.90"
73,TV 60 - 70,3.00,3,"-1,938.34"
62,STICK VACS,3.00,3,"-1,486.14"
70,TV 33 - 43,3.00,3,"-1,128.03"
49,RANGE COOKERS,3.00,3,"-1,008.87"
71,TV 44 - 50,3.00,3,-894.93
24,FRYERS,3.00,3,-517.45
30,HOME CINEMA,3.00,3,-386.20
75,TV 75+,2.00,2,"-1,538.13"
80,USA F/F,3.00,2,-371.89


In [33]:
persistent_losses = persistent_analysis[
    (persistent_analysis["Months_Present"] == 3) &
    (persistent_analysis["Loss_Months"] == 3)
]

print("CATEGORIES LOSS-MAKING IN ALL THREE MONTHS")
display(persistent_losses)

CATEGORIES LOSS-MAKING IN ALL THREE MONTHS


,Category,Months_Present,Loss_Months,Total_Profit
72,TV 51 - 59,3.00,3,"-3,493.90"
73,TV 60 - 70,3.00,3,"-1,938.34"
62,STICK VACS,3.00,3,"-1,486.14"
70,TV 33 - 43,3.00,3,"-1,128.03"
49,RANGE COOKERS,3.00,3,"-1,008.87"
71,TV 44 - 50,3.00,3,-894.93
24,FRYERS,3.00,3,-517.45
30,HOME CINEMA,3.00,3,-386.20


In [34]:
# ============================================================
#  DEFINE STANDARD MERCHANDISE POPULATION
# ============================================================

sales_products["Analysis_Class"] = np.where(
    sales_products["Category"].eq("MISC"),
    "SPECIAL_MISC_ACTIVITY",
    "STANDARD_MERCHANDISE"
)

print(sales_products["Analysis_Class"].value_counts())

Analysis_Class
STANDARD_MERCHANDISE     958
SPECIAL_MISC_ACTIVITY      2
Name: count, dtype: int64


In [35]:
# Standard merchandise population for product/category analysis

sales_merchandise = sales_products[
    sales_products["Analysis_Class"] == "STANDARD_MERCHANDISE"
].copy()

# Special MISC records retained separately
sales_misc = sales_products[
    sales_products["Analysis_Class"] == "SPECIAL_MISC_ACTIVITY"
].copy()

print(f"All product records       : {len(sales_products):,}")
print(f"Standard merchandise      : {len(sales_merchandise):,}")
print(f"Special MISC activity     : {len(sales_misc):,}")

print(
    "\nPopulation reconciles:",
    len(sales_merchandise) + len(sales_misc) == len(sales_products)
)

All product records       : 960
Standard merchandise      : 958
Special MISC activity     : 2

Population reconciles: True


The source contains two MISCELLANEOUS records whose financial values do not behave like standard merchandise sales. Their precise business/accounting meaning cannot be established from the available sales data. They are therefore retained for financial reconciliation but separated from standard merchandise-performance analysis.

### Step 4 — Product / SKU-Level Sales Analysis

#### Objective

Analyse performance at individual product/SKU level to identify:

- Highest-revenue products
- Highest-volume products
- Highest-profit products
- Loss-making products
- Products responsible for persistent category losses
- Revenue and profit concentration
- Products requiring further pricing, SOA, stock or promotional investigation

Special MISC activity is excluded from merchandise-performance analysis but
remains retained in the governed financial dataset.### Step 4 — Product / SKU-Level Sales Analysis

In [36]:
# ============================================================
#  PRODUCT PERFORMANCE SUMMARY
# ============================================================

product_sales = (
    sales_merchandise
    .groupby(
        [
            "Product_ID",
            "Product_Key",
            "Stock Code",
            "Description",
            "Category"
        ],
        as_index=False
    )
    .agg(
        Months_Present=("Source_Month", "nunique"),
        Records=("Sales_Record_ID", "count"),
        Net_Units=("Sold Period", "sum"),
        Revenue=("Sales Value", "sum"),
        Cost_Sales=("Cost Sales", "sum"),
        Profit=("Profit", "sum")
    )
)

product_sales["Gross_Margin_%"] = np.where(
    product_sales["Revenue"] > 0,
    product_sales["Profit"] / product_sales["Revenue"] * 100,
    np.nan
)

product_sales["Revenue_Share_%"] = (
    product_sales["Revenue"] /
    product_sales["Revenue"].sum() * 100
)

product_sales = (
    product_sales
    .sort_values("Revenue", ascending=False)
    .reset_index(drop=True)
)

print("PRODUCT PERFORMANCE DATASET")
print("=" * 50)
print(f"Products analysed : {len(product_sales):,}")
print(f"Total net units   : {product_sales['Net_Units'].sum():,.0f}")
print(f"Total revenue     : £{product_sales['Revenue'].sum():,.2f}")
print(f"Total profit      : £{product_sales['Profit'].sum():,.2f}")

display(product_sales.head(20))

PRODUCT PERFORMANCE DATASET
Products analysed : 769
Total net units   : 1,151
Total revenue     : £283,202.56
Total profit      : £9,530.95


,Product_ID,Product_Key,Stock Code,Description,Category,Months_Present,Records,Net_Units,Revenue,Cost_Sales,Profit,Gross_Margin_%,Revenue_Share_%
0,1167,T2351V11,T2351V11,Eufy Robot Vaccum X10 Pro Omni,ROBOT CLEANING,3,3,16,"6,818.30","6,384.32",433.98,6.36,2.41
1,116,42120,42120,Novy Panorama 120 Pro 5 Zone,DOWNDRAFT HOBS,1,1,1,"4,000.00","3,184.10",815.90,20.40,1.41
2,870,OLED65G54L,OLED65G54L,LG 65in G5 OLED TV,TV 60 - 70,2,2,2,"3,165.00","3,311.22",-146.22,-4.62,1.12
3,1320,WEK365WCS,WEK365WCS,Miele (12392780) 10kg 1400 Spin,WASHING MACHINES,1,1,3,"2,832.49","2,392.68",439.81,15.53,1.00
4,1213,TEH785WP,TEH785WP,Miele 11871830 9kg Heat Pump,TUMBLE DRYERS,2,2,3,"2,764.17","2,563.02",201.15,7.28,0.98
5,560,H7464BPBL,H7464BPBL,Miele Black 11093600 Pyro Single,SINGLE OVENS,1,1,2,"2,683.34","2,582.34",101.00,3.76,0.95
6,1425,XRFSD5265,XRFSD5265,Liebherr SXS,FRIDGE FREEZERS,1,1,1,"2,666.67","2,127.55",539.12,20.22,0.94
7,856,OLED55G54L,OLED55G54L,LG 55in G5 OLED TV,TV 51 - 59,2,2,2,"2,581.67","2,538.66",43.01,1.67,0.91
8,1037,RS70F64KEF,RS70F64KEF,Samsung Black St/St USA FF,USA F/F,3,3,3,"2,580.83","2,972.94",-392.11,-15.19,0.91
9,1246,U1ACE2AG3BNEFF,U1ACE2AG3BNeff,N50 Graphite Double Oven,DOUBLE OVENS,3,3,4,"2,538.33","2,329.68",208.65,8.22,0.90


In [37]:
# ============================================================
#  TOP PRODUCTS BY REVENUE
# ============================================================

top_revenue_products = (
    product_sales
    .sort_values("Revenue", ascending=False)
    .head(20)
)

display(
    top_revenue_products[
        [
            "Product_Key",
            "Description",
            "Category",
            "Net_Units",
            "Revenue",
            "Revenue_Share_%",
            "Profit",
            "Gross_Margin_%"
        ]
    ]
)

,Product_Key,Description,Category,Net_Units,Revenue,Revenue_Share_%,Profit,Gross_Margin_%
0,T2351V11,Eufy Robot Vaccum X10 Pro Omni,ROBOT CLEANING,16,"6,818.30",2.41,433.98,6.36
1,42120,Novy Panorama 120 Pro 5 Zone,DOWNDRAFT HOBS,1,"4,000.00",1.41,815.90,20.40
2,OLED65G54L,LG 65in G5 OLED TV,TV 60 - 70,2,"3,165.00",1.12,-146.22,-4.62
3,WEK365WCS,Miele (12392780) 10kg 1400 Spin,WASHING MACHINES,3,"2,832.49",1.00,439.81,15.53
4,TEH785WP,Miele 11871830 9kg Heat Pump,TUMBLE DRYERS,3,"2,764.17",0.98,201.15,7.28
5,H7464BPBL,Miele Black 11093600 Pyro Single,SINGLE OVENS,2,"2,683.34",0.95,101.00,3.76
6,XRFSD5265,Liebherr SXS,FRIDGE FREEZERS,1,"2,666.67",0.94,539.12,20.22
7,OLED55G54L,LG 55in G5 OLED TV,TV 51 - 59,2,"2,581.67",0.91,43.01,1.67
8,RS70F64KEF,Samsung Black St/St USA FF,USA F/F,3,"2,580.83",0.91,-392.11,-15.19
9,U1ACE2AG3BNEFF,N50 Graphite Double Oven,DOUBLE OVENS,4,"2,538.33",0.90,208.65,8.22


In [38]:
# ============================================================
#  TOP PRODUCTS BY SALES VOLUME
# ============================================================

top_volume_products = (
    product_sales
    .sort_values("Net_Units", ascending=False)
    .head(20)
)

display(
    top_volume_products[
        [
            "Product_Key",
            "Description",
            "Category",
            "Net_Units",
            "Revenue",
            "Profit",
            "Gross_Margin_%"
        ]
    ]
)

,Product_Key,Description,Category,Net_Units,Revenue,Profit,Gross_Margin_%
44,INSTALLATIO,INSTALLATION FEE,INSTALLATION,27,"1,312.52","1,312.25",99.98
0,T2351V11,Eufy Robot Vaccum X10 Pro Omni,ROBOT CLEANING,16,"6,818.30",433.98,6.36
64,VS15A6031R4SAMSUNG,Jet 60 Cordless Vacuum,STICK VACS,9,"1,102.02",18.42,1.67
584,P-SDU32GU18PNY,Elite microSDHC card 32G,IT ACCESSORIES,8,39.90,13.26,33.23
50,43LQ60006LA.LG,"43"" Smart TV",TV 33 - 43,8,"1,254.99",-241.01,-19.20
110,MC1001UK,Ninja 8-in-1 Slow Cooker,FOOD PREP,7,761.65,47.72,6.27
494,GN BAGS,BAGS 400/600/800 SERIES AND S5,VACUUM BAGS,7,77.55,24.42,31.49
520,DELIVERY-CHLOCAL,DELIVERY CHARGE,DELIVERY CHARGE,7,62.50,62.50,100.00
369,KN650A,Kenwood Electric Knife | KN650A,FOOD PREP,7,171.64,64.05,37.32
566,FD16GATT4-EPNY,USB Sliding design - Black read,IT ACCESSORIES,5,42.07,27.47,65.30


In [39]:
# ============================================================
#  TOP PROFIT-GENERATING PRODUCTS
# ============================================================

top_profit_products = (
    product_sales
    .sort_values("Profit", ascending=False)
    .head(20)
)

display(
    top_profit_products[
        [
            "Product_Key",
            "Description",
            "Category",
            "Net_Units",
            "Revenue",
            "Profit",
            "Gross_Margin_%"
        ]
    ]
)

,Product_Key,Description,Category,Net_Units,Revenue,Profit,Gross_Margin_%
44,INSTALLATIO,INSTALLATION FEE,INSTALLATION,27,"1,312.52","1,312.25",99.98
75,DV90DG52A0,Samsung Series 5 Tumble Dryer,TUMBLE DRYERS,2,"1,020.00","1,020.00",100.00
1,42120,Novy Panorama 120 Pro 5 Zone,DOWNDRAFT HOBS,1,"4,000.00",815.90,20.40
97,RS70F64KET,Samsung USA Fridge Freezer,USA F/F,1,790.83,790.83,100.00
14,USG10TY.DG,LG 3.1 Wireless Soundbar,SPEAKERS,4,"2,246.66",605.34,26.94
6,XRFSD5265,Liebherr SXS,FRIDGE FREEZERS,1,"2,666.67",539.12,20.22
174,WW80CGC04,SAMSUNG Series 5 AI Energy Washing,WASHING MACHINES,2,515.00,515.00,100.00
26,T2353V11,eufy Robot Vacuum Omni E25,ROBOT CLEANING,3,"1,765.01",451.61,25.59
3,WEK365WCS,Miele (12392780) 10kg 1400 Spin,WASHING MACHINES,3,"2,832.49",439.81,15.53
0,T2351V11,Eufy Robot Vaccum X10 Pro Omni,ROBOT CLEANING,16,"6,818.30",433.98,6.36


In [40]:
# ============================================================
#  LARGEST LOSS-MAKING PRODUCTS
# ============================================================

loss_products = (
    product_sales[
        product_sales["Profit"] < 0
    ]
    .sort_values("Profit")
    .copy()
)

print(f"Loss-making products: {len(loss_products):,}")

display(
    loss_products[
        [
            "Product_Key",
            "Description",
            "Category",
            "Months_Present",
            "Net_Units",
            "Revenue",
            "Cost_Sales",
            "Profit",
            "Gross_Margin_%"
        ]
    ].head(30)
)

Loss-making products: 220


,Product_Key,Description,Category,Months_Present,Net_Units,Revenue,Cost_Sales,Profit,Gross_Margin_%
11,OLED83C44L,"LG 83"" OLED Television",TV 75+,1,1,"2,332.50","3,774.00","-1,441.50",-61.80
28,TOLP110DFF,Rangemaster Toledo 110 Ind St/St,RANGE COOKERS,1,1,"1,666.67","2,435.13",-768.46,-46.11
106,OLED55G45L,"LG 55"" OLED Television",TV 44 - 50,1,1,770.00,"1,471.35",-701.35,-91.08
77,OLED65C44L,"LG 65"" OLED Television",TV 60 - 70,1,1,990.83,"1,671.10",-680.27,-68.66
33,QE55QN90FA,"Samsung 55"" Neo QLED 4K",TV 51 - 59,2,2,"1,540.00","2,210.22",-670.22,-43.52
73,QE65S85FAE,"Samsung 65"" OLED 4K Smart TV",TV 60+,1,1,"1,028.33","1,579.01",-550.68,-53.55
22,WW11DB8B95SAMSUNG,Blk Stl Series 8 11Kg 1400,WASHING MACHINES,2,3,"1,863.34","2,314.98",-451.64,-24.24
86,OLED65B56LALG,65in B5 OLED TV,TV 60 - 70,1,1,832.50,"1,260.63",-428.13,-51.43
46,UE55U7000FKSAMSUNG,55 in Smart Television,TV 51 - 59,2,5,"1,304.17","1,720.60",-416.43,-31.93
66,RS68A884CSLSAMSUNG,USA Plumbed FF Ice &,USA F/F,1,1,"1,082.50","1,495.49",-412.99,-38.15


In [41]:
# ============================================================
#  PRODUCT PROFITABILITY DISTRIBUTION
# ============================================================

product_profitability = pd.DataFrame({
    "Metric": [
        "Total Products",
        "Profitable Products",
        "Break-even Products",
        "Loss-making Products"
    ],
    "Count": [
        len(product_sales),
        (product_sales["Profit"] > 0).sum(),
        (product_sales["Profit"] == 0).sum(),
        (product_sales["Profit"] < 0).sum()
    ]
})

product_profitability["Share_%"] = (
    product_profitability["Count"] /
    len(product_sales) * 100
)

display(product_profitability)

,Metric,Count,Share_%
0,Total Products,769,100.00
1,Profitable Products,537,69.83
2,Break-even Products,12,1.56
3,Loss-making Products,220,28.61


In [42]:
# ============================================================
#  VALIDATE PRODUCT ANALYTICAL GRAIN
# ============================================================

print("MERCHANDISE PRODUCT IDENTIFIER CHECK")
print("=" * 55)

print(
    f"Unique Product_ID   : "
    f"{sales_merchandise['Product_ID'].nunique():,}"
)

print(
    f"Unique Product_Key  : "
    f"{sales_merchandise['Product_Key'].nunique():,}"
)

print(
    f"Unique Stock Code   : "
    f"{sales_merchandise['Stock Code'].nunique():,}"
)

print(
    f"Aggregated rows     : "
    f"{len(product_sales):,}"
)

MERCHANDISE PRODUCT IDENTIFIER CHECK
Unique Product_ID   : 765
Unique Product_Key  : 765
Unique Stock Code   : 765
Aggregated rows     : 769


In [43]:
# Number of aggregated rows generated for each Product_ID

product_grain_check = (
    product_sales
    .groupby("Product_ID", as_index=False)
    .agg(
        Aggregated_Rows=("Product_Key", "size"),
        Product_Keys=("Product_Key", "nunique"),
        Stock_Codes=("Stock Code", "nunique"),
        Descriptions=("Description", "nunique"),
        Categories=("Category", "nunique")
    )
)

split_products = (
    product_grain_check[
        product_grain_check["Aggregated_Rows"] > 1
    ]
    .sort_values("Aggregated_Rows", ascending=False)
)

print(
    f"Product_IDs split into multiple analytical rows: "
    f"{len(split_products):,}"
)

display(split_products)

Product_IDs split into multiple analytical rows: 3


,Product_ID,Aggregated_Rows,Product_Keys,Stock_Codes,Descriptions,Categories
380,615,3,1,1,3,1
554,1005,2,1,1,2,2
750,1412,2,1,1,2,1


In [44]:
# ============================================================
# INSPECT SPLIT PRODUCT RECORDS
# ============================================================

split_ids = split_products["Product_ID"].tolist()

split_product_records = (
    sales_merchandise[
        sales_merchandise["Product_ID"].isin(split_ids)
    ][
        [
            "Product_ID",
            "Product_Key",
            "Stock Code",
            "Description",
            "Category",
            "Source_Month",
            "Sold Period",
            "Sales Value",
            "Profit"
        ]
    ]
    .sort_values(["Product_ID", "Source_Month"])
)

display(split_product_records)

,Product_ID,Product_Key,Stock Code,Description,Category,Source_Month,Sold Period,Sales Value,Profit
521,615,INSTALLATIO,INSTALLATIO,INSTALLATION FEE,INSTALLATION,Dec,12,595.84,595.72
522,615,INSTALLATIO,INSTALLATIO,INSTALLATION FREE STANDING,INSTALLATION,Dec,1,33.33,33.32
523,615,INSTALLATIO,INSTALLATIO,INSTALLATION BUILT IN,INSTALLATION,Dec,1,66.67,66.66
809,615,INSTALLATIO,INSTALLATIO,INSTALLATION FEE,INSTALLATION,Jan,8,408.35,408.27
150,615,INSTALLATIO,INSTALLATIO,INSTALLATION FEE,INSTALLATION,Nov,7,308.33,308.26
535,1005,REV-ISTREA,REV-ISTREA,Roberts Revival iStream 3L Black,INTERNET RADIOS,Dec,2,332.49,66.49
574,1005,REV-ISTREA,REV-ISTREA,Roberts Revival iStream 3L Radio |,RADIOS,Dec,1,165.83,32.83
171,1005,REV-ISTREA,REV-ISTREA,Roberts Revival iStream 3L Black,INTERNET RADIOS,Nov,2,331.66,65.66
693,1412,WW90DG6U8,WW90DG6U8,Samsung Series 6 9kg 1400 Spin,WASHING MACHINES,Dec,1,374.17,-163.83
699,1412,WW90DG6U8,WW90DG6U8,Samsung White 9kg 1400 Spin,WASHING MACHINES,Dec,1,358.33,-50.51


In [45]:
product_grain_check = (
    product_sales
    .groupby("Product_ID", as_index=False)
    .agg(
        Aggregated_Rows=("Product_Key", "size"),
        Product_Keys=("Product_Key", "nunique"),
        Stock_Codes=("Stock Code", "nunique"),
        Descriptions=("Description", "nunique"),
        Categories=("Category", "nunique")
    )
)

split_products = (
    product_grain_check[
        product_grain_check["Aggregated_Rows"] > 1
    ]
    .sort_values("Aggregated_Rows", ascending=False)
)

print(
    f"Product_IDs split into multiple analytical rows: "
    f"{len(split_products):,}"
)

display(split_products)

Product_IDs split into multiple analytical rows: 3


,Product_ID,Aggregated_Rows,Product_Keys,Stock_Codes,Descriptions,Categories
380,615,3,1,1,3,1
554,1005,2,1,1,2,2
750,1412,2,1,1,2,1


In [46]:
split_ids = split_products["Product_ID"].tolist()

split_product_records = (
    sales_merchandise[
        sales_merchandise["Product_ID"].isin(split_ids)
    ][
        [
            "Product_ID",
            "Product_Key",
            "Stock Code",
            "Description",
            "Category",
            "Source_Month",
            "Sold Period",
            "Sales Value",
            "Profit"
        ]
    ]
    .sort_values(["Product_ID", "Source_Month"])
)

display(split_product_records)

,Product_ID,Product_Key,Stock Code,Description,Category,Source_Month,Sold Period,Sales Value,Profit
521,615,INSTALLATIO,INSTALLATIO,INSTALLATION FEE,INSTALLATION,Dec,12,595.84,595.72
522,615,INSTALLATIO,INSTALLATIO,INSTALLATION FREE STANDING,INSTALLATION,Dec,1,33.33,33.32
523,615,INSTALLATIO,INSTALLATIO,INSTALLATION BUILT IN,INSTALLATION,Dec,1,66.67,66.66
809,615,INSTALLATIO,INSTALLATIO,INSTALLATION FEE,INSTALLATION,Jan,8,408.35,408.27
150,615,INSTALLATIO,INSTALLATIO,INSTALLATION FEE,INSTALLATION,Nov,7,308.33,308.26
535,1005,REV-ISTREA,REV-ISTREA,Roberts Revival iStream 3L Black,INTERNET RADIOS,Dec,2,332.49,66.49
574,1005,REV-ISTREA,REV-ISTREA,Roberts Revival iStream 3L Radio |,RADIOS,Dec,1,165.83,32.83
171,1005,REV-ISTREA,REV-ISTREA,Roberts Revival iStream 3L Black,INTERNET RADIOS,Nov,2,331.66,65.66
693,1412,WW90DG6U8,WW90DG6U8,Samsung Series 6 9kg 1400 Spin,WASHING MACHINES,Dec,1,374.17,-163.83
699,1412,WW90DG6U8,WW90DG6U8,Samsung White 9kg 1400 Spin,WASHING MACHINES,Dec,1,358.33,-50.51


In [49]:
# ============================================================
#  LOAD AND INSPECT PRODUCT DIMENSION
# ============================================================

dim_product_path = "D:/STUDY/Data_Science_Courses/PROJECTS/12.Smart_AI-Retail_System/Smart_AI_Retail_System/data/processed/dim_product.csv"

dim_product = pd.read_csv(dim_product_path)

print("DIM_PRODUCT")
print("=" * 60)
print(f"Rows           : {len(dim_product):,}")
print(f"Columns        : {len(dim_product.columns):,}")
print(f"Unique products: {dim_product['Product_ID'].nunique():,}")

print("\nColumns:")
print(dim_product.columns.tolist())

display(dim_product.head())

DIM_PRODUCT
Rows           : 1,436
Columns        : 18
Unique products: 1,436

Columns:
['Product_ID', 'Product_Key', 'Product_Description', 'Product_Category', 'Record_Type', 'Source_Status', 'Source_Count', 'In_Sales', 'In_Stock', 'In_SOA', 'Sales_Stock_Code', 'Sales_Description', 'Sales_Category', 'Stock_Model', 'Stock_Description', 'Stock_Category', 'SOA_Model', 'SOA_Description']


,Product_ID,Product_Key,Product_Description,Product_Category,Record_Type,Source_Status,Source_Count,In_Sales,In_Stock,In_SOA,Sales_Stock_Code,Sales_Description,Sales_Category,Stock_Model,Stock_Description,Stock_Category,SOA_Model,SOA_Description
0,1,010-02384-10,Garmin Lily Cream Gold & White,FITNESS,PRODUCT,SALES_ONLY,1,True,False,False,010-02384-10,Garmin Lily Cream Gold & White,FITNESS,NaN,NaN,NaN,NaN,NaN
1,2,010-02784-00,Garmin Venu 3 Smartwatch - Silver,FITNESS,PRODUCT,SALES_ONLY,1,True,False,False,010-02784-00,Garmin Venu 3 Smartwatch - Silver,FITNESS,NaN,NaN,NaN,NaN,NaN
2,3,010-02784-01,Garmin Venu 3 Smartwatch - Slate,FITNESS,PRODUCT,SALES_ONLY,1,True,False,False,010-02784-01,Garmin Venu 3 Smartwatch - Slate,FITNESS,NaN,NaN,NaN,NaN,NaN
3,4,010-02839-00,"Garmin Lily 2, Cream Gold w/",FITNESS,PRODUCT,SALES_ONLY,1,True,False,False,010-02839-00,"Garmin Lily 2, Cream Gold w/",FITNESS,NaN,NaN,NaN,NaN,NaN
4,5,01950,NUTRIBULLET PRO 4pc Starter Kit,BLENDERS,PRODUCT,SALES_ONLY,1,True,False,False,01950,NUTRIBULLET PRO 4pc Starter Kit,BLENDERS,NaN,NaN,NaN,NaN,NaN


In [50]:
# ============================================================
#  PRODUCT DIMENSION GRAIN VALIDATION
# ============================================================

product_id_check = (
    dim_product
    .groupby("Product_ID")
    .size()
    .reset_index(name="Rows")
)

duplicate_product_ids = product_id_check[
    product_id_check["Rows"] > 1
]

print("PRODUCT DIMENSION GRAIN CHECK")
print("=" * 60)

print(f"Rows in dim_product          : {len(dim_product):,}")
print(f"Unique Product_IDs           : {dim_product['Product_ID'].nunique():,}")
print(f"Duplicated Product_ID groups : {len(duplicate_product_ids):,}")

if len(duplicate_product_ids) == 0:
    print("\nPASS — Product_ID is unique in dim_product.")
else:
    print("\nWARNING — Product_ID is not unique in dim_product.")
    display(duplicate_product_ids)

PRODUCT DIMENSION GRAIN CHECK
Rows in dim_product          : 1,436
Unique Product_IDs           : 1,436
Duplicated Product_ID groups : 0

PASS — Product_ID is unique in dim_product.


In [51]:
# ============================================================
#  INSPECT POSSIBLE NON-MERCHANDISE PRODUCTS
# ============================================================

service_mask = (
    sales_products["Product_Key"]
    .astype(str)
    .str.upper()
    .str.contains(
        "INSTALL|DELIVERY|SERVICE",
        regex=True,
        na=False
    )
)

possible_services = (
    sales_products.loc[
        service_mask,
        [
            "Product_ID",
            "Product_Key",
            "Stock Code",
            "Description",
            "Category",
            "Source_Month",
            "Sold Period",
            "Sales Value",
            "Profit"
        ]
    ]
    .sort_values(["Product_ID", "Source_Month"])
)

print(
    "Possible non-merchandise records:",
    len(possible_services)
)

display(possible_services)

Possible non-merchandise records: 8


,Product_ID,Product_Key,Stock Code,Description,Category,Source_Month,Sold Period,Sales Value,Profit
438,370,DELIVERY-CHLOCAL,DELIVERY-CHLOCAL,DELIVERY CHARGE,DELIVERY CHARGE,Dec,3,34.17,34.17
747,370,DELIVERY-CHLOCAL,DELIVERY-CHLOCAL,DELIVERY CHARGE,DELIVERY CHARGE,Jan,3,3.33,3.33
59,370,DELIVERY-CHLOCAL,DELIVERY-CHLOCAL,DELIVERY CHARGE,DELIVERY CHARGE,Nov,1,25.00,25.00
521,615,INSTALLATIO,INSTALLATIO,INSTALLATION FEE,INSTALLATION,Dec,12,595.84,595.72
522,615,INSTALLATIO,INSTALLATIO,INSTALLATION FREE STANDING,INSTALLATION,Dec,1,33.33,33.32
523,615,INSTALLATIO,INSTALLATIO,INSTALLATION BUILT IN,INSTALLATION,Dec,1,66.67,66.66
809,615,INSTALLATIO,INSTALLATIO,INSTALLATION FEE,INSTALLATION,Jan,8,408.35,408.27
150,615,INSTALLATIO,INSTALLATIO,INSTALLATION FEE,INSTALLATION,Nov,7,308.33,308.26


In [52]:
# ============================================================
#  VERIFY SERVICE PRODUCTS IN DIM_PRODUCT
# ============================================================

service_ids = [370, 615]

service_dimension_check = (
    dim_product[
        dim_product["Product_ID"].isin(service_ids)
    ][
        [
            "Product_ID",
            "Product_Key",
            "Product_Description",
            "Product_Category",
            "Record_Type",
            "Source_Status",
            "In_Sales",
            "In_Stock",
            "In_SOA"
        ]
    ]
)

display(service_dimension_check)

,Product_ID,Product_Key,Product_Description,Product_Category,Record_Type,Source_Status,In_Sales,In_Stock,In_SOA
369,370,DELIVERY-CHLOCAL,DELIVERY CHARGE,DELIVERY CHARGE,PRODUCT,SALES_ONLY,True,False,False
614,615,INSTALLATIO,INSTALLATION FEE,INSTALLATION,PRODUCT,SALES_ONLY,True,False,False


In [53]:
# ============================================================
#  CREATE MERCHANDISE SALES POPULATION
# ============================================================

NON_MERCHANDISE_PRODUCT_IDS = [370, 615]

sales_merchandise_clean = (
    sales_products[
        ~sales_products["Product_ID"].isin(
            NON_MERCHANDISE_PRODUCT_IDS
        )
    ]
    .copy()
)

print("MERCHANDISE SALES POPULATION")
print("=" * 60)

print(f"Sales product records     : {len(sales_products):,}")
print(
    f"Unique sales Product_IDs  : "
    f"{sales_products['Product_ID'].nunique():,}"
)

print(
    f"Merchandise records       : "
    f"{len(sales_merchandise_clean):,}"
)

print(
    f"Merchandise Product_IDs   : "
    f"{sales_merchandise_clean['Product_ID'].nunique():,}"
)

print(
    f"Excluded service IDs      : "
    f"{NON_MERCHANDISE_PRODUCT_IDS}"
)

MERCHANDISE SALES POPULATION
Sales product records     : 960
Unique sales Product_IDs  : 766
Merchandise records       : 952
Merchandise Product_IDs   : 764
Excluded service IDs      : [370, 615]


In [54]:
# ============================================================
#  PRODUCT PERFORMANCE AT TRUE PRODUCT GRAIN
# ============================================================

product_metrics = (
    sales_merchandise_clean
    .groupby("Product_ID", as_index=False)
    .agg(
        Months_Present=("Source_Month", "nunique"),
        Records=("Sales_Record_ID", "count"),
        Net_Units=("Sold Period", "sum"),
        Revenue=("Sales Value", "sum"),
        Cost_Sales=("Cost Sales", "sum"),
        Profit=("Profit", "sum")
    )
)

product_metrics["Gross_Margin_%"] = (
    product_metrics["Profit"]
    .div(product_metrics["Revenue"])
    .mul(100)
)

print("TRUE PRODUCT-GRAIN DATASET")
print("=" * 60)

print(f"Rows              : {len(product_metrics):,}")
print(
    f"Unique Product_IDs: "
    f"{product_metrics['Product_ID'].nunique():,}"
)

print(
    "One row per product:",
    len(product_metrics)
    == product_metrics["Product_ID"].nunique()
)

display(product_metrics.head())

TRUE PRODUCT-GRAIN DATASET
Rows              : 764
Unique Product_IDs: 764
One row per product: True


,Product_ID,Months_Present,Records,Net_Units,Revenue,Cost_Sales,Profit,Gross_Margin_%
0,1,1,1,1,165.83,123.97,41.86,25.24
1,2,1,1,1,307.50,269.48,38.02,12.36
2,3,1,1,1,290.83,269.51,21.32,7.33
3,4,1,1,1,165.83,150.92,14.91,8.99
4,5,1,1,1,57.50,45.90,11.60,20.17


In [55]:
# ============================================================
#  ATTACH AUTHORITATIVE PRODUCT ATTRIBUTES
# ============================================================

product_attributes = (
    dim_product[
        [
            "Product_ID",
            "Product_Key",
            "Product_Description",
            "Product_Category"
        ]
    ]
    .copy()
)

product_sales_clean = (
    product_metrics
    .merge(
        product_attributes,
        on="Product_ID",
        how="left",
        validate="one_to_one"
    )
)

# Reorder columns
product_sales_clean = product_sales_clean[
    [
        "Product_ID",
        "Product_Key",
        "Product_Description",
        "Product_Category",
        "Months_Present",
        "Records",
        "Net_Units",
        "Revenue",
        "Cost_Sales",
        "Profit",
        "Gross_Margin_%"
    ]
]

print("FINAL MERCHANDISE PRODUCT PERFORMANCE DATASET")
print("=" * 65)

print(f"Rows              : {len(product_sales_clean):,}")
print(
    f"Unique Product_IDs: "
    f"{product_sales_clean['Product_ID'].nunique():,}"
)

print(
    "Duplicate Product_IDs:",
    product_sales_clean["Product_ID"].duplicated().sum()
)

print(
    "Missing descriptions:",
    product_sales_clean["Product_Description"].isna().sum()
)

print(
    "Missing categories:",
    product_sales_clean["Product_Category"].isna().sum()
)

display(product_sales_clean.head(10))

FINAL MERCHANDISE PRODUCT PERFORMANCE DATASET
Rows              : 764
Unique Product_IDs: 764
Duplicate Product_IDs: 0
Missing descriptions: 0
Missing categories: 0


,Product_ID,Product_Key,Product_Description,Product_Category,Months_Present,Records,Net_Units,Revenue,Cost_Sales,Profit,Gross_Margin_%
0,1,010-02384-10,Garmin Lily Cream Gold & White,FITNESS,1,1,1,165.83,123.97,41.86,25.24
1,2,010-02784-00,Garmin Venu 3 Smartwatch - Silver,FITNESS,1,1,1,307.50,269.48,38.02,12.36
2,3,010-02784-01,Garmin Venu 3 Smartwatch - Slate,FITNESS,1,1,1,290.83,269.51,21.32,7.33
3,4,010-02839-00,"Garmin Lily 2, Cream Gold w/",FITNESS,1,1,1,165.83,150.92,14.91,8.99
4,5,01950,NUTRIBULLET PRO 4pc Starter Kit,BLENDERS,1,1,1,57.50,45.90,11.60,20.17
5,6,10009310,Miele GGRP Gourmet Griddle Plate,WHITES ACCESSORIES,1,1,1,83.33,130.96,-47.63,-57.16
6,7,10107860,Miele SF-AP 50 Air Clean Plus Filter,VACUUM ACCESSORIES,1,1,1,15.00,11.41,3.59,23.93
7,8,10234470,Miele Nature Flacon,WHITES ACCESSORIES,1,1,1,9.16,5.01,4.15,45.31
8,9,102785,"MR Equip Metallic Red, 1.5 litre, 3kw",KETTLES,1,1,1,21.67,0.00,21.67,100.00
9,10,103414675,ZAGG Pro Keys 2-Apple iPad Pro 13,IT ACCESSORIES,1,1,1,82.50,76.17,6.33,7.67


In [56]:

# ============================================================
#  FINAL MERCHANDISE PRODUCT KPI SUMMARY
# ============================================================

total_products = product_sales_clean["Product_ID"].nunique()
total_units = product_sales_clean["Net_Units"].sum()
total_revenue = product_sales_clean["Revenue"].sum()
total_cost_sales = product_sales_clean["Cost_Sales"].sum()
total_profit = product_sales_clean["Profit"].sum()

overall_margin = (
    total_profit / total_revenue * 100
    if total_revenue != 0
    else np.nan
)

product_kpi_summary = pd.DataFrame({
    "KPI": [
        "Merchandise Products",
        "Net Units",
        "Revenue",
        "Cost of Sales",
        "Profit",
        "Gross Margin %"
    ],
    "Value": [
        total_products,
        total_units,
        total_revenue,
        total_cost_sales,
        total_profit,
        overall_margin
    ]
})

print("FINAL MERCHANDISE PRODUCT KPIs")
print("=" * 60)

display(product_kpi_summary.round(2))

FINAL MERCHANDISE PRODUCT KPIs


,KPI,Value
0,Merchandise Products,764.00
1,Net Units,"1,127.00"
2,Revenue,"277,773.35"
3,Cost of Sales,"273,671.44"
4,Profit,"4,101.91"
5,Gross Margin %,1.48


In [57]:
# ============================================================
#  TOP PRODUCTS BY REVENUE
# ============================================================

top_revenue_products = (
    product_sales_clean[
        [
            "Product_ID",
            "Product_Key",
            "Product_Description",
            "Product_Category",
            "Net_Units",
            "Revenue",
            "Profit",
            "Gross_Margin_%"
        ]
    ]
    .sort_values("Revenue", ascending=False)
    .head(20)
)

print("TOP 20 MERCHANDISE PRODUCTS BY REVENUE")
print("=" * 60)

display(top_revenue_products.round(2))

TOP 20 MERCHANDISE PRODUCTS BY REVENUE


,Product_ID,Product_Key,Product_Description,Product_Category,Net_Units,Revenue,Profit,Gross_Margin_%
631,1167,T2351V11,Eufy Robot Vaccum X10 Pro Omni,ROBOT CLEANING,16,"6,818.30",433.98,6.36
98,116,42120,Novy Panorama 120 Pro 5 Zone,DOWNDRAFT HOBS,1,"4,000.00",815.90,20.40
506,870,OLED65G54L,LG 65in G5 OLED TV,TV 60 - 70,2,"3,165.00",-146.22,-4.62
708,1320,WEK365WCS,Miele (12392780) 10kg 1400 Spin Washer,WASHING MACHINES,3,"2,832.49",439.81,15.53
653,1213,TEH785WP,Miele 12736710 9kg Heat Pump Dryer,TUMBLE DRYERS,3,"2,764.17",201.15,7.28
347,560,H7464BPBL,Miele Black 11093600 Pyro Single Oven,SINGLE OVENS,2,"2,683.34",101.00,3.76
756,1425,XRFSD5265,Liebherr SXS,FRIDGE FREEZERS,1,"2,666.67",539.12,20.22
501,856,OLED55G54L,LG 55in G5 OLED TV,TV 51 - 59,2,"2,581.67",43.01,1.67
563,1037,RS70F64KEF,Samsung Black St/St USA FF,USA F/F,3,"2,580.83",-392.11,-15.19
675,1246,U1ACE2AG3BNEFF,N50 Graphite Double Oven,DOUBLE OVENS,4,"2,538.33",208.65,8.22


In [58]:
# ============================================================
#  TOP PRODUCTS BY NET UNITS
# ============================================================

top_unit_products = (
    product_sales_clean[
        [
            "Product_ID",
            "Product_Key",
            "Product_Description",
            "Product_Category",
            "Net_Units",
            "Revenue",
            "Profit",
            "Gross_Margin_%"
        ]
    ]
    .sort_values(
        ["Net_Units", "Revenue"],
        ascending=[False, False]
    )
    .head(20)
)

print("TOP 20 MERCHANDISE PRODUCTS BY NET UNITS")
print("=" * 60)

display(top_unit_products.round(2))

TOP 20 MERCHANDISE PRODUCTS BY NET UNITS


,Product_ID,Product_Key,Product_Description,Product_Category,Net_Units,Revenue,Profit,Gross_Margin_%
631,1167,T2351V11,Eufy Robot Vaccum X10 Pro Omni,ROBOT CLEANING,16,"6,818.30",433.98,6.36
448,767,M,MISCELANEOUS,MISC,12,"-3,954.19","-3,954.31",100.00
699,1289,VS15A6031R4SAMSUNG,Jet 60 Cordless Vacuum,STICK VACS,9,"1,102.02",18.42,1.67
103,121,43LQ60006LA.LG,"43"" Smart TV",TV 33 - 43,8,"1,254.99",-241.01,-19.20
511,886,P-SDU32GU18PNY,Elite microSDHC card 32G,IT ACCESSORIES,8,39.90,13.26,33.23
452,771,MC1001UK,Ninja 8-in-1 Slow Cooker,FOOD PREP,7,761.65,47.72,6.27
431,721,KN650A,Kenwood Electric Knife | KN650A,FOOD PREP,7,171.64,64.05,37.32
325,525,GN BAGS,BAGS 400/600/800 SERIES AND S5,VACUUM BAGS,7,77.55,24.42,31.49
126,170,55NANO81A6,LG 55 NANO TV,TV 51 - 59,5,"1,662.50",-7.90,-0.48
684,1262,UE55U7000FKSAMSUNG,55 in Smart Television,TV 51 - 59,5,"1,304.17",-416.43,-31.93


In [59]:
# ============================================================
#  TOP PRODUCTS BY PROFIT
# ============================================================

top_profit_products = (
    product_sales_clean[
        [
            "Product_ID",
            "Product_Key",
            "Product_Description",
            "Product_Category",
            "Net_Units",
            "Revenue",
            "Profit",
            "Gross_Margin_%"
        ]
    ]
    .sort_values("Profit", ascending=False)
    .head(20)
)

print("TOP 20 MERCHANDISE PRODUCTS BY PROFIT")
print("=" * 60)

display(top_profit_products.round(2))

TOP 20 MERCHANDISE PRODUCTS BY PROFIT


,Product_ID,Product_Key,Product_Description,Product_Category,Net_Units,Revenue,Profit,Gross_Margin_%
255,395,DV90DG52A0,Samsung Series 5 Tumble Dryer,TUMBLE DRYERS,2,"1,020.00","1,020.00",100.00
98,116,42120,Novy Panorama 120 Pro 5 Zone,DOWNDRAFT HOBS,1,"4,000.00",815.90,20.40
564,1039,RS70F64KET,Samsung USA Fridge Freezer,USA F/F,1,790.83,790.83,100.00
694,1280,USG10TY.DG,LG 3.1 Wireless Soundbar,SPEAKERS,4,"2,246.66",605.34,26.94
756,1425,XRFSD5265,Liebherr SXS,FRIDGE FREEZERS,1,"2,666.67",539.12,20.22
747,1407,WW80CGC04,SAMSUNG Series 5 AI Energy Washing,WASHING MACHINES,2,515.00,515.00,100.00
633,1169,T2353V11,eufy Robot Vacuum Omni E25,ROBOT CLEANING,3,"1,765.01",451.61,25.59
708,1320,WEK365WCS,Miele (12392780) 10kg 1400 Spin Washer,WASHING MACHINES,3,"2,832.49",439.81,15.53
631,1167,T2351V11,Eufy Robot Vaccum X10 Pro Omni,ROBOT CLEANING,16,"6,818.30",433.98,6.36
321,520,G7085SCVIXXMIELE,12865050 Integrated,INT DISHWASHERS,2,"2,183.34",427.22,19.57


In [60]:
# ============================================================
#  LOSS-MAKING MERCHANDISE PRODUCTS
# ============================================================

loss_products_clean = (
    product_sales_clean[
        product_sales_clean["Profit"] < 0
    ]
    .sort_values("Profit")
    .copy()
)

print(
    f"Loss-making merchandise products: "
    f"{len(loss_products_clean):,}"
)

display(
    loss_products_clean[
        [
            "Product_ID",
            "Product_Key",
            "Product_Description",
            "Product_Category",
            "Months_Present",
            "Net_Units",
            "Revenue",
            "Cost_Sales",
            "Profit",
            "Gross_Margin_%"
        ]
    ]
    .head(20)
    .round(2)
)

Loss-making merchandise products: 220


,Product_ID,Product_Key,Product_Description,Product_Category,Months_Present,Net_Units,Revenue,Cost_Sales,Profit,Gross_Margin_%
448,767,M,MISCELANEOUS,MISC,2,12,"-3,954.19",0.12,"-3,954.31",100.00
507,878,OLED83C44L,"LG 83"" OLED Television",TV 75+,1,1,"2,332.50","3,774.00","-1,441.50",-61.80
658,1226,TOLP110DFF,Rangemaster Toledo 110 Ind St/St,RANGE COOKERS,1,1,"1,666.67","2,435.13",-768.46,-46.11
500,853,OLED55G45L,"LG 55"" OLED Television",TV 44 - 50,1,1,770.00,"1,471.35",-701.35,-91.08
504,864,OLED65C44L,"LG 65"" OLED Television",TV 60 - 70,1,1,990.83,"1,671.10",-680.27,-68.66
544,978,QE55QN90FA,"Samsung 55"" Neo QLED 4K",TV 51 - 59,2,2,"1,540.00","2,210.22",-670.22,-43.52
548,988,QE65S85FAE,"Samsung 65"" OLED 4K Smart TV",TV 60+,1,1,"1,028.33","1,579.01",-550.68,-53.55
745,1402,WW11DB8B95SAMSUNG,Blk Stl Series 8 11Kg 1400,WASHING MACHINES,2,3,"1,863.34","2,314.98",-451.64,-24.24
503,863,OLED65B56LALG,65in B5 OLED TV,TV 60 - 70,1,1,832.50,"1,260.63",-428.13,-51.43
684,1262,UE55U7000FKSAMSUNG,55 in Smart Television,TV 51 - 59,2,5,"1,304.17","1,720.60",-416.43,-31.93


In [61]:
# ============================================================
#  PRODUCT PROFITABILITY DISTRIBUTION
# ============================================================

PROFIT_TOLERANCE = 0.01

profitable_count = (
    product_sales_clean["Profit"] > PROFIT_TOLERANCE
).sum()

break_even_count = (
    product_sales_clean["Profit"].abs() <= PROFIT_TOLERANCE
).sum()

loss_count = (
    product_sales_clean["Profit"] < -PROFIT_TOLERANCE
).sum()

profitability_summary = pd.DataFrame({
    "Metric": [
        "Total Merchandise Products",
        "Profitable Products",
        "Break-even Products",
        "Loss-making Products"
    ],
    "Count": [
        len(product_sales_clean),
        profitable_count,
        break_even_count,
        loss_count
    ]
})

profitability_summary["Share_%"] = (
    profitability_summary["Count"]
    / len(product_sales_clean)
    * 100
)

print("MERCHANDISE PRODUCT PROFITABILITY")
print("=" * 60)

display(profitability_summary.round(2))

print(
    "\nPopulation reconciles:",
    profitable_count + break_even_count + loss_count
    == len(product_sales_clean)
)

MERCHANDISE PRODUCT PROFITABILITY


,Metric,Count,Share_%
0,Total Merchandise Products,764,100.00
1,Profitable Products,530,69.37
2,Break-even Products,14,1.83
3,Loss-making Products,220,28.80



Population reconciles: True


In [62]:
# ============================================================
# PRODUCT LOSS CONCENTRATION
# ============================================================

product_loss_concentration = (
    product_sales_clean[
        product_sales_clean["Profit"] < -PROFIT_TOLERANCE
    ]
    .sort_values("Profit")
    .copy()
)

total_product_loss = (
    -product_loss_concentration["Profit"].sum()
)

product_loss_concentration["Loss_Amount"] = (
    -product_loss_concentration["Profit"]
)

product_loss_concentration["Loss_Contribution_%"] = (
    product_loss_concentration["Loss_Amount"]
    / total_product_loss
    * 100
)

product_loss_concentration["Cumulative_Loss_%"] = (
    product_loss_concentration[
        "Loss_Contribution_%"
    ].cumsum()
)

display(
    product_loss_concentration[
        [
            "Product_Key",
            "Product_Description",
            "Product_Category",
            "Revenue",
            "Profit",
            "Loss_Contribution_%",
            "Cumulative_Loss_%"
        ]
    ]
    .head(20)
    .round(2)
)

,Product_Key,Product_Description,Product_Category,Revenue,Profit,Loss_Contribution_%,Cumulative_Loss_%
448,M,MISCELANEOUS,MISC,"-3,954.19","-3,954.31",15.93,15.93
507,OLED83C44L,"LG 83"" OLED Television",TV 75+,"2,332.50","-1,441.50",5.81,21.74
658,TOLP110DFF,Rangemaster Toledo 110 Ind St/St,RANGE COOKERS,"1,666.67",-768.46,3.10,24.84
500,OLED55G45L,"LG 55"" OLED Television",TV 44 - 50,770.00,-701.35,2.83,27.66
504,OLED65C44L,"LG 65"" OLED Television",TV 60 - 70,990.83,-680.27,2.74,30.41
544,QE55QN90FA,"Samsung 55"" Neo QLED 4K",TV 51 - 59,"1,540.00",-670.22,2.70,33.11
548,QE65S85FAE,"Samsung 65"" OLED 4K Smart TV",TV 60+,"1,028.33",-550.68,2.22,35.33
745,WW11DB8B95SAMSUNG,Blk Stl Series 8 11Kg 1400,WASHING MACHINES,"1,863.34",-451.64,1.82,37.15
503,OLED65B56LALG,65in B5 OLED TV,TV 60 - 70,832.50,-428.13,1.73,38.87
684,UE55U7000FKSAMSUNG,55 in Smart Television,TV 51 - 59,"1,304.17",-416.43,1.68,40.55


In [63]:
# ============================================================
#  PRODUCT REVENUE CONCENTRATION
# ============================================================

revenue_concentration = (
    product_sales_clean[
        product_sales_clean["Revenue"] > 0
    ]
    .sort_values("Revenue", ascending=False)
    .copy()
)

positive_revenue_total = (
    revenue_concentration["Revenue"].sum()
)

revenue_concentration["Revenue_Contribution_%"] = (
    revenue_concentration["Revenue"]
    / positive_revenue_total
    * 100
)

revenue_concentration["Cumulative_Revenue_%"] = (
    revenue_concentration[
        "Revenue_Contribution_%"
    ].cumsum()
)

display(
    revenue_concentration[
        [
            "Product_Key",
            "Product_Description",
            "Product_Category",
            "Net_Units",
            "Revenue",
            "Revenue_Contribution_%",
            "Cumulative_Revenue_%"
        ]
    ]
    .head(20)
    .round(2)
)

,Product_Key,Product_Description,Product_Category,Net_Units,Revenue,Revenue_Contribution_%,Cumulative_Revenue_%
631,T2351V11,Eufy Robot Vaccum X10 Pro Omni,ROBOT CLEANING,16,"6,818.30",2.41,2.41
98,42120,Novy Panorama 120 Pro 5 Zone,DOWNDRAFT HOBS,1,"4,000.00",1.41,3.82
506,OLED65G54L,LG 65in G5 OLED TV,TV 60 - 70,2,"3,165.00",1.12,4.93
708,WEK365WCS,Miele (12392780) 10kg 1400 Spin Washer,WASHING MACHINES,3,"2,832.49",1.00,5.93
653,TEH785WP,Miele 12736710 9kg Heat Pump Dryer,TUMBLE DRYERS,3,"2,764.17",0.98,6.91
347,H7464BPBL,Miele Black 11093600 Pyro Single Oven,SINGLE OVENS,2,"2,683.34",0.95,7.86
756,XRFSD5265,Liebherr SXS,FRIDGE FREEZERS,1,"2,666.67",0.94,8.80
501,OLED55G54L,LG 55in G5 OLED TV,TV 51 - 59,2,"2,581.67",0.91,9.71
563,RS70F64KEF,Samsung Black St/St USA FF,USA F/F,3,"2,580.83",0.91,10.62
675,U1ACE2AG3BNEFF,N50 Graphite Double Oven,DOUBLE OVENS,4,"2,538.33",0.90,11.52


In [64]:
# ============================================================
# FINAL PRODUCT ANALYSIS INTEGRITY GATE
# ============================================================

checks = {
    "One row per Product_ID":
        len(product_sales_clean)
        == product_sales_clean["Product_ID"].nunique(),

    "No duplicate Product_ID":
        product_sales_clean["Product_ID"].duplicated().sum() == 0,

    "No missing Product_Key":
        product_sales_clean["Product_Key"].isna().sum() == 0,

    "No missing descriptions":
        product_sales_clean["Product_Description"].isna().sum() == 0,

    "No missing categories":
        product_sales_clean["Product_Category"].isna().sum() == 0,

    "Delivery excluded":
        370 not in product_sales_clean["Product_ID"].values,

    "Installation excluded":
        615 not in product_sales_clean["Product_ID"].values,

    "Profitability population reconciles":
        profitable_count + break_even_count + loss_count
        == len(product_sales_clean)
}

validation_results = pd.DataFrame({
    "Check": checks.keys(),
    "Passed": checks.values()
})

display(validation_results)

print("\n" + "=" * 60)

if validation_results["Passed"].all():
    print("PASS — Final merchandise product analysis validated.")
else:
    print("WARNING — One or more validation checks failed.")

,Check,Passed
0,One row per Product_ID,True
1,No duplicate Product_ID,True
2,No missing Product_Key,True
3,No missing descriptions,True
4,No missing categories,True
5,Delivery excluded,True
6,Installation excluded,True
7,Profitability population reconciles,True



PASS — Final merchandise product analysis validated.


In [67]:
# ============================================================
#  INVESTIGATE M / MISCELLANEOUS RECORD
# ============================================================
fact_sales = pd.read_csv("D:/STUDY/Data_Science_Courses/PROJECTS/12.Smart_AI-Retail_System/Smart_AI_Retail_System/data/processed/fact_sales.csv")
m_record = fact_sales[
    (fact_sales["Product_ID"] == 767) |
    (fact_sales["Product_Key"].astype(str).str.strip().str.upper() == "M")
].copy()

print("M / MISCELLANEOUS SOURCE RECORDS")
print("=" * 70)

print(f"Rows        : {len(m_record):,}")
print(f"Months      : {m_record['Source_Month'].nunique()}")
print(f"Net units   : {m_record['Sold Period'].sum():,.0f}")
print(f"Sales value : £{m_record['Sales Value'].sum():,.2f}")
print(f"Cost sales  : £{m_record['Cost Sales'].sum():,.2f}")
print(f"Profit      : £{m_record['Profit'].sum():,.2f}")

display(
    m_record.sort_values("Source_Month")
)

M / MISCELLANEOUS SOURCE RECORDS
Rows        : 2
Months      : 2
Net units   : 12
Sales value : £-3,954.19
Cost sales  : £0.12
Profit      : £-3,954.31


,Sales_Record_ID,Product_ID,Product_Key,Source_Month,Category,Stock Code,Description,Record_Type,Level,Sold Period,Transaction_Status,Unit Cost,Unit Price,Cost Sales,Sales Value,Profit,Profit %,Cost_Sales_Reconciliation_Flag
570,571,767,M,Dec,MISC,M,MISCELANEOUS,PRODUCT,-22,8,POSITIVE_SALES_ACTIVITY,0.01,0.49,0.08,150.00,149.92,99.95,MATCH
856,857,767,M,Jan,MISC,M,MISCELANEOUS,PRODUCT,-22,4,POSITIVE_SALES_ACTIVITY,0.01,0.49,0.04,"-4,104.19","-4,104.23",100.00,MATCH


In [68]:
# ============================================================
#  ZERO / NEAR-ZERO COST MERCHANDISE
# ============================================================

zero_cost_products = (
    product_sales_clean[
        (product_sales_clean["Net_Units"] > 0) &
        (product_sales_clean["Revenue"] > 0) &
        (product_sales_clean["Cost_Sales"].abs() <= 0.01)
    ]
    .sort_values("Revenue", ascending=False)
    .copy()
)

print("ZERO / NEAR-ZERO COST MERCHANDISE")
print("=" * 70)

print(f"Products : {len(zero_cost_products):,}")
print(
    f"Revenue  : £{zero_cost_products['Revenue'].sum():,.2f}"
)

display(
    zero_cost_products[
        [
            "Product_ID",
            "Product_Key",
            "Product_Description",
            "Product_Category",
            "Months_Present",
            "Net_Units",
            "Revenue",
            "Cost_Sales",
            "Profit",
            "Gross_Margin_%"
        ]
    ].round(2)
)

ZERO / NEAR-ZERO COST MERCHANDISE
Products : 9
Revenue  : £2,530.40


,Product_ID,Product_Key,Product_Description,Product_Category,Months_Present,Net_Units,Revenue,Cost_Sales,Profit,Gross_Margin_%
255,395,DV90DG52A0,Samsung Series 5 Tumble Dryer,TUMBLE DRYERS,2,2,"1,020.00",0.00,"1,020.00",100.00
564,1039,RS70F64KET,Samsung USA Fridge Freezer,USA F/F,1,1,790.83,0.00,790.83,100.00
747,1407,WW80CGC04,SAMSUNG Series 5 AI Energy Washing,WASHING MACHINES,1,2,515.00,0.00,515.00,100.00
605,1121,SM-A175FZKB,Samsung Galaxy A17 Black 128GB,TELEPHONES,1,1,107.50,0.00,107.50,100.00
68,80,25810,George Foreman Fit Grill Medium,SANDWICH TOASTERS,1,2,26.25,0.00,26.25,100.00
8,9,102785,"MR Equip Metallic Red, 1.5 litre, 3kw",KETTLES,1,1,21.67,0.00,21.67,100.00
559,1022,RHESB6001,Russell Hobbs Single Blanket,ELECTRIC BLANKETS,1,1,20.83,0.00,20.83,100.00
199,301,BHR8776GL,WBW Redmi Buds 6 Play Black,HEADPHONES,1,1,19.99,0.00,19.99,100.00
337,544,GVMA003WE,Groove 3mt USB-C to USB-A Cable,CABLES,1,1,8.33,0.00,8.33,100.00


In [69]:
# ============================================================
# TRACE ZERO-COST PRODUCTS TO FACT_SALES
# ============================================================

zero_cost_ids = zero_cost_products["Product_ID"].tolist()

zero_cost_source = (
    fact_sales[
        fact_sales["Product_ID"].isin(zero_cost_ids)
    ]
    .sort_values(
        ["Product_ID", "Source_Month"]
    )
)

print("SOURCE RECORDS FOR ZERO-COST MERCHANDISE")
print("=" * 70)

display(zero_cost_source)

SOURCE RECORDS FOR ZERO-COST MERCHANDISE


,Sales_Record_ID,Product_ID,Product_Key,Source_Month,Category,Stock Code,Description,Record_Type,Level,Sold Period,Transaction_Status,Unit Cost,Unit Price,Cost Sales,Sales Value,Profit,Profit %,Cost_Sales_Reconciliation_Flag
837,838,9,102785,Jan,KETTLES,102785,"MR Equip Metallic Red, 1.5 litre, 3kw",PRODUCT,3,1,POSITIVE_SALES_ACTIVITY,0.00,33.99,0.00,21.67,21.67,100.00,MATCH
869,870,80,25810,Jan,SANDWICH TOASTERS,25810,George Foreman Fit Grill Medium,PRODUCT,1,2,POSITIVE_SALES_ACTIVITY,0.00,46.99,0.00,26.25,26.25,100.00,MATCH
508,509,301,BHR8776GL,Dec,HEADPHONES,BHR8776GL,WBW Redmi Buds 6 Play Black,PRODUCT,4,1,POSITIVE_SALES_ACTIVITY,0.00,0.49,0.00,19.99,19.99,100.00,MATCH
919,920,395,DV90DG52A0,Jan,TUMBLE DRYERS,DV90DG52A0,Samsung Series 5 Tumble Dryer,PRODUCT,1,1,POSITIVE_SALES_ACTIVITY,0.00,0.49,0.00,562.50,562.50,100.00,MATCH
294,295,395,DV90DG52A0,Nov,TUMBLE DRYERS,DV90DG52A0,Samsung Series 5 Tumble Dryer,PRODUCT,1,1,POSITIVE_SALES_ACTIVITY,0.00,0.49,0.00,457.50,457.50,100.00,MATCH
729,730,544,GVMA003WE,Jan,CABLES,GVMA003WE,Groove 3mt USB-C to USB-A Cable,PRODUCT,2,1,POSITIVE_SALES_ACTIVITY,0.00,0.49,0.00,8.33,8.33,100.00,MATCH
73,74,1022,RHESB6001,Nov,ELECTRIC BLANKETS,RHESB6001,Russell Hobbs Single Blanket,PRODUCT,1,1,POSITIVE_SALES_ACTIVITY,0.00,29.99,0.00,20.83,20.83,100.00,MATCH
349,350,1039,RS70F64KET,Nov,USA F/F,RS70F64KET,Samsung USA Fridge Freezer,PRODUCT,1,1,POSITIVE_SALES_ACTIVITY,0.00,0.49,0.00,790.83,790.83,100.00,MATCH
631,632,1121,SM-A175FZKB,Dec,TELEPHONES,SM-A175FZKB,Samsung Galaxy A17 Black 128GB,PRODUCT,1,1,POSITIVE_SALES_ACTIVITY,0.00,0.49,0.00,107.50,107.50,100.00,MATCH
701,702,1407,WW80CGC04,Dec,WASHING MACHINES,WW80CGC04,SAMSUNG Series 5 AI Energy Washing,PRODUCT,0,2,POSITIVE_SALES_ACTIVITY,0.00,0.49,0.00,515.00,515.00,100.00,MATCH


In [70]:
# ============================================================
# PRODUCT ANALYTICAL CLASSIFICATION
# ============================================================

product_sales_final = product_sales_clean.copy()

# ------------------------------------------------------------
# 1. Identify non-standard / adjustment product records
# ------------------------------------------------------------

product_sales_final["Analytical_Record_Type"] = "MERCHANDISE"

product_sales_final.loc[
    product_sales_final["Product_Key"]
        .astype(str)
        .str.strip()
        .str.upper()
        .eq("M"),
    "Analytical_Record_Type"
] = "MISC_ADJUSTMENT"


# ------------------------------------------------------------
# 2. Profitability data-quality classification
# ------------------------------------------------------------

product_sales_final["Profitability_Data_Quality"] = "VALID"

zero_cost_mask = (
    product_sales_final["Net_Units"].gt(0) &
    product_sales_final["Revenue"].gt(0) &
    product_sales_final["Cost_Sales"].abs().le(0.01)
)

product_sales_final.loc[
    zero_cost_mask,
    "Profitability_Data_Quality"
] = "ZERO_COST"


# ------------------------------------------------------------
# 3. Create analytical merchandise population
# ------------------------------------------------------------

merchandise_analysis = product_sales_final[
    product_sales_final["Analytical_Record_Type"]
        .eq("MERCHANDISE")
].copy()


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("PRODUCT ANALYTICAL CLASSIFICATION")
print("=" * 70)

print(
    product_sales_final["Analytical_Record_Type"]
    .value_counts(dropna=False)
)

print()

print(
    merchandise_analysis["Profitability_Data_Quality"]
    .value_counts(dropna=False)
)

print("\nAnalytical merchandise products :",
      f"{len(merchandise_analysis):,}")

print(
    "Valid profitability products   :",
    f"{(merchandise_analysis['Profitability_Data_Quality'] == 'VALID').sum():,}"
)

print(
    "Zero-cost flagged products      :",
    f"{(merchandise_analysis['Profitability_Data_Quality'] == 'ZERO_COST').sum():,}"
)

PRODUCT ANALYTICAL CLASSIFICATION
Analytical_Record_Type
MERCHANDISE        763
MISC_ADJUSTMENT      1
Name: count, dtype: int64

Profitability_Data_Quality
VALID        754
ZERO_COST      9
Name: count, dtype: int64

Analytical merchandise products : 763
Valid profitability products   : 754
Zero-cost flagged products      : 9


In [71]:
# ============================================================
# PROFITABILITY-VALID ANALYTICAL POPULATION
# ============================================================

profitability_analysis = merchandise_analysis[
    merchandise_analysis["Profitability_Data_Quality"].eq("VALID")
].copy()


# ------------------------------------------------------------
# Core reconciliation
# ------------------------------------------------------------

print("PROFITABILITY ANALYTICAL POPULATION")
print("=" * 70)

print(
    "Analytical merchandise products :",
    f"{len(merchandise_analysis):,}"
)

print(
    "Profitability-valid products     :",
    f"{len(profitability_analysis):,}"
)

print(
    "Excluded zero-cost products      :",
    f"{(merchandise_analysis['Profitability_Data_Quality'] == 'ZERO_COST').sum():,}"
)

print()

print(
    "Population reconciles            :",
    len(profitability_analysis)
    + (merchandise_analysis["Profitability_Data_Quality"] == "ZERO_COST").sum()
    == len(merchandise_analysis)
)


# ------------------------------------------------------------
# Profitability KPIs
# ------------------------------------------------------------

valid_revenue = profitability_analysis["Revenue"].sum()
valid_cost = profitability_analysis["Cost_Sales"].sum()
valid_profit = profitability_analysis["Profit"].sum()

valid_margin = (
    valid_profit / valid_revenue * 100
    if valid_revenue != 0
    else float("nan")
)

print("\nVALIDATED PROFITABILITY KPIs")
print("=" * 70)

print(f"Revenue       : £{valid_revenue:,.2f}")
print(f"Cost of Sales : £{valid_cost:,.2f}")
print(f"Profit        : £{valid_profit:,.2f}")
print(f"Gross Margin  : {valid_margin:.2f}%")

PROFITABILITY ANALYTICAL POPULATION
Analytical merchandise products : 763
Profitability-valid products     : 754
Excluded zero-cost products      : 9

Population reconciles            : True

VALIDATED PROFITABILITY KPIs
Revenue       : £279,197.14
Cost of Sales : £273,671.32
Profit        : £5,525.82
Gross Margin  : 1.98%


In [72]:
# ============================================================
#  FINAL PROFITABILITY INTEGRITY GATE
# ============================================================

import numpy as np
import pandas as pd

print("FINAL PROFITABILITY INTEGRITY GATE")
print("=" * 75)

# ------------------------------------------------------------
# 1. Population / grain checks
# ------------------------------------------------------------

rows = len(profitability_analysis)
unique_ids = profitability_analysis["Product_ID"].nunique()
duplicate_ids = profitability_analysis["Product_ID"].duplicated().sum()

print("\n1. PRODUCT GRAIN")
print("-" * 75)
print(f"Rows                    : {rows:,}")
print(f"Unique Product_IDs      : {unique_ids:,}")
print(f"Duplicate Product_IDs   : {duplicate_ids:,}")

grain_pass = (
    rows == unique_ids
    and duplicate_ids == 0
)


# ------------------------------------------------------------
# 2. Analytical classification checks
# ------------------------------------------------------------

non_merchandise = (
    profitability_analysis["Analytical_Record_Type"]
    .ne("MERCHANDISE")
    .sum()
)

invalid_quality = (
    profitability_analysis["Profitability_Data_Quality"]
    .ne("VALID")
    .sum()
)

print("\n2. ANALYTICAL CLASSIFICATION")
print("-" * 75)
print(f"Non-merchandise records : {non_merchandise:,}")
print(f"Invalid quality records : {invalid_quality:,}")

classification_pass = (
    non_merchandise == 0
    and invalid_quality == 0
)


# ------------------------------------------------------------
# 3. Zero / near-zero cost check
# ------------------------------------------------------------

zero_cost_mask = (
    profitability_analysis["Net_Units"].gt(0)
    & profitability_analysis["Revenue"].gt(0)
    & profitability_analysis["Cost_Sales"].abs().le(0.01)
)

zero_cost_remaining = zero_cost_mask.sum()

print("\n3. ZERO-COST CHECK")
print("-" * 75)
print(f"Zero-cost products remaining : {zero_cost_remaining:,}")

zero_cost_pass = zero_cost_remaining == 0


# ------------------------------------------------------------
# 4. Missing critical fields
# ------------------------------------------------------------

critical_columns = [
    "Product_ID",
    "Product_Key",
    "Product_Description",
    "Product_Category",
    "Net_Units",
    "Revenue",
    "Cost_Sales",
    "Profit"
]

missing_counts = (
    profitability_analysis[critical_columns]
    .isna()
    .sum()
)

print("\n4. MISSING CRITICAL VALUES")
print("-" * 75)
print(missing_counts)

missing_pass = missing_counts.sum() == 0


# ------------------------------------------------------------
# 5. Financial arithmetic reconciliation
# ------------------------------------------------------------

expected_profit = (
    profitability_analysis["Revenue"]
    - profitability_analysis["Cost_Sales"]
)

profit_difference = (
    profitability_analysis["Profit"]
    - expected_profit
).abs()

financial_mismatches = (profit_difference > 0.02).sum()
max_difference = profit_difference.max()

print("\n5. FINANCIAL RECONCILIATION")
print("-" * 75)
print(
    "Rows where Profit != Revenue - Cost_Sales :",
    f"{financial_mismatches:,}"
)
print(
    "Maximum absolute difference               :",
    f"£{max_difference:,.4f}"
)

financial_pass = financial_mismatches == 0


# ------------------------------------------------------------
# 6. Aggregate financial reconciliation
# ------------------------------------------------------------

total_revenue = profitability_analysis["Revenue"].sum()
total_cost = profitability_analysis["Cost_Sales"].sum()
total_profit = profitability_analysis["Profit"].sum()

aggregate_difference = abs(
    total_profit - (total_revenue - total_cost)
)

aggregate_pass = aggregate_difference <= 0.02

print("\n6. AGGREGATE RECONCILIATION")
print("-" * 75)
print(f"Revenue             : £{total_revenue:,.2f}")
print(f"Cost of Sales       : £{total_cost:,.2f}")
print(f"Profit              : £{total_profit:,.2f}")
print(
    f"Reconciliation diff : £{aggregate_difference:,.4f}"
)


# ------------------------------------------------------------
# 7. Final gate
# ------------------------------------------------------------

checks = {
    "Product grain": grain_pass,
    "Analytical classification": classification_pass,
    "Zero-cost exclusion": zero_cost_pass,
    "Critical-field completeness": missing_pass,
    "Row-level financial reconciliation": financial_pass,
    "Aggregate financial reconciliation": aggregate_pass
}

gate_results = pd.DataFrame(
    {
        "Check": checks.keys(),
        "Status": [
            "PASS" if result else "FAIL"
            for result in checks.values()
        ]
    }
)

print("\nFINAL GATE RESULTS")
print("=" * 75)

display(gate_results)

overall_pass = all(checks.values())

print("\nOVERALL STATUS")
print("=" * 75)

if overall_pass:
    print(
        "PASS — profitability_analysis is analytically valid "
        "for downstream profitability analysis."
    )
else:
    print(
        "FAIL — resolve failed integrity checks before "
        "using profitability_analysis downstream."
    )

FINAL PROFITABILITY INTEGRITY GATE

1. PRODUCT GRAIN
---------------------------------------------------------------------------
Rows                    : 754
Unique Product_IDs      : 754
Duplicate Product_IDs   : 0

2. ANALYTICAL CLASSIFICATION
---------------------------------------------------------------------------
Non-merchandise records : 0
Invalid quality records : 0

3. ZERO-COST CHECK
---------------------------------------------------------------------------
Zero-cost products remaining : 0

4. MISSING CRITICAL VALUES
---------------------------------------------------------------------------
Product_ID             0
Product_Key            0
Product_Description    0
Product_Category       0
Net_Units              0
Revenue                0
Cost_Sales             0
Profit                 0
dtype: int64

5. FINANCIAL RECONCILIATION
---------------------------------------------------------------------------
Rows where Profit != Revenue - Cost_Sales : 0
Maximum absolute differe

,Check,Status
0,Product grain,PASS
1,Analytical classification,PASS
2,Zero-cost exclusion,PASS
3,Critical-field completeness,PASS
4,Row-level financial reconciliation,PASS
5,Aggregate financial reconciliation,PASS



OVERALL STATUS
PASS — profitability_analysis is analytically valid for downstream profitability analysis.


In [73]:
# ============================================================
# GOVERNED MERCHANDISE KPI SUMMARY
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# SALES / VOLUME KPIs
# Population: merchandise_analysis
# ------------------------------------------------------------

merchandise_products = merchandise_analysis["Product_ID"].nunique()
net_units = merchandise_analysis["Net_Units"].sum()
sales_revenue = merchandise_analysis["Revenue"].sum()


# ------------------------------------------------------------
# PROFITABILITY KPIs
# Population: profitability_analysis
# ------------------------------------------------------------

profitability_products = profitability_analysis["Product_ID"].nunique()

profitability_revenue = profitability_analysis["Revenue"].sum()
cost_of_sales = profitability_analysis["Cost_Sales"].sum()
gross_profit = profitability_analysis["Profit"].sum()

gross_margin_pct = (
    gross_profit / profitability_revenue * 100
    if profitability_revenue != 0
    else np.nan
)


# ------------------------------------------------------------
# DATA-QUALITY EXCLUSIONS
# ------------------------------------------------------------

zero_cost_products = (
    merchandise_analysis["Profitability_Data_Quality"]
    .eq("ZERO_COST")
    .sum()
)

zero_cost_revenue = merchandise_analysis.loc[
    merchandise_analysis["Profitability_Data_Quality"].eq("ZERO_COST"),
    "Revenue"
].sum()


# ------------------------------------------------------------
# GOVERNED KPI TABLE
# ------------------------------------------------------------

governed_kpis = pd.DataFrame({

    "KPI": [
        "Merchandise Products",
        "Net Units",
        "Merchandise Revenue",
        "Profitability-valid Products",
        "Profitability-valid Revenue",
        "Cost of Sales",
        "Gross Profit",
        "Gross Margin %",
        "Zero-cost Flagged Products",
        "Zero-cost Revenue"
    ],

    "Value": [
        merchandise_products,
        net_units,
        sales_revenue,
        profitability_products,
        profitability_revenue,
        cost_of_sales,
        gross_profit,
        gross_margin_pct,
        zero_cost_products,
        zero_cost_revenue
    ],

    "Population": [
        "MERCHANDISE",
        "MERCHANDISE",
        "MERCHANDISE",
        "PROFITABILITY_VALID",
        "PROFITABILITY_VALID",
        "PROFITABILITY_VALID",
        "PROFITABILITY_VALID",
        "PROFITABILITY_VALID",
        "DATA_QUALITY",
        "DATA_QUALITY"
    ]
})


print("GOVERNED MERCHANDISE KPI LAYER")
print("=" * 75)

display(governed_kpis)

GOVERNED MERCHANDISE KPI LAYER


,KPI,Value,Population
0,Merchandise Products,763.00,MERCHANDISE
1,Net Units,"1,115.00",MERCHANDISE
2,Merchandise Revenue,"281,727.54",MERCHANDISE
3,Profitability-valid Products,754.00,PROFITABILITY_VALID
4,Profitability-valid Revenue,"279,197.14",PROFITABILITY_VALID
5,Cost of Sales,"273,671.32",PROFITABILITY_VALID
6,Gross Profit,"5,525.82",PROFITABILITY_VALID
7,Gross Margin %,1.98,PROFITABILITY_VALID
8,Zero-cost Flagged Products,9.00,DATA_QUALITY
9,Zero-cost Revenue,"2,530.40",DATA_QUALITY


In [74]:
# ============================================================
#  GOVERNED KPI RECONCILIATION GATE
# ============================================================

checks = []

# 1. Product population
checks.append({
    "Check": "Merchandise product population",
    "Status": "PASS"
    if merchandise_products == 763 else "FAIL"
})

# 2. Profitability population
checks.append({
    "Check": "Profitability-valid population",
    "Status": "PASS"
    if profitability_products == 754 else "FAIL"
})

# 3. Population bridge
checks.append({
    "Check": "Population bridge",
    "Status": "PASS"
    if merchandise_products ==
       profitability_products + zero_cost_products
    else "FAIL"
})

# 4. Revenue bridge
revenue_bridge_diff = (
    sales_revenue
    - zero_cost_revenue
    - profitability_revenue
)

checks.append({
    "Check": "Revenue population bridge",
    "Status": "PASS"
    if abs(revenue_bridge_diff) < 0.01 else "FAIL"
})

# 5. Profit reconciliation
profit_diff = (
    profitability_revenue
    - cost_of_sales
    - gross_profit
)

checks.append({
    "Check": "Profit reconciliation",
    "Status": "PASS"
    if abs(profit_diff) < 0.01 else "FAIL"
})

# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

kpi_governance_gate = pd.DataFrame(checks)

print("GOVERNED KPI RECONCILIATION GATE")
print("=" * 75)

display(kpi_governance_gate)

print()
print(f"Revenue bridge difference : £{revenue_bridge_diff:,.4f}")
print(f"Profit reconciliation diff: £{profit_diff:,.4f}")

overall_status = (
    "PASS"
    if (kpi_governance_gate["Status"] == "PASS").all()
    else "FAIL"
)

print()
print("OVERALL KPI GOVERNANCE STATUS")
print("=" * 75)
print(overall_status)

GOVERNED KPI RECONCILIATION GATE


,Check,Status
0,Merchandise product population,PASS
1,Profitability-valid population,PASS
2,Population bridge,PASS
3,Revenue population bridge,PASS
4,Profit reconciliation,PASS



Revenue bridge difference : £-0.0000
Profit reconciliation diff: £0.0000

OVERALL KPI GOVERNANCE STATUS
PASS


### Step 5 — Revenue Performance Analysis

Analyse revenue performance across the governed merchandise population.

Population rule:
- Include: Analytical_Record_Type == "MERCHANDISE"
- Revenue analysis includes zero-cost flagged products because the cost-quality
  issue does not invalidate their recorded sales revenue.
- Profitability exclusions are therefore not applied to revenue KPIs.

In [78]:
# ============================================================
# REVENUE ANALYSIS POPULATION
# ============================================================

revenue_analysis = merchandise_analysis.copy()

print("REVENUE ANALYSIS POPULATION")
print("=" * 75)

print(f"Products           : {len(revenue_analysis):,}")
print(
    f"Unique Product_IDs : "
    f"{revenue_analysis['Product_ID'].nunique():,}"
)
print(
    f"Net Units          : "
    f"{revenue_analysis['Net_Units'].sum():,.0f}"
)
print(
    f"Revenue            : "
    f"£{revenue_analysis['Revenue'].sum():,.2f}"
)

print(
    "\nPopulation valid:",
    len(revenue_analysis)
    == revenue_analysis["Product_ID"].nunique()
)

display(
    revenue_analysis[
        [
            "Product_ID",
            "Product_Key",
            "Product_Description",
            "Product_Category",
            "Months_Present",
            "Net_Units",
            "Revenue"
        ]
    ]
    .sort_values("Revenue", ascending=False)
    .head(10)
)

REVENUE ANALYSIS POPULATION
Products           : 763
Unique Product_IDs : 763
Net Units          : 1,115
Revenue            : £281,727.54

Population valid: True


,Product_ID,Product_Key,Product_Description,Product_Category,Months_Present,Net_Units,Revenue
631,1167,T2351V11,Eufy Robot Vaccum X10 Pro Omni,ROBOT CLEANING,3,16,"6,818.30"
98,116,42120,Novy Panorama 120 Pro 5 Zone,DOWNDRAFT HOBS,1,1,"4,000.00"
506,870,OLED65G54L,LG 65in G5 OLED TV,TV 60 - 70,2,2,"3,165.00"
708,1320,WEK365WCS,Miele (12392780) 10kg 1400 Spin Washer,WASHING MACHINES,1,3,"2,832.49"
653,1213,TEH785WP,Miele 12736710 9kg Heat Pump Dryer,TUMBLE DRYERS,2,3,"2,764.17"
347,560,H7464BPBL,Miele Black 11093600 Pyro Single Oven,SINGLE OVENS,1,2,"2,683.34"
756,1425,XRFSD5265,Liebherr SXS,FRIDGE FREEZERS,1,1,"2,666.67"
501,856,OLED55G54L,LG 55in G5 OLED TV,TV 51 - 59,2,2,"2,581.67"
563,1037,RS70F64KEF,Samsung Black St/St USA FF,USA F/F,3,3,"2,580.83"
675,1246,U1ACE2AG3BNEFF,N50 Graphite Double Oven,DOUBLE OVENS,3,4,"2,538.33"


#### Step 5.1 - Revenue Concentration & Pareto Analysis

Measure how merchandise revenue is distributed across products.

Objectives:
- Rank products by revenue.
- Calculate each product's contribution to total merchandise revenue.
- Calculate cumulative revenue contribution.
- Measure the number of products required to generate 50%, 80%, and 90%
  of total merchandise revenue.
- Identify whether revenue is concentrated in a relatively small product group.

In [79]:
# ============================================================
#  REVENUE CONCENTRATION / PARETO ANALYSIS
# ============================================================

revenue_pareto = (
    revenue_analysis
    .sort_values("Revenue", ascending=False)
    .reset_index(drop=True)
    .copy()
)

total_revenue = revenue_pareto["Revenue"].sum()
total_products = len(revenue_pareto)

# ------------------------------------------------------------
# Revenue contribution
# ------------------------------------------------------------

revenue_pareto["Revenue_Contribution_%"] = (
    revenue_pareto["Revenue"] / total_revenue * 100
)

revenue_pareto["Cumulative_Revenue_%"] = (
    revenue_pareto["Revenue_Contribution_%"].cumsum()
)

revenue_pareto["Product_Rank"] = (
    range(1, total_products + 1)
)

# ------------------------------------------------------------
# Concentration thresholds
# ------------------------------------------------------------

def products_to_reach_threshold(df, threshold):
    reached = df[
        df["Cumulative_Revenue_%"] >= threshold
    ]

    if len(reached) == 0:
        return None

    return int(reached.iloc[0]["Product_Rank"])


n_50 = products_to_reach_threshold(revenue_pareto, 50)
n_80 = products_to_reach_threshold(revenue_pareto, 80)
n_90 = products_to_reach_threshold(revenue_pareto, 90)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("REVENUE CONCENTRATION / PARETO ANALYSIS")
print("=" * 75)

print(f"Total products : {total_products:,}")
print(f"Total revenue  : £{total_revenue:,.2f}")

print("\nPRODUCTS REQUIRED TO GENERATE REVENUE")
print("-" * 75)

for threshold, n in [(50, n_50), (80, n_80), (90, n_90)]:

    if n is not None:
        share_products = n / total_products * 100

        print(
            f"{threshold}% of revenue : "
            f"{n:,} products "
            f"({share_products:.2f}% of merchandise products)"
        )

# ------------------------------------------------------------
# Top-N concentration
# ------------------------------------------------------------

print("\nTOP PRODUCT REVENUE CONCENTRATION")
print("-" * 75)

for n in [10, 20, 50, 100]:

    n_actual = min(n, total_products)

    contribution = (
        revenue_pareto
        .head(n_actual)["Revenue"]
        .sum()
        / total_revenue
        * 100
    )

    print(
        f"Top {n_actual:>3} products : "
        f"{contribution:>6.2f}% of revenue"
    )

# ------------------------------------------------------------
# Display leading products
# ------------------------------------------------------------

display(
    revenue_pareto[
        [
            "Product_Rank",
            "Product_ID",
            "Product_Key",
            "Product_Description",
            "Product_Category",
            "Net_Units",
            "Revenue",
            "Revenue_Contribution_%",
            "Cumulative_Revenue_%"
        ]
    ].head(20)
)

REVENUE CONCENTRATION / PARETO ANALYSIS
Total products : 763
Total revenue  : £281,727.54

PRODUCTS REQUIRED TO GENERATE REVENUE
---------------------------------------------------------------------------
50% of revenue : 89 products (11.66% of merchandise products)
80% of revenue : 236 products (30.93% of merchandise products)
90% of revenue : 336 products (44.04% of merchandise products)

TOP PRODUCT REVENUE CONCENTRATION
---------------------------------------------------------------------------
Top  10 products :  11.58% of revenue
Top  20 products :  19.55% of revenue
Top  50 products :  35.85% of revenue
Top 100 products :  53.36% of revenue


,Product_Rank,Product_ID,Product_Key,Product_Description,Product_Category,Net_Units,Revenue,Revenue_Contribution_%,Cumulative_Revenue_%
0,1,1167,T2351V11,Eufy Robot Vaccum X10 Pro Omni,ROBOT CLEANING,16,"6,818.30",2.42,2.42
1,2,116,42120,Novy Panorama 120 Pro 5 Zone,DOWNDRAFT HOBS,1,"4,000.00",1.42,3.84
2,3,870,OLED65G54L,LG 65in G5 OLED TV,TV 60 - 70,2,"3,165.00",1.12,4.96
3,4,1320,WEK365WCS,Miele (12392780) 10kg 1400 Spin Washer,WASHING MACHINES,3,"2,832.49",1.01,5.97
4,5,1213,TEH785WP,Miele 12736710 9kg Heat Pump Dryer,TUMBLE DRYERS,3,"2,764.17",0.98,6.95
5,6,560,H7464BPBL,Miele Black 11093600 Pyro Single Oven,SINGLE OVENS,2,"2,683.34",0.95,7.90
6,7,1425,XRFSD5265,Liebherr SXS,FRIDGE FREEZERS,1,"2,666.67",0.95,8.85
7,8,856,OLED55G54L,LG 55in G5 OLED TV,TV 51 - 59,2,"2,581.67",0.92,9.77
8,9,1037,RS70F64KEF,Samsung Black St/St USA FF,USA F/F,3,"2,580.83",0.92,10.68
9,10,1246,U1ACE2AG3BNEFF,N50 Graphite Double Oven,DOUBLE OVENS,4,"2,538.33",0.90,11.58


#### Step 5.2  — Product Revenue Distribution

Segment merchandise products into revenue-performance bands to understand
the distribution of revenue across the product portfolio.

Revenue bands are descriptive analytical groups and are not yet ABC
inventory classifications.

In [81]:
# ============================================================
#  PRODUCT REVENUE DISTRIBUTION
# ============================================================

revenue_distribution = revenue_analysis.copy()

# ------------------------------------------------------------
# Revenue bands
# ------------------------------------------------------------

def classify_revenue(value):

    if value < 0:
        return "NEGATIVE_REVENUE"
    elif value == 0:
        return "ZERO_REVENUE"
    elif value < 100:
        return "£0–£99"
    elif value < 500:
        return "£100–£499"
    elif value < 1000:
        return "£500–£999"
    elif value < 2000:
        return "£1,000–£1,999"
    else:
        return "£2,000+"

revenue_distribution["Revenue_Band"] = (
    revenue_distribution["Revenue"].apply(classify_revenue)
)

band_order = [
    "NEGATIVE_REVENUE",
    "ZERO_REVENUE",
    "£0–£99",
    "£100–£499",
    "£500–£999",
    "£1,000–£1,999",
    "£2,000+"
]

# ------------------------------------------------------------
# Band summary
# ------------------------------------------------------------

revenue_band_summary = (
    revenue_distribution
    .groupby("Revenue_Band", observed=True)
    .agg(
        Products=("Product_ID", "nunique"),
        Net_Units=("Net_Units", "sum"),
        Revenue=("Revenue", "sum")
    )
    .reindex(band_order, fill_value=0)
    .reset_index()
)

revenue_band_summary["Product_Share_%"] = (
    revenue_band_summary["Products"]
    / len(revenue_distribution)
    * 100
)

revenue_band_summary["Revenue_Share_%"] = (
    revenue_band_summary["Revenue"]
    / revenue_distribution["Revenue"].sum()
    * 100
)

# ------------------------------------------------------------
# Portfolio statistics
# ------------------------------------------------------------

print("PRODUCT REVENUE DISTRIBUTION")
print("=" * 75)

print(f"Products             : {len(revenue_distribution):,}")
print(
    f"Positive revenue     : "
    f"{(revenue_distribution['Revenue'] > 0).sum():,}"
)
print(
    f"Zero revenue         : "
    f"{(revenue_distribution['Revenue'] == 0).sum():,}"
)
print(
    f"Negative revenue     : "
    f"{(revenue_distribution['Revenue'] < 0).sum():,}"
)

print("\nREVENUE STATISTICS")
print("-" * 75)

print(
    f"Mean revenue/product   : "
    f"£{revenue_distribution['Revenue'].mean():,.2f}"
)

print(
    f"Median revenue/product : "
    f"£{revenue_distribution['Revenue'].median():,.2f}"
)

print(
    f"Maximum revenue        : "
    f"£{revenue_distribution['Revenue'].max():,.2f}"
)

print(
    f"Minimum revenue        : "
    f"£{revenue_distribution['Revenue'].min():,.2f}"
)

print("\nREVENUE BANDS")
print("-" * 75)

display(
    revenue_band_summary.style.format({
        "Net_Units": "{:,.0f}",
        "Revenue": "£{:,.2f}",
        "Product_Share_%": "{:.2f}%",
        "Revenue_Share_%": "{:.2f}%"
    })
)

PRODUCT REVENUE DISTRIBUTION
Products             : 763
Positive revenue     : 751
Zero revenue         : 5
Negative revenue     : 7

REVENUE STATISTICS
---------------------------------------------------------------------------
Mean revenue/product   : £369.24
Median revenue/product : £165.00
Maximum revenue        : £6,818.30
Minimum revenue        : £-750.00

REVENUE BANDS
---------------------------------------------------------------------------


,Revenue_Band,Products,Net_Units,Revenue,Product_Share_%,Revenue_Share_%
0,NEGATIVE_REVENUE,7,-10,"£-1,641.61",0.92%,-0.58%
1,ZERO_REVENUE,5,0,£0.00,0.66%,0.00%
2,£0–£99,297,426,"£11,824.29",38.93%,4.20%
3,£100–£499,277,370,"£71,450.92",36.30%,25.36%
4,£500–£999,101,154,"£69,876.09",13.24%,24.80%
5,"£1,000–£1,999",56,119,"£75,133.70",7.34%,26.67%
6,"£2,000+",20,56,"£55,084.15",2.62%,19.55%


#### Step 5.3 — Non-Positive Revenue Investigation

Investigate merchandise products with zero or negative aggregate revenue.

Purpose:
- Separate normal positive-sales performance from returns/reversals.
- Identify products with zero-revenue activity.
- Prevent non-positive transactional behaviour from being incorrectly
  interpreted as ordinary product underperformance.

In [82]:
# ============================================================
#  NON-POSITIVE REVENUE INVESTIGATION
# ============================================================

non_positive_revenue = (
    revenue_analysis.loc[
        revenue_analysis["Revenue"] <= 0
    ]
    .copy()
    .sort_values("Revenue")
)

negative_revenue = non_positive_revenue.loc[
    non_positive_revenue["Revenue"] < 0
].copy()

zero_revenue = non_positive_revenue.loc[
    non_positive_revenue["Revenue"] == 0
].copy()


print("NON-POSITIVE REVENUE INVESTIGATION")
print("=" * 75)

print(f"Total affected products : {len(non_positive_revenue):,}")
print(f"Negative-revenue products: {len(negative_revenue):,}")
print(f"Zero-revenue products    : {len(zero_revenue):,}")

print("\nNEGATIVE-REVENUE SUMMARY")
print("-" * 75)

print(
    f"Net units : "
    f"{negative_revenue['Net_Units'].sum():,.0f}"
)

print(
    f"Revenue   : "
    f"£{negative_revenue['Revenue'].sum():,.2f}"
)

print("\nZERO-REVENUE SUMMARY")
print("-" * 75)

print(
    f"Net units : "
    f"{zero_revenue['Net_Units'].sum():,.0f}"
)

print("\nAFFECTED PRODUCTS")
print("-" * 75)

display(
    non_positive_revenue[
        [
            "Product_ID",
            "Product_Key",
            "Product_Description",
            "Product_Category",
            "Months_Present",
            "Net_Units",
            "Revenue"
        ]
    ]
)

NON-POSITIVE REVENUE INVESTIGATION
Total affected products : 12
Negative-revenue products: 7
Zero-revenue products    : 5

NEGATIVE-REVENUE SUMMARY
---------------------------------------------------------------------------
Net units : -10
Revenue   : £-1,641.61

ZERO-REVENUE SUMMARY
---------------------------------------------------------------------------
Net units : 0

AFFECTED PRODUCTS
---------------------------------------------------------------------------


,Product_ID,Product_Key,Product_Description,Product_Category,Months_Present,Net_Units,Revenue
630,1166,T2080GA1,Eufy Robot Vacuum S1 Pro,ROBOT CLEANING,1,-1,-750.00
128,181,561727-01,Dyson Supersonic Nural Strawberry,HAIRCARE,1,-1,-332.50
76,89,26771,Russell Hobbs Bronte Toaster Stone,TOASTERS,1,-5,-249.95
723,1366,WHULT900NBSONY,"Black Bluetooth 5.2, NC",HEADPHONES,1,-1,-99.17
227,354,CTI4003.M,DeLonghi Distinta X 4SL Toaster,TOASTERS,1,-1,-75.00
620,1153,SP01Z01Z321,Nokia C22 - 2+64GB Black,TELEPHONES,1,-1,-74.99
715,1339,WGG254Z0GB*BOSCH,Series 6 10kg 1400 Spin,WASHING MACHINES,2,0,-60.00
175,268,B2ACH7AG7BNEFF,N50 Graphite Grey Single Oven,SINGLE OVENS,2,0,0.00
601,1117,SKE735BTR4,Sage Black Truffle Kettle,KETTLES,2,0,0.00
464,788,MQ3025,Braun 700W Handblender,FOOD PREP,2,0,0.00


#### Step 5.4 — Revenue Activity Classification

Classify merchandise products according to their aggregate revenue activity.

Classes:
- POSITIVE_REVENUE — net positive merchandise revenue.
- ZERO_REVENUE — no net revenue contribution.
- NET_RETURN_REVERSAL — negative aggregate revenue associated with
  net-negative unit activity.

This classification preserves returns/reversals within the governed
merchandise population while preventing them from being interpreted
as ordinary positive-sales product performance.

In [83]:
# ============================================================
#  REVENUE ACTIVITY CLASSIFICATION
# ============================================================

revenue_analysis = revenue_analysis.copy()

# ------------------------------------------------------------
# Classification
# ------------------------------------------------------------

def classify_revenue_activity(row):

    revenue = row["Revenue"]
    units = row["Net_Units"]

    if revenue < 0 and units < 0:
        return "NET_RETURN_REVERSAL"

    elif revenue == 0 and units == 0:
        return "ZERO_REVENUE"

    elif revenue > 0:
        return "POSITIVE_REVENUE"

    else:
        return "REVIEW_REQUIRED"


revenue_analysis["Revenue_Activity_Type"] = (
    revenue_analysis.apply(
        classify_revenue_activity,
        axis=1
    )
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

revenue_activity_summary = (
    revenue_analysis
    .groupby("Revenue_Activity_Type")
    .agg(
        Products=("Product_ID", "nunique"),
        Net_Units=("Net_Units", "sum"),
        Revenue=("Revenue", "sum")
    )
    .reset_index()
)

revenue_activity_summary["Product_Share_%"] = (
    revenue_activity_summary["Products"]
    / len(revenue_analysis)
    * 100
)

revenue_activity_summary["Revenue_Share_%"] = (
    revenue_activity_summary["Revenue"]
    / revenue_analysis["Revenue"].sum()
    * 100
)

print("REVENUE ACTIVITY CLASSIFICATION")
print("=" * 75)

display(
    revenue_activity_summary.style.format({
        "Net_Units": "{:,.0f}",
        "Revenue": "£{:,.2f}",
        "Product_Share_%": "{:.2f}%",
        "Revenue_Share_%": "{:.2f}%"
    })
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

review_required = (
    revenue_analysis["Revenue_Activity_Type"]
    .eq("REVIEW_REQUIRED")
    .sum()
)

classified_products = revenue_activity_summary["Products"].sum()

print("\nCLASSIFICATION VALIDATION")
print("-" * 75)

print(f"Merchandise products : {len(revenue_analysis):,}")
print(f"Classified products  : {classified_products:,}")
print(f"Review required      : {review_required:,}")

print(
    "Population reconciles:",
    classified_products == len(revenue_analysis)
)

# Show unexpected combinations if any
if review_required > 0:

    print("\nRECORDS REQUIRING REVIEW")
    print("-" * 75)

    display(
        revenue_analysis.loc[
            revenue_analysis["Revenue_Activity_Type"]
            .eq("REVIEW_REQUIRED"),
            [
                "Product_ID",
                "Product_Key",
                "Product_Description",
                "Product_Category",
                "Net_Units",
                "Revenue"
            ]
        ]
    )

REVENUE ACTIVITY CLASSIFICATION


,Revenue_Activity_Type,Products,Net_Units,Revenue,Product_Share_%,Revenue_Share_%
0,NET_RETURN_REVERSAL,6,-10,"£-1,581.61",0.79%,-0.56%
1,POSITIVE_REVENUE,751,"1,125","£283,369.15",98.43%,100.58%
2,REVIEW_REQUIRED,1,0,£-60.00,0.13%,-0.02%
3,ZERO_REVENUE,5,0,£0.00,0.66%,0.00%



CLASSIFICATION VALIDATION
---------------------------------------------------------------------------
Merchandise products : 763
Classified products  : 763
Review required      : 1
Population reconciles: True

RECORDS REQUIRING REVIEW
---------------------------------------------------------------------------


,Product_ID,Product_Key,Product_Description,Product_Category,Net_Units,Revenue
715,1339,WGG254Z0GB*BOSCH,Series 6 10kg 1400 Spin,WASHING MACHINES,0,-60.00


In [84]:
# ============================================================
#  INVESTIGATE REVIEW_REQUIRED PRODUCT
# ============================================================

review_ids = revenue_analysis.loc[
    revenue_analysis["Revenue_Activity_Type"].eq("REVIEW_REQUIRED"),
    "Product_ID"
].unique()

review_source = fact_sales[
    fact_sales["Product_ID"].isin(review_ids)
].copy()

print("REVIEW_REQUIRED — SOURCE RECORD INVESTIGATION")
print("=" * 80)

print(f"Products requiring review : {len(review_ids):,}")
print(f"Source records            : {len(review_source):,}")

display(
    review_source.sort_values(
        ["Product_ID", "Source_Month"]
    )
)

# ------------------------------------------------------------
# Financial summary by product
# ------------------------------------------------------------

review_summary = (
    review_source
    .groupby("Product_ID", as_index=False)
    .agg(
        Source_Records=("Product_ID", "size"),
        Net_Units=("Sold Period", "sum"),
        Revenue=("Sales Value", "sum"),
        Cost_Sales=("Cost Sales", "sum")
    )
)

print("\nSOURCE-LEVEL RECONCILIATION")
print("=" * 80)

display(
    review_summary.style.format({
        "Net_Units": "{:,.0f}",
        "Revenue": "£{:,.2f}",
        "Cost_Sales": "£{:,.2f}"
    })
)

REVIEW_REQUIRED — SOURCE RECORD INVESTIGATION
Products requiring review : 1
Source records            : 2


,Sales_Record_ID,Product_ID,Product_Key,Source_Month,Category,Stock Code,Description,Record_Type,Level,Sold Period,Transaction_Status,Unit Cost,Unit Price,Cost Sales,Sales Value,Profit,Profit %,Cost_Sales_Reconciliation_Flag
955,956,1339,WGG254Z0GB*BOSCH,Jan,WASHING MACHINES,WGG254Z0GB*Bosch,Series 6 10kg 1400 Spin,PRODUCT,0,1,POSITIVE_SALES_ACTIVITY,398.65,602.99,398.65,380.83,-17.82,-4.68,MATCH
360,361,1339,WGG254Z0GB*BOSCH,Nov,WASHING MACHINES,WGG254Z0GB*Bosch,Series 6 10kg 1400 Spin,PRODUCT,0,-1,NEGATIVE_SALES_ACTIVITY,398.65,602.99,-398.65,-440.83,-42.18,9.57,MATCH



SOURCE-LEVEL RECONCILIATION


,Product_ID,Source_Records,Net_Units,Revenue,Cost_Sales
0,1339,2,0,£-60.00,£0.00


In [85]:
# ============================================================
# CREVIEW PRODUCT TRANSACTION DIAGNOSTIC
# ============================================================

review_detail = fact_sales.loc[
    fact_sales["Product_ID"].eq(1339)
].copy()

# Show only diagnostically useful fields that actually exist
wanted_cols = [
    "Source_Month",
    "Product_ID",
    "Product_Key",
    "Stock Code",
    "Description",
    "Level",
    "Sold Period",
    "Sales Value",
    "Cost Sales",
    "Profit",
    "Transaction_Status",
    "Record_Type"
]

available_cols = [
    col for col in wanted_cols
    if col in review_detail.columns
]

print("PRODUCT 1339 — MONTHLY TRANSACTION DIAGNOSTIC")
print("=" * 80)

display(
    review_detail[available_cols]
    .sort_values("Source_Month")
    .reset_index(drop=True)
)

print("\nMONTHLY NUMERIC RECONCILIATION")
print("=" * 80)

for _, row in review_detail.sort_values("Source_Month").iterrows():

    print(f"\nMonth       : {row['Source_Month']}")

    for col in [
        "Level",
        "Sold Period",
        "Sales Value",
        "Cost Sales",
        "Profit",
        "Transaction_Status"
    ]:
        if col in review_detail.columns:
            print(f"{col:<12}: {row[col]}")

PRODUCT 1339 — MONTHLY TRANSACTION DIAGNOSTIC


,Source_Month,Product_ID,Product_Key,Stock Code,Description,Level,Sold Period,Sales Value,Cost Sales,Profit,Transaction_Status,Record_Type
0,Jan,1339,WGG254Z0GB*BOSCH,WGG254Z0GB*Bosch,Series 6 10kg 1400 Spin,0,1,380.83,398.65,-17.82,POSITIVE_SALES_ACTIVITY,PRODUCT
1,Nov,1339,WGG254Z0GB*BOSCH,WGG254Z0GB*Bosch,Series 6 10kg 1400 Spin,0,-1,-440.83,-398.65,-42.18,NEGATIVE_SALES_ACTIVITY,PRODUCT



MONTHLY NUMERIC RECONCILIATION

Month       : Jan
Level       : 0
Sold Period : 1
Sales Value : 380.83
Cost Sales  : 398.65
Profit      : -17.82
Transaction_Status: POSITIVE_SALES_ACTIVITY

Month       : Nov
Level       : 0
Sold Period : -1
Sales Value : -440.83
Cost Sales  : -398.65
Profit      : -42.18
Transaction_Status: NEGATIVE_SALES_ACTIVITY


In [86]:
# ============================================================
#  RESOLVE REVERSAL VALUE VARIANCE
# ============================================================

# Identify products where:
#   1. Net units = 0
#   2. Revenue != 0
#   3. Underlying records contain both positive and negative units
#
# These represent offsetting unit transactions whose monetary
# values do not fully reverse.

def classify_reversal_variance(product_id):

    src = fact_sales[
        fact_sales["Product_ID"].eq(product_id)
    ]

    has_positive_units = (src["Sold Period"] > 0).any()
    has_negative_units = (src["Sold Period"] < 0).any()

    return has_positive_units and has_negative_units


review_mask = (
    revenue_analysis["Revenue_Activity_Type"]
    .eq("REVIEW_REQUIRED")
)

for idx in revenue_analysis.index[review_mask]:

    product_id = revenue_analysis.loc[idx, "Product_ID"]

    if classify_reversal_variance(product_id):

        revenue_analysis.loc[
            idx,
            "Revenue_Activity_Type"
        ] = "REVERSAL_VALUE_VARIANCE"


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

classification_summary = (
    revenue_analysis
    .groupby("Revenue_Activity_Type", as_index=False)
    .agg(
        Products=("Product_ID", "count"),
        Net_Units=("Net_Units", "sum"),
        Revenue=("Revenue", "sum")
    )
)

classification_summary["Product_Share_%"] = (
    classification_summary["Products"]
    / len(revenue_analysis)
    * 100
)

classification_summary["Revenue_Share_%"] = (
    classification_summary["Revenue"]
    / revenue_analysis["Revenue"].sum()
    * 100
)

print("FINAL REVENUE ACTIVITY CLASSIFICATION")
print("=" * 80)

display(
    classification_summary.style.format({
        "Net_Units": "{:,.0f}",
        "Revenue": "£{:,.2f}",
        "Product_Share_%": "{:.2f}%",
        "Revenue_Share_%": "{:.2f}%"
    })
)

print("\nCLASSIFICATION CONTROL")
print("=" * 80)

print(
    "Unresolved REVIEW_REQUIRED :",
    (
        revenue_analysis["Revenue_Activity_Type"]
        == "REVIEW_REQUIRED"
    ).sum()
)

print(
    "Reversal value variances  :",
    (
        revenue_analysis["Revenue_Activity_Type"]
        == "REVERSAL_VALUE_VARIANCE"
    ).sum()
)

print(
    "Population reconciles     :",
    len(revenue_analysis) == 763
)

FINAL REVENUE ACTIVITY CLASSIFICATION


,Revenue_Activity_Type,Products,Net_Units,Revenue,Product_Share_%,Revenue_Share_%
0,NET_RETURN_REVERSAL,6,-10,"£-1,581.61",0.79%,-0.56%
1,POSITIVE_REVENUE,751,"1,125","£283,369.15",98.43%,100.58%
2,REVERSAL_VALUE_VARIANCE,1,0,£-60.00,0.13%,-0.02%
3,ZERO_REVENUE,5,0,£0.00,0.66%,0.00%



CLASSIFICATION CONTROL
Unresolved REVIEW_REQUIRED : 0
Reversal value variances  : 1
Population reconciles     : True


In [87]:
# ============================================================
# FINAL REVENUE ANALYSIS INTEGRITY GATE
# ============================================================

print("FINAL REVENUE ANALYSIS INTEGRITY GATE")
print("=" * 80)

# ------------------------------------------------------------
# 1. PRODUCT GRAIN
# ------------------------------------------------------------

rows = len(revenue_analysis)
unique_products = revenue_analysis["Product_ID"].nunique()
duplicate_products = revenue_analysis["Product_ID"].duplicated().sum()

print("\n1. PRODUCT GRAIN")
print("-" * 80)

print(f"Rows                  : {rows:,}")
print(f"Unique Product_IDs    : {unique_products:,}")
print(f"Duplicate Product_IDs : {duplicate_products:,}")

grain_pass = (
    rows == unique_products
    and duplicate_products == 0
)

# ------------------------------------------------------------
# 2. CLASSIFICATION COMPLETENESS
# ------------------------------------------------------------

allowed_classes = {
    "POSITIVE_REVENUE",
    "NET_RETURN_REVERSAL",
    "REVERSAL_VALUE_VARIANCE",
    "ZERO_REVENUE"
}

actual_classes = set(
    revenue_analysis["Revenue_Activity_Type"]
    .dropna()
    .unique()
)

missing_classification = (
    revenue_analysis["Revenue_Activity_Type"]
    .isna()
    .sum()
)

unexpected_classes = actual_classes - allowed_classes

print("\n2. REVENUE ACTIVITY CLASSIFICATION")
print("-" * 80)

print(f"Missing classifications : {missing_classification:,}")
print(f"Unexpected classes      : {unexpected_classes}")

classification_pass = (
    missing_classification == 0
    and len(unexpected_classes) == 0
)

# ------------------------------------------------------------
# 3. CRITICAL FIELD COMPLETENESS
# ------------------------------------------------------------

critical_fields = [
    "Product_ID",
    "Product_Key",
    "Product_Description",
    "Product_Category",
    "Net_Units",
    "Revenue",
    "Revenue_Activity_Type"
]

missing_critical = (
    revenue_analysis[critical_fields]
    .isna()
    .sum()
)

print("\n3. CRITICAL FIELD COMPLETENESS")
print("-" * 80)

print(missing_critical)

critical_pass = (missing_critical.sum() == 0)

# ------------------------------------------------------------
# 4. CLASSIFICATION LOGIC
# ------------------------------------------------------------

positive_invalid = revenue_analysis.loc[
    revenue_analysis["Revenue_Activity_Type"].eq("POSITIVE_REVENUE")
    & (revenue_analysis["Revenue"] <= 0)
]

zero_invalid = revenue_analysis.loc[
    revenue_analysis["Revenue_Activity_Type"].eq("ZERO_REVENUE")
    & (~np.isclose(revenue_analysis["Revenue"], 0))
]

return_invalid = revenue_analysis.loc[
    revenue_analysis["Revenue_Activity_Type"].eq("NET_RETURN_REVERSAL")
    & (
        (revenue_analysis["Net_Units"] >= 0)
        | (revenue_analysis["Revenue"] >= 0)
    )
]

variance_invalid = revenue_analysis.loc[
    revenue_analysis["Revenue_Activity_Type"].eq(
        "REVERSAL_VALUE_VARIANCE"
    )
    & (
        (~np.isclose(revenue_analysis["Net_Units"], 0))
        | (np.isclose(revenue_analysis["Revenue"], 0))
    )
]

print("\n4. CLASSIFICATION LOGIC")
print("-" * 80)

print(f"Invalid positive revenue       : {len(positive_invalid):,}")
print(f"Invalid zero revenue           : {len(zero_invalid):,}")
print(f"Invalid net-return reversal    : {len(return_invalid):,}")
print(f"Invalid reversal-value variance: {len(variance_invalid):,}")

logic_pass = (
    len(positive_invalid) == 0
    and len(zero_invalid) == 0
    and len(return_invalid) == 0
    and len(variance_invalid) == 0
)

# ------------------------------------------------------------
# 5. GOVERNED KPI RECONCILIATION
# ------------------------------------------------------------

analysis_products = len(revenue_analysis)
analysis_units = revenue_analysis["Net_Units"].sum()
analysis_revenue = revenue_analysis["Revenue"].sum()

expected_products = 763
expected_units = 1115
expected_revenue = 281727.54

product_diff = analysis_products - expected_products
unit_diff = analysis_units - expected_units
revenue_diff = analysis_revenue - expected_revenue

print("\n5. GOVERNED KPI RECONCILIATION")
print("-" * 80)

print(f"Products       : {analysis_products:,}")
print(f"Net Units      : {analysis_units:,.0f}")
print(f"Revenue        : £{analysis_revenue:,.2f}")

print(f"\nProduct diff   : {product_diff:,}")
print(f"Net-unit diff  : {unit_diff:,.0f}")
print(f"Revenue diff   : £{revenue_diff:,.4f}")

reconciliation_pass = (
    product_diff == 0
    and np.isclose(unit_diff, 0)
    and np.isclose(revenue_diff, 0, atol=0.01)
)

# ------------------------------------------------------------
# 6. FINAL GATE
# ------------------------------------------------------------

gate_results = pd.DataFrame({
    "Check": [
        "Product grain",
        "Classification completeness",
        "Critical-field completeness",
        "Classification logic",
        "Governed KPI reconciliation"
    ],
    "Status": [
        "PASS" if grain_pass else "FAIL",
        "PASS" if classification_pass else "FAIL",
        "PASS" if critical_pass else "FAIL",
        "PASS" if logic_pass else "FAIL",
        "PASS" if reconciliation_pass else "FAIL"
    ]
})

print("\nFINAL GATE RESULTS")
print("=" * 80)

display(gate_results)

overall_pass = (gate_results["Status"] == "PASS").all()

print("\nOVERALL STATUS")
print("=" * 80)

if overall_pass:
    print(
        "PASS — revenue_analysis is analytically valid "
        "for downstream revenue analysis."
    )
else:
    print(
        "FAIL — revenue_analysis requires investigation "
        "before downstream use."
    )

FINAL REVENUE ANALYSIS INTEGRITY GATE

1. PRODUCT GRAIN
--------------------------------------------------------------------------------
Rows                  : 763
Unique Product_IDs    : 763
Duplicate Product_IDs : 0

2. REVENUE ACTIVITY CLASSIFICATION
--------------------------------------------------------------------------------
Missing classifications : 0
Unexpected classes      : set()

3. CRITICAL FIELD COMPLETENESS
--------------------------------------------------------------------------------
Product_ID               0
Product_Key              0
Product_Description      0
Product_Category         0
Net_Units                0
Revenue                  0
Revenue_Activity_Type    0
dtype: int64

4. CLASSIFICATION LOGIC
--------------------------------------------------------------------------------
Invalid positive revenue       : 0
Invalid zero revenue           : 0
Invalid net-return reversal    : 0
Invalid reversal-value variance: 0

5. GOVERNED KPI RECONCILIATION
-----------

,Check,Status
0,Product grain,PASS
1,Classification completeness,PASS
2,Critical-field completeness,PASS
3,Classification logic,PASS
4,Governed KPI reconciliation,PASS



OVERALL STATUS
PASS — revenue_analysis is analytically valid for downstream revenue analysis.


In [88]:
# ============================================================
# UNIT ANALYSIS POPULATION & VALIDATION
# ============================================================

# ------------------------------------------------------------
# 1. Create governed unit-analysis population
# ------------------------------------------------------------

unit_analysis = revenue_analysis.copy()

# Keep fields relevant to product sales/unit analysis
unit_cols = [
    "Product_ID",
    "Product_Key",
    "Product_Description",
    "Product_Category",
    "Months_Present",
    "Net_Units",
    "Revenue",
    "Revenue_Activity_Type"
]

unit_analysis = unit_analysis[unit_cols].copy()


# ------------------------------------------------------------
# 2. Basic population controls
# ------------------------------------------------------------

product_count = len(unit_analysis)

unique_products = (
    unit_analysis["Product_ID"]
    .nunique()
)

duplicate_products = (
    unit_analysis["Product_ID"]
    .duplicated()
    .sum()
)

total_net_units = (
    unit_analysis["Net_Units"]
    .sum()
)

total_revenue = (
    unit_analysis["Revenue"]
    .sum()
)


print("UNIT ANALYSIS POPULATION")
print("=" * 80)

print(f"Products              : {product_count:,}")
print(f"Unique Product_IDs    : {unique_products:,}")
print(f"Duplicate Product_IDs : {duplicate_products:,}")
print(f"Net Units             : {total_net_units:,.0f}")
print(f"Revenue               : £{total_revenue:,.2f}")


# ------------------------------------------------------------
# 3. Unit activity profile
# ------------------------------------------------------------

positive_unit_products = (
    unit_analysis["Net_Units"] > 0
).sum()

zero_unit_products = (
    unit_analysis["Net_Units"] == 0
).sum()

negative_unit_products = (
    unit_analysis["Net_Units"] < 0
).sum()


print("\nUNIT ACTIVITY PROFILE")
print("=" * 80)

print(
    f"Positive-unit products : "
    f"{positive_unit_products:,}"
)

print(
    f"Zero-unit products     : "
    f"{zero_unit_products:,}"
)

print(
    f"Negative-unit products : "
    f"{negative_unit_products:,}"
)


# ------------------------------------------------------------
# 4. Critical-field completeness
# ------------------------------------------------------------

critical_fields = [
    "Product_ID",
    "Product_Key",
    "Product_Description",
    "Product_Category",
    "Months_Present",
    "Net_Units"
]

missing_values = (
    unit_analysis[critical_fields]
    .isna()
    .sum()
)


print("\nCRITICAL FIELD COMPLETENESS")
print("=" * 80)

print(missing_values)


# ------------------------------------------------------------
# 5. Population validation
# ------------------------------------------------------------

population_valid = (
    product_count == 763
    and unique_products == 763
    and duplicate_products == 0
    and np.isclose(total_net_units, 1115)
    and np.isclose(total_revenue, 281727.54, atol=0.01)
    and missing_values.sum() == 0
)


print("\nPOPULATION VALIDATION")
print("=" * 80)

print(f"Population valid : {population_valid}")


# ------------------------------------------------------------
# 6. Preview
# ------------------------------------------------------------

display(
    unit_analysis
    .sort_values(
        "Net_Units",
        ascending=False
    )
    .head(10)
    .reset_index(drop=True)
)

UNIT ANALYSIS POPULATION
Products              : 763
Unique Product_IDs    : 763
Duplicate Product_IDs : 0
Net Units             : 1,115
Revenue               : £281,727.54

UNIT ACTIVITY PROFILE
Positive-unit products : 748
Zero-unit products     : 9
Negative-unit products : 6

CRITICAL FIELD COMPLETENESS
Product_ID             0
Product_Key            0
Product_Description    0
Product_Category       0
Months_Present         0
Net_Units              0
dtype: int64

POPULATION VALIDATION
Population valid : True


,Product_ID,Product_Key,Product_Description,Product_Category,Months_Present,Net_Units,Revenue,Revenue_Activity_Type
0,1167,T2351V11,Eufy Robot Vaccum X10 Pro Omni,ROBOT CLEANING,3,16,"6,818.30",POSITIVE_REVENUE
1,1289,VS15A6031R4SAMSUNG,Jet 60 Cordless Vacuum,STICK VACS,3,9,"1,102.02",POSITIVE_REVENUE
2,121,43LQ60006LA.LG,"43"" Smart TV",TV 33 - 43,3,8,"1,254.99",POSITIVE_REVENUE
3,886,P-SDU32GU18PNY,Elite microSDHC card 32G,IT ACCESSORIES,2,8,39.90,POSITIVE_REVENUE
4,525,GN BAGS,BAGS 400/600/800 SERIES AND S5,VACUUM BAGS,3,7,77.55,POSITIVE_REVENUE
5,771,MC1001UK,Ninja 8-in-1 Slow Cooker,FOOD PREP,3,7,761.65,POSITIVE_REVENUE
6,721,KN650A,Kenwood Electric Knife | KN650A,FOOD PREP,2,7,171.64,POSITIVE_REVENUE
7,569,HD301UK,Shark SppedStyle Hair Dryer,HAIRCARE,3,5,415.82,POSITIVE_REVENUE
8,23,11891600,Miele Ultra Phase 1 WA UP1 1402 L,WHITES ACCESSORIES,2,5,62.46,POSITIVE_REVENUE
9,491,FD16GATT4-EPNY,USB Sliding design - Black read,IT ACCESSORIES,2,5,42.07,POSITIVE_REVENUE


In [89]:
# ============================================================
# UNIT ACTIVITY CLASSIFICATION
# ============================================================

# ------------------------------------------------------------
# 1. Classify products by net-unit movement
# ------------------------------------------------------------

unit_analysis["Unit_Activity_Type"] = np.select(
    [
        unit_analysis["Net_Units"] > 0,
        np.isclose(unit_analysis["Net_Units"], 0),
        unit_analysis["Net_Units"] < 0
    ],
    [
        "POSITIVE_UNIT_ACTIVITY",
        "ZERO_NET_UNIT_ACTIVITY",
        "NET_RETURN_ACTIVITY"
    ],
    default="REVIEW_REQUIRED"
)


# ------------------------------------------------------------
# 2. Classification summary
# ------------------------------------------------------------

unit_activity_summary = (
    unit_analysis
    .groupby(
        "Unit_Activity_Type",
        as_index=False
    )
    .agg(
        Products=("Product_ID", "count"),
        Net_Units=("Net_Units", "sum"),
        Revenue=("Revenue", "sum")
    )
)

unit_activity_summary["Product_Share_%"] = (
    unit_activity_summary["Products"]
    / len(unit_analysis)
    * 100
)

# Unit contribution is meaningful primarily for positive
# unit activity, but retained here for reconciliation.
unit_activity_summary["Net_Unit_Share_%"] = (
    unit_activity_summary["Net_Units"]
    / unit_analysis["Net_Units"].sum()
    * 100
)


# ------------------------------------------------------------
# 3. Display
# ------------------------------------------------------------

print("UNIT ACTIVITY CLASSIFICATION")
print("=" * 80)

display(
    unit_activity_summary.style.format({
        "Products": "{:,.0f}",
        "Net_Units": "{:,.0f}",
        "Revenue": "£{:,.2f}",
        "Product_Share_%": "{:.2f}%",
        "Net_Unit_Share_%": "{:.2f}%"
    })
)


# ------------------------------------------------------------
# 4. Classification controls
# ------------------------------------------------------------

classified_products = (
    unit_activity_summary["Products"].sum()
)

unresolved_products = (
    unit_analysis["Unit_Activity_Type"]
    .eq("REVIEW_REQUIRED")
    .sum()
)

positive_count = (
    unit_analysis["Unit_Activity_Type"]
    .eq("POSITIVE_UNIT_ACTIVITY")
    .sum()
)

zero_count = (
    unit_analysis["Unit_Activity_Type"]
    .eq("ZERO_NET_UNIT_ACTIVITY")
    .sum()
)

negative_count = (
    unit_analysis["Unit_Activity_Type"]
    .eq("NET_RETURN_ACTIVITY")
    .sum()
)


print("\nCLASSIFICATION VALIDATION")
print("=" * 80)

print(
    f"Merchandise products   : "
    f"{len(unit_analysis):,}"
)

print(
    f"Classified products    : "
    f"{classified_products:,}"
)

print(
    f"Positive-unit products : "
    f"{positive_count:,}"
)

print(
    f"Zero-unit products     : "
    f"{zero_count:,}"
)

print(
    f"Negative-unit products : "
    f"{negative_count:,}"
)

print(
    f"Unresolved products    : "
    f"{unresolved_products:,}"
)

population_reconciles = (
    classified_products == len(unit_analysis)
    and positive_count == 748
    and zero_count == 9
    and negative_count == 6
    and unresolved_products == 0
)

print(
    f"Population reconciles  : "
    f"{population_reconciles}"
)

UNIT ACTIVITY CLASSIFICATION


,Unit_Activity_Type,Products,Net_Units,Revenue,Product_Share_%,Net_Unit_Share_%
0,NET_RETURN_ACTIVITY,6,-10,"£-1,581.61",0.79%,-0.90%
1,POSITIVE_UNIT_ACTIVITY,748,"1,125","£283,310.80",98.03%,100.90%
2,ZERO_NET_UNIT_ACTIVITY,9,0,£-1.65,1.18%,0.00%



CLASSIFICATION VALIDATION
Merchandise products   : 763
Classified products    : 763
Positive-unit products : 748
Zero-unit products     : 9
Negative-unit products : 6
Unresolved products    : 0
Population reconciles  : True


In [90]:
# ============================================================
# GROSS UNIT MOVEMENT & RETURN IMPACT
# ============================================================

# Positive movement
gross_positive_units = (
    unit_analysis.loc[
        unit_analysis["Net_Units"] > 0,
        "Net_Units"
    ].sum()
)

# Returned / reversed units expressed as absolute quantity
return_units = abs(
    unit_analysis.loc[
        unit_analysis["Net_Units"] < 0,
        "Net_Units"
    ].sum()
)

# Net movement
net_units = unit_analysis["Net_Units"].sum()

# Validation
unit_reconciliation = gross_positive_units - return_units

# Return rate relative to gross positive movement
return_rate = (
    return_units / gross_positive_units * 100
    if gross_positive_units != 0
    else 0
)

# Net realization rate
net_realization_rate = (
    net_units / gross_positive_units * 100
    if gross_positive_units != 0
    else 0
)


print("GROSS UNIT MOVEMENT & RETURN IMPACT")
print("=" * 80)

print(f"Gross positive units : {gross_positive_units:,.0f}")
print(f"Return units         : {return_units:,.0f}")
print(f"Net units            : {net_units:,.0f}")

print("\nUNIT RECONCILIATION")
print("=" * 80)

print(
    f"{gross_positive_units:,.0f} - "
    f"{return_units:,.0f} = "
    f"{unit_reconciliation:,.0f}"
)

print(
    f"Matches governed net units : "
    f"{unit_reconciliation == net_units}"
)

print("\nRETURN IMPACT")
print("=" * 80)

print(f"Return rate          : {return_rate:.2f}%")
print(f"Net realization rate : {net_realization_rate:.2f}%")

GROSS UNIT MOVEMENT & RETURN IMPACT
Gross positive units : 1,125
Return units         : 10
Net units            : 1,115

UNIT RECONCILIATION
1,125 - 10 = 1,115
Matches governed net units : True

RETURN IMPACT
Return rate          : 0.89%
Net realization rate : 99.11%


In [91]:
# ============================================================
#  PRODUCT UNIT VELOCITY
# ============================================================

unit_velocity = unit_analysis.copy()

# ------------------------------------------------------------
# 1. Calculate monthly unit velocity
# ------------------------------------------------------------

unit_velocity["Units_Per_Month"] = (
    unit_velocity["Net_Units"] /
    unit_velocity["Months_Present"]
)

# ------------------------------------------------------------
# 2. Validation
# ------------------------------------------------------------

invalid_months = (
    unit_velocity["Months_Present"].isna() |
    (unit_velocity["Months_Present"] <= 0)
).sum()

missing_velocity = unit_velocity["Units_Per_Month"].isna().sum()

print("PRODUCT UNIT VELOCITY")
print("=" * 80)

print(f"Products                 : {len(unit_velocity):,}")
print(f"Invalid Months_Present   : {invalid_months:,}")
print(f"Missing velocity values  : {missing_velocity:,}")

# ------------------------------------------------------------
# 3. Velocity statistics
# ------------------------------------------------------------

print("\nUNIT VELOCITY STATISTICS")
print("=" * 80)

print(
    f"Mean units/month     : "
    f"{unit_velocity['Units_Per_Month'].mean():,.2f}"
)

print(
    f"Median units/month   : "
    f"{unit_velocity['Units_Per_Month'].median():,.2f}"
)

print(
    f"Maximum units/month  : "
    f"{unit_velocity['Units_Per_Month'].max():,.2f}"
)

print(
    f"Minimum units/month  : "
    f"{unit_velocity['Units_Per_Month'].min():,.2f}"
)

# ------------------------------------------------------------
# 4. Highest velocity products
# ------------------------------------------------------------

top_velocity = (
    unit_velocity
    .sort_values(
        ["Units_Per_Month", "Net_Units"],
        ascending=[False, False]
    )
    [
        [
            "Product_ID",
            "Product_Key",
            "Product_Description",
            "Product_Category",
            "Months_Present",
            "Net_Units",
            "Units_Per_Month",
            "Revenue"
        ]
    ]
    .head(20)
    .reset_index(drop=True)
)

top_velocity.index += 1
top_velocity.index.name = "Velocity_Rank"

print("\nTOP 20 PRODUCTS BY UNIT VELOCITY")
print("=" * 80)

display(
    top_velocity.style.format({
        "Net_Units": "{:,.0f}",
        "Units_Per_Month": "{:,.2f}",
        "Revenue": "£{:,.2f}"
    })
)

PRODUCT UNIT VELOCITY
Products                 : 763
Invalid Months_Present   : 0
Missing velocity values  : 0

UNIT VELOCITY STATISTICS
Mean units/month     : 1.15
Median units/month   : 1.00
Maximum units/month  : 5.33
Minimum units/month  : -5.00

TOP 20 PRODUCTS BY UNIT VELOCITY


,Product_ID,Product_Key,Product_Description,Product_Category,Months_Present,Net_Units,Units_Per_Month,Revenue
Velocity_Rank,,,,,,,,
1,1167,T2351V11,Eufy Robot Vaccum X10 Pro Omni,ROBOT CLEANING,3,16,5.33,"£6,818.30"
2,722,KN650B,Kenwood Electric Kitchen Knife,FOOD PREP,1,5,5.00,£77.75
3,886,P-SDU32GU18PNY,Elite microSDHC card 32G,IT ACCESSORIES,2,8,4.00,£39.90
4,101,300300,"MR Crystal Clear 2400watt, Precision",FOOD PREP,1,4,4.00,£78.31
5,1055,S6005UK,Shark Floor & Handheld Steam,STEAM CLEANERS,1,4,4.00,£481.65
6,721,KN650A,Kenwood Electric Knife | KN650A,FOOD PREP,2,7,3.50,£171.64
7,1289,VS15A6031R4SAMSUNG,Jet 60 Cordless Vacuum,STICK VACS,3,9,3.00,"£1,102.02"
8,20,112072,3xRCA M - 3xRCA M Lead 1.5m,CABLES,1,3,3.00,£7.48
9,76,25111,Eclipse Kettle - Midnight Blue,KETTLES,1,3,3.00,£110.41


In [92]:
# ============================================================
#  POSITIVE VELOCITY DISTRIBUTION & QUANTILES
# ============================================================

# Analyse only products with genuine positive net movement
positive_velocity = unit_velocity.loc[
    unit_velocity["Net_Units"] > 0
].copy()

print("POSITIVE PRODUCT VELOCITY DISTRIBUTION")
print("=" * 80)

print(f"Positive-unit products : {len(positive_velocity):,}")
print(
    f"Share of merchandise  : "
    f"{len(positive_velocity) / len(unit_velocity) * 100:.2f}%"
)

# ------------------------------------------------------------
# Distribution statistics
# ------------------------------------------------------------

velocity_stats = positive_velocity["Units_Per_Month"].describe(
    percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
)

print("\nVELOCITY DISTRIBUTION STATISTICS")
print("=" * 80)

print(velocity_stats.round(2))

# ------------------------------------------------------------
# Explicit quantile thresholds
# ------------------------------------------------------------

q25 = positive_velocity["Units_Per_Month"].quantile(0.25)
q50 = positive_velocity["Units_Per_Month"].quantile(0.50)
q75 = positive_velocity["Units_Per_Month"].quantile(0.75)
q90 = positive_velocity["Units_Per_Month"].quantile(0.90)
q95 = positive_velocity["Units_Per_Month"].quantile(0.95)

print("\nVELOCITY QUANTILE THRESHOLDS")
print("=" * 80)

print(f"25th percentile : {q25:.2f}")
print(f"50th percentile : {q50:.2f}")
print(f"75th percentile : {q75:.2f}")
print(f"90th percentile : {q90:.2f}")
print(f"95th percentile : {q95:.2f}")

# ------------------------------------------------------------
# Frequency distribution
# ------------------------------------------------------------

velocity_frequency = (
    positive_velocity
    .groupby("Units_Per_Month")
    .agg(
        Products=("Product_ID", "count"),
        Net_Units=("Net_Units", "sum"),
        Revenue=("Revenue", "sum")
    )
    .reset_index()
    .sort_values("Units_Per_Month")
)

velocity_frequency["Product_Share_%"] = (
    velocity_frequency["Products"] /
    len(positive_velocity) * 100
)

print("\nOBSERVED VELOCITY FREQUENCY")
print("=" * 80)

display(
    velocity_frequency.style.format({
        "Units_Per_Month": "{:.2f}",
        "Products": "{:,.0f}",
        "Net_Units": "{:,.0f}",
        "Revenue": "£{:,.2f}",
        "Product_Share_%": "{:.2f}%"
    })
)

POSITIVE PRODUCT VELOCITY DISTRIBUTION
Positive-unit products : 748
Share of merchandise  : 98.03%

VELOCITY DISTRIBUTION STATISTICS
count   748.00
mean      1.19
std       0.50
min       0.50
10%       1.00
25%       1.00
50%       1.00
75%       1.00
90%       2.00
95%       2.00
max       5.33
Name: Units_Per_Month, dtype: float64

VELOCITY QUANTILE THRESHOLDS
25th percentile : 1.00
50th percentile : 1.00
75th percentile : 1.00
90th percentile : 2.00
95th percentile : 2.00

OBSERVED VELOCITY FREQUENCY


,Units_Per_Month,Products,Net_Units,Revenue,Product_Share_%
0,0.50,5,5,£434.12,0.67%
1,1.00,614,718,"£209,208.62",82.09%
2,1.33,4,16,"£2,834.87",0.53%
3,1.50,31,93,"£17,355.67",4.14%
4,1.67,4,20,"£3,232.86",0.53%
5,2.00,62,134,"£28,685.24",8.29%
6,2.33,2,14,£839.20,0.27%
7,2.50,5,25,"£2,898.69",0.67%
8,2.67,1,8,"£1,254.99",0.13%
9,3.00,14,48,"£8,898.99",1.87%


In [93]:
# ============================================================
#  PRODUCT VELOCITY CLASSIFICATION
# ============================================================

import numpy as np

velocity_classified = unit_velocity.copy()

# ------------------------------------------------------------
# 1. Classify product velocity
# ------------------------------------------------------------

conditions = [
    velocity_classified["Units_Per_Month"] < 0,
    velocity_classified["Units_Per_Month"] == 0,
    (
        (velocity_classified["Units_Per_Month"] > 0) &
        (velocity_classified["Units_Per_Month"] <= 1)
    ),
    (
        (velocity_classified["Units_Per_Month"] > 1) &
        (velocity_classified["Units_Per_Month"] <= 2)
    ),
    velocity_classified["Units_Per_Month"] > 2
]

labels = [
    "RETURN_ACTIVITY",
    "ZERO_ACTIVITY",
    "LOW_VELOCITY",
    "MEDIUM_VELOCITY",
    "HIGH_VELOCITY"
]

velocity_classified["Velocity_Class"] = np.select(
    conditions,
    labels,
    default="UNCLASSIFIED"
)

# ------------------------------------------------------------
# 2. Summary
# ------------------------------------------------------------

velocity_summary = (
    velocity_classified
    .groupby("Velocity_Class", as_index=False)
    .agg(
        Products=("Product_ID", "count"),
        Net_Units=("Net_Units", "sum"),
        Revenue=("Revenue", "sum")
    )
)

velocity_summary["Product_Share_%"] = (
    velocity_summary["Products"] /
    len(velocity_classified) * 100
)

velocity_summary["Net_Unit_Share_%"] = (
    velocity_summary["Net_Units"] /
    velocity_classified["Net_Units"].sum() * 100
)

# Logical ordering
class_order = [
    "RETURN_ACTIVITY",
    "ZERO_ACTIVITY",
    "LOW_VELOCITY",
    "MEDIUM_VELOCITY",
    "HIGH_VELOCITY",
    "UNCLASSIFIED"
]

velocity_summary["Velocity_Class"] = pd.Categorical(
    velocity_summary["Velocity_Class"],
    categories=class_order,
    ordered=True
)

velocity_summary = (
    velocity_summary
    .sort_values("Velocity_Class")
    .reset_index(drop=True)
)

print("PRODUCT VELOCITY CLASSIFICATION")
print("=" * 80)

display(
    velocity_summary.style.format({
        "Products": "{:,.0f}",
        "Net_Units": "{:,.0f}",
        "Revenue": "£{:,.2f}",
        "Product_Share_%": "{:.2f}%",
        "Net_Unit_Share_%": "{:.2f}%"
    })
)

# ------------------------------------------------------------
# 3. Classification controls
# ------------------------------------------------------------

classified_products = (
    velocity_classified["Velocity_Class"] != "UNCLASSIFIED"
).sum()

unclassified_products = (
    velocity_classified["Velocity_Class"] == "UNCLASSIFIED"
).sum()

print("\nCLASSIFICATION VALIDATION")
print("=" * 80)

print(f"Merchandise products : {len(unit_velocity):,}")
print(f"Classified products  : {classified_products:,}")
print(f"Unclassified products: {unclassified_products:,}")

print(
    "Population reconciles:",
    classified_products == len(unit_velocity)
)

PRODUCT VELOCITY CLASSIFICATION


,Velocity_Class,Products,Net_Units,Revenue,Product_Share_%,Net_Unit_Share_%
0,RETURN_ACTIVITY,6,-10,"£-1,581.61",0.79%,-0.90%
1,ZERO_ACTIVITY,9,0,£-1.65,1.18%,0.00%
2,LOW_VELOCITY,619,723,"£209,642.74",81.13%,64.84%
3,MEDIUM_VELOCITY,101,263,"£52,108.64",13.24%,23.59%
4,HIGH_VELOCITY,28,139,"£21,559.42",3.67%,12.47%



CLASSIFICATION VALIDATION
Merchandise products : 763
Classified products  : 763
Unclassified products: 0
Population reconciles: True


In [94]:
# ============================================================
#  CATEGORY UNIT PERFORMANCE
# ============================================================

category_units = (
    unit_analysis
    .groupby("Product_Category", as_index=False)
    .agg(
        Products=("Product_ID", "count"),
        Net_Units=("Net_Units", "sum"),
        Revenue=("Revenue", "sum")
    )
)

# ------------------------------------------------------------
# Shares
# ------------------------------------------------------------

category_units["Product_Share_%"] = (
    category_units["Products"] /
    unit_analysis["Product_ID"].nunique() * 100
)

category_units["Net_Unit_Share_%"] = (
    category_units["Net_Units"] /
    unit_analysis["Net_Units"].sum() * 100
)

category_units["Revenue_Share_%"] = (
    category_units["Revenue"] /
    unit_analysis["Revenue"].sum() * 100
)

# ------------------------------------------------------------
# Productivity metrics
# ------------------------------------------------------------

category_units["Units_Per_Product"] = (
    category_units["Net_Units"] /
    category_units["Products"]
)

category_units["Revenue_Per_Product"] = (
    category_units["Revenue"] /
    category_units["Products"]
)

# Rank by unit movement
category_units = (
    category_units
    .sort_values(
        ["Net_Units", "Revenue"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

category_units.insert(
    0,
    "Unit_Rank",
    range(1, len(category_units) + 1)
)

print("CATEGORY UNIT PERFORMANCE")
print("=" * 80)

print(f"Categories          : {len(category_units):,}")
print(f"Products            : {category_units['Products'].sum():,}")
print(f"Net units           : {category_units['Net_Units'].sum():,.0f}")
print(f"Revenue             : £{category_units['Revenue'].sum():,.2f}")

print("\nTOP 20 CATEGORIES BY NET UNIT MOVEMENT")
print("=" * 80)

display(
    category_units.head(20).style.format({
        "Products": "{:,.0f}",
        "Net_Units": "{:,.0f}",
        "Revenue": "£{:,.2f}",
        "Product_Share_%": "{:.2f}%",
        "Net_Unit_Share_%": "{:.2f}%",
        "Revenue_Share_%": "{:.2f}%",
        "Units_Per_Product": "{:.2f}",
        "Revenue_Per_Product": "£{:,.2f}"
    })
)

# ------------------------------------------------------------
# Reconciliation
# ------------------------------------------------------------

product_check = (
    category_units["Products"].sum()
    == len(unit_analysis)
)

unit_check = (
    category_units["Net_Units"].sum()
    == unit_analysis["Net_Units"].sum()
)

revenue_diff = (
    category_units["Revenue"].sum()
    - unit_analysis["Revenue"].sum()
)

print("\nCATEGORY RECONCILIATION")
print("=" * 80)

print(f"Product population reconciles : {product_check}")
print(f"Net units reconcile           : {unit_check}")
print(f"Revenue reconciliation diff   : £{revenue_diff:,.4f}")

CATEGORY UNIT PERFORMANCE
Categories          : 85
Products            : 763
Net units           : 1,115
Revenue             : £281,727.54

TOP 20 CATEGORIES BY NET UNIT MOVEMENT


,Unit_Rank,Product_Category,Products,Net_Units,Revenue,Product_Share_%,Net_Unit_Share_%,Revenue_Share_%,Units_Per_Product,Revenue_Per_Product
0,1,FOOD PREP,39,68,"£4,397.13",5.11%,6.10%,1.56%,1.74,£112.75
1,2,WASHING MACHINES,30,50,"£22,279.54",3.93%,4.48%,7.91%,1.67,£742.65
2,3,IT ACCESSORIES,24,47,£723.05,3.15%,4.22%,0.26%,1.96,£30.13
3,4,KETTLES,34,44,"£2,239.30",4.46%,3.95%,0.79%,1.29,£65.86
4,5,CABLES,25,41,£366.44,3.28%,3.68%,0.13%,1.64,£14.66
5,6,STICK VACS,21,37,"£8,297.41",2.75%,3.32%,2.95%,1.76,£395.11
6,7,ACCESSORIES,20,37,£494.75,2.62%,3.32%,0.18%,1.85,£24.74
7,8,TV 51 - 59,18,33,"£17,919.17",2.36%,2.96%,6.36%,1.83,£995.51
8,9,SINGLE OVENS,27,29,"£17,204.18",3.54%,2.60%,6.11%,1.07,£637.19
9,10,FRYERS,13,29,"£3,172.47",1.70%,2.60%,1.13%,2.23,£244.04



CATEGORY RECONCILIATION
Product population reconciles : True
Net units reconcile           : True
Revenue reconciliation diff   : £0.0000


In [95]:
# ============================================================
#  UNIT CONCENTRATION / PARETO ANALYSIS
# ============================================================

# Use positive-unit products only for demand concentration
unit_pareto = (
    unit_analysis.loc[unit_analysis["Net_Units"] > 0]
    .copy()
    .sort_values(
        ["Net_Units", "Revenue"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

unit_pareto["Unit_Rank"] = range(1, len(unit_pareto) + 1)

gross_positive_units = unit_pareto["Net_Units"].sum()

unit_pareto["Gross_Unit_Contribution_%"] = (
    unit_pareto["Net_Units"] /
    gross_positive_units * 100
)

unit_pareto["Cumulative_Gross_Unit_%"] = (
    unit_pareto["Gross_Unit_Contribution_%"].cumsum()
)

# ------------------------------------------------------------
# Function: products required to reach threshold
# ------------------------------------------------------------

def products_to_reach_unit_share(df, threshold):
    mask = df["Cumulative_Gross_Unit_%"] >= threshold

    if not mask.any():
        return len(df)

    return int(df.loc[mask, "Unit_Rank"].iloc[0])


unit_50 = products_to_reach_unit_share(unit_pareto, 50)
unit_80 = products_to_reach_unit_share(unit_pareto, 80)
unit_90 = products_to_reach_unit_share(unit_pareto, 90)


print("UNIT CONCENTRATION / PARETO ANALYSIS")
print("=" * 80)

print(f"Positive-unit products : {len(unit_pareto):,}")
print(f"Gross positive units   : {gross_positive_units:,.0f}")

print("\nPRODUCTS REQUIRED TO GENERATE GROSS UNIT MOVEMENT")
print("=" * 80)

for threshold, count in [
    (50, unit_50),
    (80, unit_80),
    (90, unit_90)
]:
    print(
        f"{threshold}% of units : "
        f"{count:,} products "
        f"({count / len(unit_pareto) * 100:.2f}% "
        f"of positive-unit products)"
    )

# ------------------------------------------------------------
# Top-N concentration
# ------------------------------------------------------------

print("\nTOP PRODUCT UNIT CONCENTRATION")
print("=" * 80)

for n in [10, 20, 50, 100]:
    n_actual = min(n, len(unit_pareto))

    contribution = (
        unit_pareto.head(n_actual)["Net_Units"].sum()
        / gross_positive_units * 100
    )

    print(
        f"Top {n_actual:>3} products : "
        f"{contribution:.2f}% of gross positive units"
    )

# ------------------------------------------------------------
# Display leading products
# ------------------------------------------------------------

display_cols = [
    "Unit_Rank",
    "Product_ID",
    "Product_Key",
    "Product_Description",
    "Product_Category",
    "Months_Present",
    "Net_Units",
    "Units_Per_Month",
    "Revenue",
    "Gross_Unit_Contribution_%",
    "Cumulative_Gross_Unit_%"
]

# Units_Per_Month already exists in unit_velocity,
# so merge it safely into the Pareto table.
unit_pareto = unit_pareto.merge(
    unit_velocity[
        ["Product_ID", "Units_Per_Month"]
    ],
    on="Product_ID",
    how="left",
    validate="one_to_one"
)

print("\nTOP 20 PRODUCTS BY GROSS UNIT MOVEMENT")
print("=" * 80)

display(
    unit_pareto[display_cols]
    .head(20)
    .style.format({
        "Months_Present": "{:,.0f}",
        "Net_Units": "{:,.0f}",
        "Units_Per_Month": "{:.2f}",
        "Revenue": "£{:,.2f}",
        "Gross_Unit_Contribution_%": "{:.2f}%",
        "Cumulative_Gross_Unit_%": "{:.2f}%"
    })
)

# ------------------------------------------------------------
# Reconciliation
# ------------------------------------------------------------

print("\nPARETO VALIDATION")
print("=" * 80)

print(
    f"Positive-unit products reconcile : "
    f"{len(unit_pareto) == (unit_analysis['Net_Units'] > 0).sum()}"
)

print(
    f"Gross positive units reconcile   : "
    f"{unit_pareto['Net_Units'].sum() == gross_positive_units}"
)

print(
    f"Final cumulative share           : "
    f"{unit_pareto['Cumulative_Gross_Unit_%'].iloc[-1]:.2f}%"
)

UNIT CONCENTRATION / PARETO ANALYSIS
Positive-unit products : 748
Gross positive units   : 1,125

PRODUCTS REQUIRED TO GENERATE GROSS UNIT MOVEMENT
50% of units : 204 products (27.27% of positive-unit products)
80% of units : 524 products (70.05% of positive-unit products)
90% of units : 636 products (85.03% of positive-unit products)

TOP PRODUCT UNIT CONCENTRATION
Top  10 products : 6.84% of gross positive units
Top  20 products : 11.02% of gross positive units
Top  50 products : 19.73% of gross positive units
Top 100 products : 31.56% of gross positive units

TOP 20 PRODUCTS BY GROSS UNIT MOVEMENT


,Unit_Rank,Product_ID,Product_Key,Product_Description,Product_Category,Months_Present,Net_Units,Units_Per_Month,Revenue,Gross_Unit_Contribution_%,Cumulative_Gross_Unit_%
0,1,1167,T2351V11,Eufy Robot Vaccum X10 Pro Omni,ROBOT CLEANING,3,16,5.33,"£6,818.30",1.42%,1.42%
1,2,1289,VS15A6031R4SAMSUNG,Jet 60 Cordless Vacuum,STICK VACS,3,9,3.00,"£1,102.02",0.80%,2.22%
2,3,121,43LQ60006LA.LG,"43"" Smart TV",TV 33 - 43,3,8,2.67,"£1,254.99",0.71%,2.93%
3,4,886,P-SDU32GU18PNY,Elite microSDHC card 32G,IT ACCESSORIES,2,8,4.00,£39.90,0.71%,3.64%
4,5,771,MC1001UK,Ninja 8-in-1 Slow Cooker,FOOD PREP,3,7,2.33,£761.65,0.62%,4.27%
5,6,721,KN650A,Kenwood Electric Knife | KN650A,FOOD PREP,2,7,3.50,£171.64,0.62%,4.89%
6,7,525,GN BAGS,BAGS 400/600/800 SERIES AND S5,VACUUM BAGS,3,7,2.33,£77.55,0.62%,5.51%
7,8,170,55NANO81A6,LG 55 NANO TV,TV 51 - 59,3,5,1.67,"£1,662.50",0.44%,5.96%
8,9,1262,UE55U7000FKSAMSUNG,55 in Smart Television,TV 51 - 59,2,5,2.50,"£1,304.17",0.44%,6.40%
9,10,1005,REV-ISTREA,Roberts Revival iStream 3L Black,INTERNET RADIOS,2,5,2.50,£829.98,0.44%,6.84%



PARETO VALIDATION
Positive-unit products reconcile : True
Gross positive units reconcile   : True
Final cumulative share           : 100.00%


### Step 6 — Unit Velocity × Revenue Performance Matrix

This analysis combines product unit velocity and revenue contribution to distinguish
between products that generate value through high movement, high monetary value,
or both.

The matrix uses median values among positive-activity merchandise products as
data-driven segmentation thresholds.

Segments:

- HIGH_VELOCITY_HIGH_REVENUE
- HIGH_VELOCITY_LOW_REVENUE
- LOW_VELOCITY_HIGH_REVENUE
- LOW_VELOCITY_LOW_REVENUE

Return and zero-activity products are retained separately and are not forced into
commercial performance segments.

In [101]:
# ============================================================
# 4.10B.9 — UNIT VELOCITY × REVENUE PERFORMANCE MATRIX
# ============================================================

performance_matrix =  unit_velocity.copy()

# ------------------------------------------------------------
# 1. Define eligible population
# ------------------------------------------------------------

positive_population = performance_matrix[
    (performance_matrix["Net_Units"] > 0) &
    (performance_matrix["Revenue"] > 0)
].copy()

velocity_threshold = positive_population["Units_Per_Month"].median()
revenue_threshold = positive_population["Revenue"].median()

print("UNIT VELOCITY × REVENUE PERFORMANCE MATRIX")
print("=" * 80)

print(f"Merchandise products        : {len(performance_matrix):,}")
print(f"Positive activity products  : {len(positive_population):,}")
print()
print(f"Velocity median threshold   : {velocity_threshold:,.2f} units/month")
print(f"Revenue median threshold    : £{revenue_threshold:,.2f}")


# ------------------------------------------------------------
# 2. Classification function
# ------------------------------------------------------------

def classify_product_performance(row):

    # Returns / net negative activity
    if row["Net_Units"] < 0:
        return "RETURN_ACTIVITY"

    # No net unit movement
    if row["Net_Units"] == 0:
        return "ZERO_UNIT_ACTIVITY"

    # Positive units but non-positive revenue
    if row["Revenue"] <= 0:
        return "REVENUE_REVIEW"

    # Commercial performance matrix
    if (
        row["Units_Per_Month"] >= velocity_threshold
        and row["Revenue"] >= revenue_threshold
    ):
        return "HIGH_VELOCITY_HIGH_REVENUE"

    elif (
        row["Units_Per_Month"] >= velocity_threshold
        and row["Revenue"] < revenue_threshold
    ):
        return "HIGH_VELOCITY_LOW_REVENUE"

    elif (
        row["Units_Per_Month"] < velocity_threshold
        and row["Revenue"] >= revenue_threshold
    ):
        return "LOW_VELOCITY_HIGH_REVENUE"

    else:
        return "LOW_VELOCITY_LOW_REVENUE"


performance_matrix["Performance_Segment"] = performance_matrix.apply(
    classify_product_performance,
    axis=1
)


# ------------------------------------------------------------
# 3. Segment summary
# ------------------------------------------------------------

performance_summary = (
    performance_matrix
    .groupby("Performance_Segment", as_index=False)
    .agg(
        Products=("Product_ID", "nunique"),
        Net_Units=("Net_Units", "sum"),
        Revenue=("Revenue", "sum")
    )
)

performance_summary["Product_Share_%"] = (
    performance_summary["Products"]
    / performance_matrix["Product_ID"].nunique()
    * 100
)

performance_summary["Revenue_Share_%"] = (
    performance_summary["Revenue"]
    / performance_matrix["Revenue"].sum()
    * 100
)

performance_summary = performance_summary.sort_values(
    "Revenue",
    ascending=False
).reset_index(drop=True)


# ------------------------------------------------------------
# 4. Display formatted summary
# ------------------------------------------------------------

display_summary = performance_summary.copy()

display_summary["Revenue"] = display_summary["Revenue"].map(
    lambda x: f"£{x:,.2f}"
)

display_summary["Product_Share_%"] = display_summary[
    "Product_Share_%"
].map(lambda x: f"{x:.2f}%")

display_summary["Revenue_Share_%"] = display_summary[
    "Revenue_Share_%"
].map(lambda x: f"{x:.2f}%")

display(display_summary)


# ------------------------------------------------------------
# 5. Validation
# ------------------------------------------------------------

classified_products = performance_summary["Products"].sum()

print("\nCLASSIFICATION VALIDATION")
print("=" * 80)

print(
    f"Merchandise products  : "
    f"{performance_matrix['Product_ID'].nunique():,}"
)

print(
    f"Classified products   : "
    f"{classified_products:,}"
)

print(
    f"Unclassified products : "
    f"{performance_matrix['Performance_Segment'].isna().sum():,}"
)

print(
    "Population reconciles:",
    classified_products
    == performance_matrix["Product_ID"].nunique()
)

print(
    "Revenue reconciliation diff:",
    f"£{abs(performance_summary['Revenue'].sum() - performance_matrix['Revenue'].sum()):,.4f}"
)

UNIT VELOCITY × REVENUE PERFORMANCE MATRIX
Merchandise products        : 763
Positive activity products  : 748

Velocity median threshold   : 1.00 units/month
Revenue median threshold    : £165.83


,Performance_Segment,Products,Net_Units,Revenue,Product_Share_%,Revenue_Share_%
0,HIGH_VELOCITY_HIGH_REVENUE,379,601,"£261,459.40",49.67%,92.81%
1,HIGH_VELOCITY_LOW_REVENUE,364,519,"£21,417.28",47.71%,7.60%
2,LOW_VELOCITY_HIGH_REVENUE,1,1,£283.33,0.13%,0.10%
3,LOW_VELOCITY_LOW_REVENUE,4,4,£150.79,0.52%,0.05%
4,ZERO_UNIT_ACTIVITY,9,0,£-1.65,1.18%,-0.00%
5,RETURN_ACTIVITY,6,-10,"£-1,581.61",0.79%,-0.56%



CLASSIFICATION VALIDATION
Merchandise products  : 763
Classified products   : 763
Unclassified products : 0
Population reconciles: True
Revenue reconciliation diff: £0.0000


### Step 7 — Commercial Unit Performance Segmentation

The previous velocity–revenue matrix established that the median positive-product
velocity is 1.00 unit per month.

Because a large proportion of products operate exactly at this threshold, the terms
"high velocity" and "low velocity" could overstate the commercial interpretation.

Therefore, the governed segmentation uses threshold-relative terminology:

- AT_OR_ABOVE_MEDIAN_VELOCITY_HIGH_REVENUE
- AT_OR_ABOVE_MEDIAN_VELOCITY_LOW_REVENUE
- BELOW_MEDIAN_VELOCITY_HIGH_REVENUE
- BELOW_MEDIAN_VELOCITY_LOW_REVENUE

Return and zero-unit activity remain separately classified.

This preserves the quantitative methodology while improving analytical interpretability.

In [102]:
# ============================================================
#  COMMERCIAL UNIT PERFORMANCE SEGMENTATION
# ============================================================

commercial_unit_segments = unit_velocity.copy()

# ------------------------------------------------------------
# 1. Governed thresholds
# ------------------------------------------------------------

eligible_population = commercial_unit_segments[
    (commercial_unit_segments["Net_Units"] > 0) &
    (commercial_unit_segments["Revenue"] > 0)
].copy()

velocity_threshold = eligible_population["Units_Per_Month"].median()
revenue_threshold = eligible_population["Revenue"].median()


# ------------------------------------------------------------
# 2. Governed classification
# ------------------------------------------------------------

def classify_commercial_unit_performance(row):

    if row["Net_Units"] < 0:
        return "RETURN_ACTIVITY"

    if row["Net_Units"] == 0:
        return "ZERO_UNIT_ACTIVITY"

    if row["Revenue"] <= 0:
        return "REVENUE_REVIEW"

    velocity_group = (
        "AT_OR_ABOVE_MEDIAN_VELOCITY"
        if row["Units_Per_Month"] >= velocity_threshold
        else "BELOW_MEDIAN_VELOCITY"
    )

    revenue_group = (
        "HIGH_REVENUE"
        if row["Revenue"] >= revenue_threshold
        else "LOW_REVENUE"
    )

    return f"{velocity_group}_{revenue_group}"


commercial_unit_segments["Commercial_Unit_Segment"] = (
    commercial_unit_segments.apply(
        classify_commercial_unit_performance,
        axis=1
    )
)


# ------------------------------------------------------------
# 3. Segment summary
# ------------------------------------------------------------

commercial_segment_summary = (
    commercial_unit_segments
    .groupby("Commercial_Unit_Segment", as_index=False)
    .agg(
        Products=("Product_ID", "nunique"),
        Net_Units=("Net_Units", "sum"),
        Revenue=("Revenue", "sum")
    )
)

total_products = commercial_unit_segments["Product_ID"].nunique()
total_net_units = commercial_unit_segments["Net_Units"].sum()
total_revenue = commercial_unit_segments["Revenue"].sum()

commercial_segment_summary["Product_Share_%"] = (
    commercial_segment_summary["Products"]
    / total_products * 100
)

commercial_segment_summary["Net_Unit_Share_%"] = (
    commercial_segment_summary["Net_Units"]
    / total_net_units * 100
)

commercial_segment_summary["Revenue_Share_%"] = (
    commercial_segment_summary["Revenue"]
    / total_revenue * 100
)

commercial_segment_summary = commercial_segment_summary.sort_values(
    "Revenue",
    ascending=False
).reset_index(drop=True)


# ------------------------------------------------------------
# 4. Display
# ------------------------------------------------------------

display_summary = commercial_segment_summary.copy()

display_summary["Revenue"] = display_summary["Revenue"].map(
    lambda x: f"£{x:,.2f}"
)

for col in [
    "Product_Share_%",
    "Net_Unit_Share_%",
    "Revenue_Share_%"
]:
    display_summary[col] = display_summary[col].map(
        lambda x: f"{x:.2f}%"
    )

print("COMMERCIAL UNIT PERFORMANCE SEGMENTATION")
print("=" * 85)

print(f"Velocity threshold : {velocity_threshold:.2f} units/month")
print(f"Revenue threshold  : £{revenue_threshold:,.2f}")
print()

display(display_summary)


# ------------------------------------------------------------
# 5. Governance validation
# ------------------------------------------------------------

classified_products = commercial_segment_summary["Products"].sum()

population_ok = classified_products == total_products

unit_diff = abs(
    commercial_segment_summary["Net_Units"].sum()
    - total_net_units
)

revenue_diff = abs(
    commercial_segment_summary["Revenue"].sum()
    - total_revenue
)

unclassified = (
    commercial_unit_segments["Commercial_Unit_Segment"]
    .isna()
    .sum()
)

print("\nSEGMENTATION VALIDATION")
print("=" * 85)

print(f"Merchandise products       : {total_products:,}")
print(f"Classified products        : {classified_products:,}")
print(f"Unclassified products      : {unclassified:,}")
print(f"Population reconciles      : {population_ok}")
print(f"Net-unit reconciliation diff: {unit_diff:,.4f}")
print(f"Revenue reconciliation diff : £{revenue_diff:,.4f}")

COMMERCIAL UNIT PERFORMANCE SEGMENTATION
Velocity threshold : 1.00 units/month
Revenue threshold  : £165.83



,Commercial_Unit_Segment,Products,Net_Units,Revenue,Product_Share_%,Net_Unit_Share_%,Revenue_Share_%
0,AT_OR_ABOVE_MEDIAN_VELOCITY_HIGH_REVENUE,379,601,"£261,459.40",49.67%,53.90%,92.81%
1,AT_OR_ABOVE_MEDIAN_VELOCITY_LOW_REVENUE,364,519,"£21,417.28",47.71%,46.55%,7.60%
2,BELOW_MEDIAN_VELOCITY_HIGH_REVENUE,1,1,£283.33,0.13%,0.09%,0.10%
3,BELOW_MEDIAN_VELOCITY_LOW_REVENUE,4,4,£150.79,0.52%,0.36%,0.05%
4,ZERO_UNIT_ACTIVITY,9,0,£-1.65,1.18%,0.00%,-0.00%
5,RETURN_ACTIVITY,6,-10,"£-1,581.61",0.79%,-0.90%,-0.56%



SEGMENTATION VALIDATION
Merchandise products       : 763
Classified products        : 763
Unclassified products      : 0
Population reconciles      : True
Net-unit reconciliation diff: 0.0000
Revenue reconciliation diff : £0.0000


In [103]:
# ============================================================
# UNIT ANALYSIS FINAL INTEGRITY GATE
# ============================================================

print("FINAL UNIT ANALYSIS INTEGRITY GATE")
print("=" * 85)

# ------------------------------------------------------------
# 1. Product grain
# ------------------------------------------------------------

total_rows = len(unit_velocity)
unique_products = unit_velocity["Product_ID"].nunique()
duplicate_products = unit_velocity["Product_ID"].duplicated().sum()

print("\n1. PRODUCT GRAIN")
print("-" * 85)
print(f"Rows                  : {total_rows:,}")
print(f"Unique Product_IDs    : {unique_products:,}")
print(f"Duplicate Product_IDs : {duplicate_products:,}")


# ------------------------------------------------------------
# 2. Critical-field completeness
# ------------------------------------------------------------

critical_fields = [
    "Product_ID",
    "Product_Key",
    "Product_Description",
    "Product_Category",
    "Months_Present",
    "Net_Units",
    "Revenue",
    "Units_Per_Month"
]

missing_values = unit_velocity[critical_fields].isna().sum()

print("\n2. CRITICAL FIELD COMPLETENESS")
print("-" * 85)
print(missing_values)


# ------------------------------------------------------------
# 3. Velocity validity
# ------------------------------------------------------------

invalid_months = (unit_velocity["Months_Present"] <= 0).sum()
missing_velocity = unit_velocity["Units_Per_Month"].isna().sum()

print("\n3. VELOCITY VALIDITY")
print("-" * 85)
print(f"Invalid Months_Present : {invalid_months:,}")
print(f"Missing velocity       : {missing_velocity:,}")


# ------------------------------------------------------------
# 4. Unit activity reconciliation
# ------------------------------------------------------------

positive_units = unit_velocity.loc[
    unit_velocity["Net_Units"] > 0, "Net_Units"
].sum()

return_units = abs(
    unit_velocity.loc[
        unit_velocity["Net_Units"] < 0, "Net_Units"
    ].sum()
)

net_units = unit_velocity["Net_Units"].sum()

unit_reconciliation_diff = abs(
    (positive_units - return_units) - net_units
)

print("\n4. UNIT MOVEMENT RECONCILIATION")
print("-" * 85)
print(f"Gross positive units : {positive_units:,.0f}")
print(f"Return units         : {return_units:,.0f}")
print(f"Net units            : {net_units:,.0f}")
print(f"Reconciliation diff  : {unit_reconciliation_diff:,.4f}")


# ------------------------------------------------------------
# 5. Commercial segmentation reconciliation
# ------------------------------------------------------------

segment_products = commercial_segment_summary["Products"].sum()
segment_units = commercial_segment_summary["Net_Units"].sum()
segment_revenue = commercial_segment_summary["Revenue"].sum()

segment_product_diff = abs(segment_products - unique_products)
segment_unit_diff = abs(segment_units - net_units)
segment_revenue_diff = abs(
    segment_revenue - unit_velocity["Revenue"].sum()
)

print("\n5. COMMERCIAL SEGMENTATION RECONCILIATION")
print("-" * 85)
print(f"Product difference : {segment_product_diff:,}")
print(f"Net-unit difference: {segment_unit_diff:,.4f}")
print(f"Revenue difference : £{segment_revenue_diff:,.4f}")


# ------------------------------------------------------------
# 6. Final gate
# ------------------------------------------------------------

gate_results = pd.DataFrame({
    "Check": [
        "Product grain",
        "Critical-field completeness",
        "Months-present validity",
        "Velocity completeness",
        "Unit movement reconciliation",
        "Commercial segmentation population",
        "Commercial segmentation units",
        "Commercial segmentation revenue"
    ],
    "Status": [
        "PASS" if duplicate_products == 0 else "FAIL",
        "PASS" if missing_values.sum() == 0 else "FAIL",
        "PASS" if invalid_months == 0 else "FAIL",
        "PASS" if missing_velocity == 0 else "FAIL",
        "PASS" if unit_reconciliation_diff < 0.0001 else "FAIL",
        "PASS" if segment_product_diff == 0 else "FAIL",
        "PASS" if segment_unit_diff < 0.0001 else "FAIL",
        "PASS" if segment_revenue_diff < 0.0001 else "FAIL"
    ]
})

print("\nFINAL GATE RESULTS")
print("=" * 85)

display(gate_results)

overall_status = (
    "PASS"
    if (gate_results["Status"] == "PASS").all()
    else "FAIL"
)

print("\nOVERALL STATUS")
print("=" * 85)

if overall_status == "PASS":
    print(
        "PASS — unit analysis is analytically valid "
        "for downstream product-performance analysis."
    )
else:
    print(
        "FAIL — unit analysis requires remediation "
        "before downstream use."
    )

FINAL UNIT ANALYSIS INTEGRITY GATE

1. PRODUCT GRAIN
-------------------------------------------------------------------------------------
Rows                  : 763
Unique Product_IDs    : 763
Duplicate Product_IDs : 0

2. CRITICAL FIELD COMPLETENESS
-------------------------------------------------------------------------------------
Product_ID             0
Product_Key            0
Product_Description    0
Product_Category       0
Months_Present         0
Net_Units              0
Revenue                0
Units_Per_Month        0
dtype: int64

3. VELOCITY VALIDITY
-------------------------------------------------------------------------------------
Invalid Months_Present : 0
Missing velocity       : 0

4. UNIT MOVEMENT RECONCILIATION
-------------------------------------------------------------------------------------
Gross positive units : 1,125
Return units         : 10
Net units            : 1,115
Reconciliation diff  : 0.0000

5. COMMERCIAL SEGMENTATION RECONCILIATION
----------

,Check,Status
0,Product grain,PASS
1,Critical-field completeness,PASS
2,Months-present validity,PASS
3,Velocity completeness,PASS
4,Unit movement reconciliation,PASS
5,Commercial segmentation population,PASS
6,Commercial segmentation units,PASS
7,Commercial segmentation revenue,PASS



OVERALL STATUS
PASS — unit analysis is analytically valid for downstream product-performance analysis.


In [104]:
# ============================================================
# INTEGRATED PRODUCT PERFORMANCE POPULATION
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Start from governed merchandise population
# ------------------------------------------------------------

product_performance = commercial_unit_segments[
    [
        "Product_ID",
        "Product_Key",
        "Product_Description",
        "Product_Category",
        "Months_Present",
        "Net_Units",
        "Revenue",
        "Units_Per_Month",
        "Commercial_Unit_Segment"
    ]
].copy()


# ------------------------------------------------------------
# 2. Add revenue activity classification
# ------------------------------------------------------------

revenue_attributes = revenue_analysis[
    [
        "Product_ID",
        "Revenue_Activity_Type"
    ]
].copy()

product_performance = product_performance.merge(
    revenue_attributes,
    on="Product_ID",
    how="left",
    validate="one_to_one"
)


# ------------------------------------------------------------
# 3. Add profitability quality from merchandise_analysis
# ------------------------------------------------------------

profitability_quality = merchandise_analysis[
    [
        "Product_ID",
        "Profitability_Data_Quality"
    ]
].copy()

product_performance = product_performance.merge(
    profitability_quality,
    on="Product_ID",
    how="left",
    validate="one_to_one"
)


# ------------------------------------------------------------
# 4. Add validated profitability metrics
# ------------------------------------------------------------

profitability_metrics = profitability_analysis[
    [
        "Product_ID",
        "Cost_Sales",
        "Profit",
        "Gross_Margin_%"
    ]
].copy()

product_performance = product_performance.merge(
    profitability_metrics,
    on="Product_ID",
    how="left",
    validate="one_to_one"
)


# ------------------------------------------------------------
# 5. Define profitability usability
# ------------------------------------------------------------

product_performance["Profitability_Usable"] = (
    product_performance["Profitability_Data_Quality"]
    .eq("VALID")
)


# ------------------------------------------------------------
# 6. Population validation
# ------------------------------------------------------------

print("INTEGRATED PRODUCT PERFORMANCE POPULATION")
print("=" * 85)

print(
    f"Products                  : "
    f"{len(product_performance):,}"
)

print(
    f"Unique Product_IDs        : "
    f"{product_performance['Product_ID'].nunique():,}"
)

print(
    f"Duplicate Product_IDs     : "
    f"{product_performance['Product_ID'].duplicated().sum():,}"
)

print(
    f"Profitability usable      : "
    f"{product_performance['Profitability_Usable'].sum():,}"
)

print(
    f"Profitability not usable  : "
    f"{(~product_performance['Profitability_Usable']).sum():,}"
)

print(
    f"Net Units                 : "
    f"{product_performance['Net_Units'].sum():,.0f}"
)

print(
    f"Revenue                   : "
    f"£{product_performance['Revenue'].sum():,.2f}"
)


# ------------------------------------------------------------
# 7. Critical field completeness
# ------------------------------------------------------------

critical_fields = [
    "Product_ID",
    "Product_Key",
    "Product_Description",
    "Product_Category",
    "Months_Present",
    "Net_Units",
    "Revenue",
    "Units_Per_Month",
    "Commercial_Unit_Segment",
    "Revenue_Activity_Type",
    "Profitability_Data_Quality"
]

missing_critical = (
    product_performance[critical_fields]
    .isna()
    .sum()
)

print("\nCRITICAL FIELD COMPLETENESS")
print("=" * 85)

print(missing_critical)


# ------------------------------------------------------------
# 8. Expected profitability null behavior
# ------------------------------------------------------------

missing_profit = product_performance["Profit"].isna().sum()
missing_cost = product_performance["Cost_Sales"].isna().sum()
missing_margin = product_performance["Gross_Margin_%"].isna().sum()

print("\nPROFITABILITY METRIC AVAILABILITY")
print("=" * 85)

print(f"Missing Cost_Sales      : {missing_cost:,}")
print(f"Missing Profit          : {missing_profit:,}")
print(f"Missing Gross_Margin_%  : {missing_margin:,}")


# ------------------------------------------------------------
# 9. Final integration check
# ------------------------------------------------------------

integration_valid = (
    len(product_performance) == 763
    and product_performance["Product_ID"].nunique() == 763
    and product_performance["Product_ID"].duplicated().sum() == 0
    and product_performance["Profitability_Usable"].sum() == 754
    and (~product_performance["Profitability_Usable"]).sum() == 9
    and np.isclose(
        product_performance["Net_Units"].sum(),
        1115
    )
    and np.isclose(
        product_performance["Revenue"].sum(),
        281727.54,
        atol=0.01
    )
    and missing_critical.sum() == 0
)

print("\nINTEGRATION VALIDATION")
print("=" * 85)

print(f"Integrated population valid : {integration_valid}")


# ------------------------------------------------------------
# 10. Preview
# ------------------------------------------------------------

display(
    product_performance[
        [
            "Product_ID",
            "Product_Key",
            "Product_Description",
            "Product_Category",
            "Net_Units",
            "Units_Per_Month",
            "Revenue",
            "Revenue_Activity_Type",
            "Profitability_Data_Quality",
            "Cost_Sales",
            "Profit",
            "Gross_Margin_%",
            "Commercial_Unit_Segment"
        ]
    ]
    .sort_values("Revenue", ascending=False)
    .head(15)
)

INTEGRATED PRODUCT PERFORMANCE POPULATION
Products                  : 763
Unique Product_IDs        : 763
Duplicate Product_IDs     : 0
Profitability usable      : 754
Profitability not usable  : 9
Net Units                 : 1,115
Revenue                   : £281,727.54

CRITICAL FIELD COMPLETENESS
Product_ID                    0
Product_Key                   0
Product_Description           0
Product_Category              0
Months_Present                0
Net_Units                     0
Revenue                       0
Units_Per_Month               0
Commercial_Unit_Segment       0
Revenue_Activity_Type         0
Profitability_Data_Quality    0
dtype: int64

PROFITABILITY METRIC AVAILABILITY
Missing Cost_Sales      : 9
Missing Profit          : 9
Missing Gross_Margin_%  : 14

INTEGRATION VALIDATION
Integrated population valid : True


,Product_ID,Product_Key,Product_Description,Product_Category,Net_Units,Units_Per_Month,Revenue,Revenue_Activity_Type,Profitability_Data_Quality,Cost_Sales,Profit,Gross_Margin_%,Commercial_Unit_Segment
630,1167,T2351V11,Eufy Robot Vaccum X10 Pro Omni,ROBOT CLEANING,16,5.33,"6,818.30",POSITIVE_REVENUE,VALID,"6,384.32",433.98,6.36,AT_OR_ABOVE_MEDIAN_VELOCITY_HIGH_REVENUE
98,116,42120,Novy Panorama 120 Pro 5 Zone,DOWNDRAFT HOBS,1,1.00,"4,000.00",POSITIVE_REVENUE,VALID,"3,184.10",815.90,20.40,AT_OR_ABOVE_MEDIAN_VELOCITY_HIGH_REVENUE
505,870,OLED65G54L,LG 65in G5 OLED TV,TV 60 - 70,2,1.00,"3,165.00",POSITIVE_REVENUE,VALID,"3,311.22",-146.22,-4.62,AT_OR_ABOVE_MEDIAN_VELOCITY_HIGH_REVENUE
707,1320,WEK365WCS,Miele (12392780) 10kg 1400 Spin Washer,WASHING MACHINES,3,3.00,"2,832.49",POSITIVE_REVENUE,VALID,"2,392.68",439.81,15.53,AT_OR_ABOVE_MEDIAN_VELOCITY_HIGH_REVENUE
652,1213,TEH785WP,Miele 12736710 9kg Heat Pump Dryer,TUMBLE DRYERS,3,1.50,"2,764.17",POSITIVE_REVENUE,VALID,"2,563.02",201.15,7.28,AT_OR_ABOVE_MEDIAN_VELOCITY_HIGH_REVENUE
347,560,H7464BPBL,Miele Black 11093600 Pyro Single Oven,SINGLE OVENS,2,2.00,"2,683.34",POSITIVE_REVENUE,VALID,"2,582.34",101.00,3.76,AT_OR_ABOVE_MEDIAN_VELOCITY_HIGH_REVENUE
755,1425,XRFSD5265,Liebherr SXS,FRIDGE FREEZERS,1,1.00,"2,666.67",POSITIVE_REVENUE,VALID,"2,127.55",539.12,20.22,AT_OR_ABOVE_MEDIAN_VELOCITY_HIGH_REVENUE
500,856,OLED55G54L,LG 55in G5 OLED TV,TV 51 - 59,2,1.00,"2,581.67",POSITIVE_REVENUE,VALID,"2,538.66",43.01,1.67,AT_OR_ABOVE_MEDIAN_VELOCITY_HIGH_REVENUE
562,1037,RS70F64KEF,Samsung Black St/St USA FF,USA F/F,3,1.00,"2,580.83",POSITIVE_REVENUE,VALID,"2,972.94",-392.11,-15.19,AT_OR_ABOVE_MEDIAN_VELOCITY_HIGH_REVENUE
674,1246,U1ACE2AG3BNEFF,N50 Graphite Double Oven,DOUBLE OVENS,4,1.33,"2,538.33",POSITIVE_REVENUE,VALID,"2,329.68",208.65,8.22,AT_OR_ABOVE_MEDIAN_VELOCITY_HIGH_REVENUE


In [105]:
# ============================================================
#  PROFITABILITY DISTRIBUTION DIAGNOSTIC
# ============================================================

profitability_population = product_performance[
    product_performance["Profitability_Usable"]
].copy()

print("PROFITABILITY DISTRIBUTION DIAGNOSTIC")
print("=" * 85)

print(f"Profitability-valid products : {len(profitability_population):,}")
print(
    f"Revenue                     : "
    f"£{profitability_population['Revenue'].sum():,.2f}"
)
print(
    f"Cost of Sales               : "
    f"£{profitability_population['Cost_Sales'].sum():,.2f}"
)
print(
    f"Gross Profit                 : "
    f"£{profitability_population['Profit'].sum():,.2f}"
)

print("\nPRODUCT PROFIT STATUS")
print("=" * 85)

profitable = (profitability_population["Profit"] > 0).sum()
breakeven = (profitability_population["Profit"] == 0).sum()
loss_making = (profitability_population["Profit"] < 0).sum()

print(f"Profitable products   : {profitable:,}")
print(f"Break-even products   : {breakeven:,}")
print(f"Loss-making products  : {loss_making:,}")


print("\nGROSS MARGIN % DISTRIBUTION")
print("=" * 85)

margin_stats = profitability_population["Gross_Margin_%"].describe(
    percentiles=[
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95
    ]
)

print(margin_stats.round(2))


print("\nPROFIT DISTRIBUTION")
print("=" * 85)

profit_stats = profitability_population["Profit"].describe(
    percentiles=[
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95
    ]
)

print(profit_stats.round(2))


# ------------------------------------------------------------
# Reconciliation
# ------------------------------------------------------------

print("\nPROFITABILITY RECONCILIATION")
print("=" * 85)

print(
    f"Status population      : "
    f"{profitable + breakeven + loss_making:,}"
)

print(
    "Population reconciles :",
    profitable + breakeven + loss_making
    == len(profitability_population)
)

print(
    "Gross profit reconciles:",
    np.isclose(
        profitability_population["Profit"].sum(),
        5525.82,
        atol=0.01
    )
)

PROFITABILITY DISTRIBUTION DIAGNOSTIC
Profitability-valid products : 754
Revenue                     : £279,197.14
Cost of Sales               : £273,671.32
Gross Profit                 : £5,525.82

PRODUCT PROFIT STATUS
Profitable products   : 523
Break-even products   : 12
Loss-making products  : 219

GROSS MARGIN % DISTRIBUTION
count    749.00
mean       8.85
std       26.50
min     -201.82
10%      -22.63
25%       -2.90
50%       12.19
75%       23.05
90%       37.81
95%       47.65
max      100.00
Name: Gross_Margin_%, dtype: float64

PROFIT DISTRIBUTION
count      754.00
mean         7.33
std        130.88
min     -1,441.50
10%        -69.72
25%         -4.16
50%          7.25
75%         31.83
90%         97.64
95%        167.28
max        815.90
Name: Profit, dtype: float64

PROFITABILITY RECONCILIATION
Status population      : 754
Population reconciles : True
Gross profit reconciles: True


In [106]:
# ============================================================
#  GOVERNED PROFITABILITY CLASSIFICATION
# ============================================================

profitability_segment = product_performance.copy()

print("GOVERNED PROFITABILITY CLASSIFICATION")
print("=" * 85)


# ------------------------------------------------------------
# 1. Classification function
# ------------------------------------------------------------

def classify_profitability(row):

    # Profitability cannot be evaluated because cost is invalid
    if not row["Profitability_Usable"]:
        return "PROFITABILITY_NOT_USABLE"

    # Revenue = 0 means margin % is mathematically undefined
    if row["Revenue"] == 0:
        return "ZERO_REVENUE"

    # Commercial profit status
    if row["Profit"] < 0:
        return "LOSS_MAKING"

    if row["Profit"] == 0:
        return "BREAK_EVEN"

    return "PROFITABLE"


profitability_segment["Profitability_Segment"] = (
    profitability_segment.apply(
        classify_profitability,
        axis=1
    )
)


# ------------------------------------------------------------
# 2. Segment summary
# ------------------------------------------------------------

profitability_segment_summary = (
    profitability_segment
    .groupby("Profitability_Segment", as_index=False)
    .agg(
        Products=("Product_ID", "nunique"),
        Net_Units=("Net_Units", "sum"),
        Revenue=("Revenue", "sum"),
        Cost_Sales=("Cost_Sales", "sum"),
        Gross_Profit=("Profit", "sum")
    )
)


# ------------------------------------------------------------
# 3. Shares
# ------------------------------------------------------------

total_products = profitability_segment["Product_ID"].nunique()
total_revenue = profitability_segment["Revenue"].sum()

profitability_segment_summary["Product_Share_%"] = (
    profitability_segment_summary["Products"]
    / total_products
    * 100
)

profitability_segment_summary["Revenue_Share_%"] = (
    profitability_segment_summary["Revenue"]
    / total_revenue
    * 100
)


# ------------------------------------------------------------
# 4. Sort
# ------------------------------------------------------------

segment_order = {
    "PROFITABLE": 1,
    "BREAK_EVEN": 2,
    "LOSS_MAKING": 3,
    "ZERO_REVENUE": 4,
    "PROFITABILITY_NOT_USABLE": 5
}

profitability_segment_summary["Sort_Order"] = (
    profitability_segment_summary["Profitability_Segment"]
    .map(segment_order)
)

profitability_segment_summary = (
    profitability_segment_summary
    .sort_values("Sort_Order")
    .drop(columns="Sort_Order")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. Display
# ------------------------------------------------------------

display_profitability_summary = profitability_segment_summary.copy()

for col in ["Revenue", "Cost_Sales", "Gross_Profit"]:
    display_profitability_summary[col] = (
        display_profitability_summary[col]
        .map(lambda x: f"£{x:,.2f}")
    )

for col in ["Product_Share_%", "Revenue_Share_%"]:
    display_profitability_summary[col] = (
        display_profitability_summary[col]
        .map(lambda x: f"{x:.2f}%")
    )

display(display_profitability_summary)


# ------------------------------------------------------------
# 6. Validation
# ------------------------------------------------------------

classified_products = (
    profitability_segment_summary["Products"].sum()
)

print("\nCLASSIFICATION VALIDATION")
print("=" * 85)

print(
    f"Merchandise products       : "
    f"{total_products:,}"
)

print(
    f"Classified products        : "
    f"{classified_products:,}"
)

print(
    f"Unclassified products      : "
    f"{profitability_segment['Profitability_Segment'].isna().sum():,}"
)

print(
    "Population reconciles     :",
    classified_products == total_products
)

print(
    "Revenue reconciliation    :",
    np.isclose(
        profitability_segment_summary["Revenue"].sum(),
        profitability_segment["Revenue"].sum(),
        atol=0.01
    )
)

print(
    "Gross profit reconciliation:",
    np.isclose(
        profitability_segment_summary["Gross_Profit"].sum(),
        profitability_segment["Profit"].sum(),
        atol=0.01
    )
)

GOVERNED PROFITABILITY CLASSIFICATION


,Profitability_Segment,Products,Net_Units,Revenue,Cost_Sales,Gross_Profit,Product_Share_%,Revenue_Share_%
0,PROFITABLE,523,779,"£182,573.94","£156,185.43","£26,388.51",68.55%,64.81%
1,BREAK_EVEN,7,12,"£1,448.07","£1,448.07",£0.00,0.92%,0.51%
2,LOSS_MAKING,219,312,"£95,175.13","£116,037.82","£-20,862.69",28.70%,33.78%
3,ZERO_REVENUE,5,0,£0.00,£0.00,£0.00,0.66%,0.00%
4,PROFITABILITY_NOT_USABLE,9,12,"£2,530.40",£0.00,£0.00,1.18%,0.90%



CLASSIFICATION VALIDATION
Merchandise products       : 763
Classified products        : 763
Unclassified products      : 0
Population reconciles     : True
Revenue reconciliation    : True
Gross profit reconciliation: True


In [107]:
# ============================================================
#  INTEGRATED COMMERCIAL PERFORMANCE SEGMENTATION
# ============================================================

commercial_performance = profitability_segment.copy()

print("INTEGRATED COMMERCIAL PERFORMANCE SEGMENTATION")
print("=" * 90)


# ------------------------------------------------------------
# 1. Validate required columns
# ------------------------------------------------------------

required_columns = [
    "Product_ID",
    "Product_Key",
    "Product_Description",
    "Product_Category",
    "Net_Units",
    "Revenue",
    "Units_Per_Month",
    "Commercial_Unit_Segment",
    "Revenue_Activity_Type",
    "Profitability_Usable",
    "Profitability_Segment"
]

missing_columns = [
    col for col in required_columns
    if col not in commercial_performance.columns
]

print("Missing required columns :", missing_columns)

assert len(missing_columns) == 0, (
    f"Missing required columns: {missing_columns}"
)


# ------------------------------------------------------------
# 2. Integrated commercial classification
# ------------------------------------------------------------

def classify_integrated_performance(row):

    profit_segment = row["Profitability_Segment"]
    unit_segment = row["Commercial_Unit_Segment"]

    # --------------------------------------------------------
    # Data-quality / non-commercial populations
    # --------------------------------------------------------

    if profit_segment == "PROFITABILITY_NOT_USABLE":
        return "DATA_QUALITY_REVIEW"

    if profit_segment == "ZERO_REVENUE":
        return "ZERO_REVENUE_ACTIVITY"

    if unit_segment == "RETURN_ACTIVITY":
        return "RETURN_ACTIVITY"

    # --------------------------------------------------------
    # Loss-making products
    # --------------------------------------------------------

    if profit_segment == "LOSS_MAKING":

        if unit_segment == "AT_OR_ABOVE_MEDIAN_VELOCITY_HIGH_REVENUE":
            return "HIGH_ACTIVITY_HIGH_REVENUE_LOSS"

        if unit_segment == "AT_OR_ABOVE_MEDIAN_VELOCITY_LOW_REVENUE":
            return "HIGH_ACTIVITY_LOW_REVENUE_LOSS"

        if unit_segment == "BELOW_MEDIAN_VELOCITY_HIGH_REVENUE":
            return "LOW_ACTIVITY_HIGH_REVENUE_LOSS"

        if unit_segment == "BELOW_MEDIAN_VELOCITY_LOW_REVENUE":
            return "LOW_ACTIVITY_LOW_REVENUE_LOSS"

        return "OTHER_LOSS_MAKING"

    # --------------------------------------------------------
    # Break-even products
    # --------------------------------------------------------

    if profit_segment == "BREAK_EVEN":
        return "BREAK_EVEN"

    # --------------------------------------------------------
    # Profitable products
    # --------------------------------------------------------

    if profit_segment == "PROFITABLE":

        if unit_segment == "AT_OR_ABOVE_MEDIAN_VELOCITY_HIGH_REVENUE":
            return "CORE_PROFIT_DRIVER"

        if unit_segment == "AT_OR_ABOVE_MEDIAN_VELOCITY_LOW_REVENUE":
            return "VOLUME_PROFITABLE"

        if unit_segment == "BELOW_MEDIAN_VELOCITY_HIGH_REVENUE":
            return "VALUE_PROFITABLE"

        if unit_segment == "BELOW_MEDIAN_VELOCITY_LOW_REVENUE":
            return "LOW_ACTIVITY_PROFITABLE"

        return "OTHER_PROFITABLE"

    return "UNCLASSIFIED"


commercial_performance["Commercial_Performance_Segment"] = (
    commercial_performance.apply(
        classify_integrated_performance,
        axis=1
    )
)


# ------------------------------------------------------------
# 3. Segment summary
# ------------------------------------------------------------

commercial_segment_summary = (
    commercial_performance
    .groupby("Commercial_Performance_Segment", as_index=False)
    .agg(
        Products=("Product_ID", "nunique"),
        Net_Units=("Net_Units", "sum"),
        Revenue=("Revenue", "sum"),
        Gross_Profit=("Profit", "sum")
    )
)


# ------------------------------------------------------------
# 4. Contribution measures
# ------------------------------------------------------------

total_products = commercial_performance["Product_ID"].nunique()
total_revenue = commercial_performance["Revenue"].sum()

commercial_segment_summary["Product_Share_%"] = (
    commercial_segment_summary["Products"]
    / total_products
    * 100
)

commercial_segment_summary["Revenue_Share_%"] = (
    commercial_segment_summary["Revenue"]
    / total_revenue
    * 100
)


# ------------------------------------------------------------
# 5. Sort by commercial importance
# ------------------------------------------------------------

commercial_segment_summary = (
    commercial_segment_summary
    .sort_values(
        ["Gross_Profit", "Revenue"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. Display formatted summary
# ------------------------------------------------------------

display_commercial_summary = commercial_segment_summary.copy()

for col in ["Revenue", "Gross_Profit"]:
    display_commercial_summary[col] = (
        display_commercial_summary[col]
        .map(lambda x: f"£{x:,.2f}")
    )

for col in ["Product_Share_%", "Revenue_Share_%"]:
    display_commercial_summary[col] = (
        display_commercial_summary[col]
        .map(lambda x: f"{x:.2f}%")
    )

display(display_commercial_summary)


# ------------------------------------------------------------
# 7. Validation
# ------------------------------------------------------------

classified_products = commercial_segment_summary["Products"].sum()

unclassified_products = (
    commercial_performance[
        "Commercial_Performance_Segment"
    ]
    .eq("UNCLASSIFIED")
    .sum()
)

print("\nINTEGRATION VALIDATION")
print("=" * 90)

print(f"Merchandise products     : {total_products:,}")
print(f"Classified products      : {classified_products:,}")
print(f"Unclassified products    : {unclassified_products:,}")

print(
    "Population reconciles   :",
    classified_products == total_products
)

print(
    "Net-unit reconciliation :",
    np.isclose(
        commercial_segment_summary["Net_Units"].sum(),
        commercial_performance["Net_Units"].sum()
    )
)

print(
    "Revenue reconciliation  :",
    np.isclose(
        commercial_segment_summary["Revenue"].sum(),
        commercial_performance["Revenue"].sum(),
        atol=0.01
    )
)

print(
    "Profit reconciliation   :",
    np.isclose(
        commercial_segment_summary["Gross_Profit"].sum(),
        commercial_performance["Profit"].sum(),
        atol=0.01
    )
)

INTEGRATED COMMERCIAL PERFORMANCE SEGMENTATION
Missing required columns : []


,Commercial_Performance_Segment,Products,Net_Units,Revenue,Gross_Profit,Product_Share_%,Revenue_Share_%
0,CORE_PROFIT_DRIVER,233,366,"£167,055.67","£22,874.85",30.54%,59.30%
1,VOLUME_PROFITABLE,283,411,"£15,249.94","£3,403.42",37.09%,5.41%
2,OTHER_PROFITABLE,3,0,£58.35,£58.35,0.39%,0.02%
3,VALUE_PROFITABLE,1,1,£283.33,£30.66,0.13%,0.10%
4,LOW_ACTIVITY_PROFITABLE,2,2,£25.82,£6.89,0.26%,0.01%
5,DATA_QUALITY_REVIEW,9,12,"£2,530.40",£0.00,1.18%,0.90%
6,BREAK_EVEN,7,12,"£1,448.07",£0.00,0.92%,0.51%
7,ZERO_REVENUE_ACTIVITY,5,0,£0.00,£0.00,0.66%,0.00%
8,OTHER_LOSS_MAKING,1,0,£-60.00,£-60.00,0.13%,-0.02%
9,LOW_ACTIVITY_LOW_REVENUE_LOSS,2,2,£124.97,£-77.16,0.26%,0.04%



INTEGRATION VALIDATION
Merchandise products     : 763
Classified products      : 763
Unclassified products    : 0
Population reconciles   : True
Net-unit reconciliation : True
Revenue reconciliation  : True
Profit reconciliation   : True


In [108]:
# ============================================================
# OTHER SEGMENT DIAGNOSTIC
# ============================================================

other_segment_products = commercial_performance[
    commercial_performance[
        "Commercial_Performance_Segment"
    ].isin([
        "OTHER_PROFITABLE",
        "OTHER_LOSS_MAKING"
    ])
].copy()

diagnostic_columns = [
    "Product_ID",
    "Product_Key",
    "Product_Description",
    "Product_Category",
    "Months_Present",
    "Net_Units",
    "Revenue",
    "Cost_Sales",
    "Profit",
    "Gross_Margin_%",
    "Units_Per_Month",
    "Revenue_Activity_Type",
    "Commercial_Unit_Segment",
    "Profitability_Segment",
    "Commercial_Performance_Segment"
]

print("OTHER COMMERCIAL SEGMENT DIAGNOSTIC")
print("=" * 90)

print(f"Products requiring diagnostic : {len(other_segment_products):,}")

display(
    other_segment_products[
        diagnostic_columns
    ].sort_values(
        ["Commercial_Performance_Segment", "Revenue"],
        ascending=[True, False]
    )
)

OTHER COMMERCIAL SEGMENT DIAGNOSTIC
Products requiring diagnostic : 4


,Product_ID,Product_Key,Product_Description,Product_Category,Months_Present,Net_Units,Revenue,Cost_Sales,Profit,Gross_Margin_%,Units_Per_Month,Revenue_Activity_Type,Commercial_Unit_Segment,Profitability_Segment,Commercial_Performance_Segment
714,1339,WGG254Z0GB*BOSCH,Series 6 10kg 1400 Spin,WASHING MACHINES,2,0,-60.00,0.00,-60.00,100.00,0.00,REVERSAL_VALUE_VARIANCE,ZERO_UNIT_ACTIVITY,LOSS_MAKING,OTHER_LOSS_MAKING
352,571,HD440BPUK,SHARK FLEX STYLE – MALIBU,HAIRCARE,1,0,33.34,0.00,33.34,100.00,0.00,POSITIVE_REVENUE,ZERO_UNIT_ACTIVITY,PROFITABLE,OTHER_PROFITABLE
379,616,INTEGRATA6,Elica 5414601 60cm Integrata,COOKER HOODS,2,0,25.00,0.00,25.00,100.00,0.00,POSITIVE_REVENUE,ZERO_UNIT_ACTIVITY,PROFITABLE,OTHER_PROFITABLE
624,1161,SV1240,One For All Total Control Amplified,TV ACCESSORIES,1,0,0.01,0.00,0.01,100.00,0.00,POSITIVE_REVENUE,ZERO_UNIT_ACTIVITY,PROFITABLE,OTHER_PROFITABLE


In [109]:
# ============================================================
#  RESOLVE VALUE-ONLY COMMERCIAL ACTIVITY
# ============================================================

# Zero units + negative revenue
mask_negative_value_only = (
    (commercial_performance["Net_Units"] == 0) &
    (commercial_performance["Revenue"] < 0)
)

commercial_performance.loc[
    mask_negative_value_only,
    "Commercial_Performance_Segment"
] = "REVERSAL_VALUE_VARIANCE"


# Zero units + positive revenue
mask_positive_value_only = (
    (commercial_performance["Net_Units"] == 0) &
    (commercial_performance["Revenue"] > 0)
)

commercial_performance.loc[
    mask_positive_value_only,
    "Commercial_Performance_Segment"
] = "POSITIVE_VALUE_ONLY_ACTIVITY"


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("VALUE-ONLY ACTIVITY RESOLUTION")
print("=" * 80)

print(
    "Reversal value variance      :",
    mask_negative_value_only.sum()
)

print(
    "Positive value-only activity :",
    mask_positive_value_only.sum()
)

print(
    "Remaining OTHER_PROFITABLE   :",
    (
        commercial_performance["Commercial_Performance_Segment"]
        == "OTHER_PROFITABLE"
    ).sum()
)

print(
    "Remaining OTHER_LOSS_MAKING  :",
    (
        commercial_performance["Commercial_Performance_Segment"]
        == "OTHER_LOSS_MAKING"
    ).sum()
)

print(
    "Unclassified products        :",
    commercial_performance[
        "Commercial_Performance_Segment"
    ].isna().sum()
)

VALUE-ONLY ACTIVITY RESOLUTION
Reversal value variance      : 1
Positive value-only activity : 3
Remaining OTHER_PROFITABLE   : 0
Remaining OTHER_LOSS_MAKING  : 0
Unclassified products        : 0


In [111]:
# ============================================================
# 4.10C.6 — FINAL INTEGRATED COMMERCIAL PERFORMANCE GATE
# ============================================================

print("FINAL INTEGRATED COMMERCIAL PERFORMANCE GATE")
print("=" * 80)

# ------------------------------------------------------------
# 1. Population integrity
# ------------------------------------------------------------

total_rows = len(commercial_performance)
unique_products = commercial_performance["Product_ID"].nunique()
duplicate_products = commercial_performance["Product_ID"].duplicated().sum()

# ------------------------------------------------------------
# 2. Classification completeness
# ------------------------------------------------------------

classified_products = (
    commercial_performance["Commercial_Performance_Segment"]
    .notna()
    .sum()
)

unclassified_products = (
    commercial_performance["Commercial_Performance_Segment"]
    .isna()
    .sum()
)

remaining_other = (
    commercial_performance["Commercial_Performance_Segment"]
    .isin(["OTHER_PROFITABLE", "OTHER_LOSS_MAKING"])
    .sum()
)

# ------------------------------------------------------------
# 3. Financial totals
# ------------------------------------------------------------

net_units = commercial_performance["Net_Units"].sum()
revenue = commercial_performance["Revenue"].sum()
gross_profit = commercial_performance["Profit"].sum()

# ------------------------------------------------------------
# 4. Critical field completeness
# ------------------------------------------------------------

required_columns = [
    "Product_ID",
    "Product_Key",
    "Product_Description",
    "Product_Category",
    "Net_Units",
    "Revenue",
    "Commercial_Performance_Segment"
]

missing_required_columns = [
    col for col in required_columns
    if col not in commercial_performance.columns
]

if not missing_required_columns:
    missing_required_values = (
        commercial_performance[required_columns]
        .isna()
        .sum()
        .sum()
    )
else:
    missing_required_values = None

# ------------------------------------------------------------
# 5. Display controls
# ------------------------------------------------------------

print(f"Products                    : {total_rows:,}")
print(f"Unique Product_IDs          : {unique_products:,}")
print(f"Duplicate Product_IDs       : {duplicate_products:,}")
print(f"Classified products         : {classified_products:,}")
print(f"Unclassified products       : {unclassified_products:,}")
print(f"Remaining OTHER segments    : {remaining_other:,}")

print()

print(f"Net units                   : {net_units:,.0f}")
print(f"Revenue                     : £{revenue:,.2f}")
print(f"Gross profit                : £{gross_profit:,.2f}")

print()

print(f"Missing required columns    : {missing_required_columns}")
print(f"Missing required values     : {missing_required_values}")

# ------------------------------------------------------------
# 6. Gate conditions
# ------------------------------------------------------------

population_pass = (
    total_rows == 763
    and unique_products == 763
    and duplicate_products == 0
)

classification_pass = (
    classified_products == 763
    and unclassified_products == 0
    and remaining_other == 0
)

schema_pass = (
    len(missing_required_columns) == 0
    and missing_required_values == 0
)

# ------------------------------------------------------------
# 7. Final gate
# ------------------------------------------------------------

final_gate_pass = (
    population_pass
    and classification_pass
    and schema_pass
)

print("\nGATE CONDITIONS")
print("=" * 80)

print("Population integrity       :", population_pass)
print("Classification completeness:", classification_pass)
print("Schema/data completeness   :", schema_pass)

print("\nFINAL COMMERCIAL GATE")
print("=" * 80)

print("STATUS:", "PASS" if final_gate_pass else "FAIL")

FINAL INTEGRATED COMMERCIAL PERFORMANCE GATE
Products                    : 763
Unique Product_IDs          : 763
Duplicate Product_IDs       : 0
Classified products         : 763
Unclassified products       : 0
Remaining OTHER segments    : 0

Net units                   : 1,115
Revenue                     : £281,727.54
Gross profit                : £5,525.82

Missing required columns    : []
Missing required values     : 0

GATE CONDITIONS
Population integrity       : True
Classification completeness: True
Schema/data completeness   : True

FINAL COMMERCIAL GATE
STATUS: PASS


# Phase 3 — Sales Analysis: Final Key Findings & Analytical Conclusion

## 1. Governed Sales Population

The Sales Analysis was completed using a governed merchandise population with validated product-level grain.

| KPI | Validated Result |
|---|---:|
| Merchandise Products | 763 |
| Unique Product IDs | 763 |
| Net Units | 1,115 |
| Net Revenue | £281,727.54 |
| Duplicate Product IDs | 0 |
| Unclassified Products | 0 |

**Key Finding:**  
The product-level sales population is internally consistent and sufficiently governed for downstream commercial, inventory, SOA, and cross-functional analysis.

---

## 2. Revenue Performance

Revenue activity was separated into normal positive sales, returns/reversals, zero-revenue activity, and reversal-value variance.

| Revenue Activity | Products | Net Units | Revenue |
|---|---:|---:|---:|
| Positive Revenue | 751 | 1,125 | £283,369.15 |
| Net Return / Reversal | 6 | -10 | -£1,581.61 |
| Reversal Value Variance | 1 | 0 | -£60.00 |
| Zero Revenue | 5 | 0 | £0.00 |
| **Net Merchandise Position** | **763** | **1,115** | **£281,727.54** |

**Key Finding:**  
The merchandise portfolio is overwhelmingly revenue-generating, while returns/reversals and value variances reduce gross positive revenue to the final governed net revenue of **£281,727.54**.

---

## 3. Return & Reversal Impact

Gross positive unit movement was **1,125 units**, while returns/reversals accounted for **10 units**.

**Unit Reconciliation:**

`1,125 Gross Positive Units - 10 Return Units = 1,115 Net Units`

- Return Rate: **0.89%**
- Net Realization Rate: **99.11%**

**Key Finding:**  
Returns currently have a relatively small impact on overall unit movement. Approximately **99.1% of gross positive unit movement remains after returns/reversals**.

Returns and reversals are retained as legitimate commercial activity and are not treated as ordinary product underperformance.

---

## 4. Product Unit Activity

The merchandise population was classified according to net unit movement.

- Positive-unit products: **748**
- Zero-unit products: **9**
- Negative-unit / return products: **6**

**Key Finding:**  
Approximately **98% of merchandise products recorded positive net unit movement** during the observed sales period.

Therefore, the principal commercial issue is not simply whether products sell, but **how efficiently, consistently, and profitably they sell**.

---

## 5. Product Sales Velocity

Product velocity was normalized using:

`Units Per Month = Net Units / Months Present`

Observed velocity statistics:

| Statistic | Units / Month |
|---|---:|
| Mean | 1.15 |
| Median | 1.00 |
| Maximum | 5.33 |
| Minimum | -5.00 |

For positive-unit products:

| Percentile | Units / Month |
|---|---:|
| 25th | 1.00 |
| 50th | 1.00 |
| 75th | 1.00 |
| 90th | 2.00 |
| 95th | 2.00 |

**Key Finding:**  
Product movement is generally low and highly concentrated around **1 unit per month**.

Because the 25th, 50th, and 75th percentiles are all approximately 1 unit/month, conventional quartile-based velocity segmentation was not considered appropriate.

---

## 6. Product Velocity Segmentation

A distribution-aware velocity classification was created.

| Velocity Class | Products | Net Units | Revenue |
|---|---:|---:|---:|
| Low Velocity | 619 | 723 | £209,642.74 |
| Medium Velocity | 101 | 263 | £52,108.64 |
| High Velocity | 28 | 139 | £21,559.42 |
| Zero Activity | 9 | 0 | -£1.65 |
| Return Activity | 6 | -10 | -£1,581.61 |

**Key Finding:**  
The merchandise portfolio is strongly weighted toward lower-velocity products.

**619 products (81.13% of the assortment)** are classified as low velocity, but collectively they still generate **723 net units and £209.64k revenue**.

Therefore, low velocity should **not automatically be interpreted as poor product performance**.

---

## 7. Unit Demand Concentration

Gross positive unit demand is relatively broadly distributed across products.

Products required to generate gross positive unit movement:

- 50% of units → **204 products**
- 80% of units → **524 products**
- 90% of units → **636 products**

Top-product concentration:

- Top 10 products → **6.84%**
- Top 20 products → **11.02%**
- Top 50 products → **19.73%**
- Top 100 products → **31.56%**

**Key Finding:**  
The assortment does **not follow a classic 80/20 unit-sales pattern**.

Approximately **70% of positive-selling products are required to generate 80% of gross unit movement**, indicating a broad, long-tail demand structure.

This has an important inventory implication: stocking decisions cannot be based only on a small group of best-selling SKUs.

---

## 8. Revenue Concentration

Revenue is more concentrated than unit movement, but still does not follow a strict 80/20 pattern.

- 50% of revenue → **89 products (11.66%)**
- 80% of revenue → **236 products (30.93%)**
- 90% of revenue → **336 products (44.04%)**

Top-product revenue concentration:

- Top 10 products → **11.58%**
- Top 20 products → increasing but still diversified
- Top 100 products → **53.36%**

**Key Finding:**  
Revenue is concentrated within a smaller portion of the assortment than unit demand, but the business is not dependent on only a handful of individual SKUs.

---

## 9. Revenue Distribution

Revenue performance is strongly right-skewed.

- Mean revenue per product: **£369.24**
- Median revenue per product: **£165.00**

Approximately **574 products (75.23%)** generate less than £500 each, while only **76 products (9.96%)** generate £1,000 or more.

Those 76 products contribute approximately **46.22% of total revenue**.

**Key Finding:**  
A relatively small group of higher-value products materially increases overall revenue, while a large long-tail product population generates smaller individual revenue contributions.

---

## 10. Category-Level Sales Performance

The governed merchandise population spans **85 product categories**.

Leading categories by net unit movement included:

| Category | Net Units |
|---|---:|
| Food Prep | 68 |
| Washing Machines | 50 |
| IT Accessories | 47 |
| Kettles | 44 |
| Cables | 41 |
| Stick Vacs | 37 |
| Accessories | 37 |
| TV 51–59 | 33 |
| Single Ovens | 29 |
| Fryers | 29 |
| Tumble Dryers | 28 |
| Coffee Makers | 28 |
| TV 33–43 | 26 |

**Key Finding:**  
Unit demand is distributed across both lower-value accessories/small appliances and higher-value major electrical categories.

Consequently, **unit volume alone is not an adequate measure of category commercial importance**.

---

## 11. Velocity × Revenue Performance

Positive commercial products were evaluated using data-derived median thresholds:

- Velocity threshold: **1.00 unit/month**
- Revenue threshold: **£165.83**

The largest commercially significant segment contained:

**379 products at/above median velocity with high revenue**

These products generated:

- **601 net units**
- **£261,459.40 revenue**
- approximately **92.81% of total merchandise revenue**

A further **364 products** were at/above median velocity but generated comparatively low revenue.

**Key Finding:**  
Products with similar unit movement can have dramatically different commercial values.

Therefore, **unit velocity should not be used independently when prioritising products**.

---

## 12. Profitability Population

Profitability analysis was restricted to products with valid cost information.

| KPI | Result |
|---|---:|
| Profitability-Valid Products | 754 |
| Profitability-Unusable Products | 9 |
| Profitability-Valid Revenue | £279,197.14 |
| Cost of Sales | £273,671.32 |
| Gross Profit | £5,525.82 |

The 9 products with unusable profitability information were retained in the merchandise population but excluded from profitability-based judgement.

**Key Finding:**  
Revenue and unit KPIs can use the complete merchandise population, while profitability KPIs require a separately governed population because missing/invalid cost information would otherwise distort margin analysis.

---

## 13. Product Profitability

Within the profitability-valid population:

- Profitable products: **523**
- Loss-making products: **219**
- Remaining products represent break-even / zero-revenue activity under the governed classification.

Profitable products generated approximately:

**+£26,388.51 Gross Profit**

Loss-making products generated approximately:

**-£20,862.69 Gross Profit**

Resulting net gross profit:

**£5,525.82**

**Key Finding:**  
Profit erosion is one of the most significant commercial findings from the Sales Analysis.

A substantial amount of positive gross profit generated by profitable products is being offset by loss-making merchandise.

---

## 14. Gross Margin Distribution

Gross-margin performance varies substantially across the product portfolio.

| Statistic | Gross Margin % |
|---|---:|
| Mean | 8.85% |
| Median | 12.19% |
| 25th Percentile | -2.90% |
| 10th Percentile | -22.63% |
| 75th Percentile | 23.05% |
| 90th Percentile | 37.81% |
| 95th Percentile | 47.65% |

**Key Finding:**  
The product portfolio contains substantial variation in margin performance.

A meaningful portion of products operate at negative gross margins, while stronger products generate considerably higher positive margins.

Aggregate revenue therefore masks major differences in underlying SKU economics.

---

## 15. Core Profit Drivers

Integrated analysis combining unit movement, revenue, and profitability identified a commercially important group of:

### Core Profit Drivers

- Products: **233**
- Net Units: **366**
- Revenue: **£167,055.67**
- Gross Profit: **£22,874.85**
- Product Share: **30.54%**
- Revenue Share: **59.30%**

**Key Finding:**  
Approximately **30.5% of merchandise products generate 59.3% of total revenue while producing £22.87k of positive gross profit**.

These products represent the strongest combination of sales activity, commercial value, and profitability in the current portfolio.

---

## 16. High-Activity Loss-Making Products

The integrated commercial analysis also identified a significant risk group:

### High Activity + High Revenue + Loss

- Products: **140**
- Net Units: **226**
- Revenue: **£90,998.68**
- Gross Profit: **-£19,115.38**

A further high-activity/low-revenue loss-making group contained:

- Products: **71**
- Net Units: **93**
- Revenue: **£5,593.92**
- Gross Profit: **-£1,417.40**

**Key Finding:**  
Some of the strongest-selling products are simultaneously destroying gross profit.

The **140 high-activity/high-revenue loss-making products generate approximately £91k in revenue while producing approximately £19.1k in gross losses**.

These products should become priority candidates for future investigation involving:

- Pricing
- Product cost
- Promotional discounting
- Sell-Out Allowances (SOA)
- Supplier support
- Margin recovery

---

## 17. Special Transaction Activity

The analysis identified legitimate commercial transactions where net units and financial values did not behave like ordinary product sales.

These were separately governed as:

- `NET_RETURN_REVERSAL`
- `REVERSAL_VALUE_VARIANCE`
- `POSITIVE_VALUE_ONLY_ACTIVITY`
- `ZERO_REVENUE`
- `RETURN_ACTIVITY`

One investigated example contained offsetting +1/-1 unit transactions whose cost fully reversed but whose sales values differed, leaving a **-£60 residual revenue variance**.

**Key Finding:**  
Returns, reversals, and value-only transactions should not be automatically interpreted as poor-performing products or data errors.

Explicit transaction governance prevents these activities from contaminating ordinary sales and profitability analysis.

---

# Overall Sales Analysis Conclusion

The Sales Analysis demonstrates that the primary commercial challenge is **not simply lack of product sales**.

Approximately **98% of merchandise products recorded positive unit movement**, and customer demand is distributed broadly across the product assortment.

However, three major structural findings emerge:

### 1. The assortment is dominated by relatively low-velocity products

More than **81% of products are classified as low velocity**, although these products collectively remain commercially important.

### 2. Unit movement and commercial value are not equivalent

Products with similar sales velocities can generate dramatically different revenue contributions. Revenue therefore needs to be evaluated alongside unit movement.

### 3. Profitability is the major commercial concern

The most significant finding is the presence of a substantial loss-making product population.

While profitable products generate approximately **£26.39k positive gross profit**, loss-making products remove approximately **£20.86k**, leaving only approximately **£5.53k net gross profit** across the profitability-valid population.

Particularly important are the **140 high-activity/high-revenue loss-making products**, which generate approximately **£91k revenue while producing approximately £19.1k in gross losses**.

Therefore, the key business question is no longer simply:

> **Which products are selling?**

The next analytical question becomes:

> **Are current inventory levels aligned with product demand, sales velocity, revenue contribution, profitability, and commercial importance?**

This provides the analytical bridge from the **Sales Analysis** into the **Inventory / Stock Analysis**.

---

# Final Validation Status

The Sales Analysis completed all required governance and reconciliation controls.

- Revenue Analysis Gate: **PASS**
- Unit Performance Gate: **PASS**
- Profitability Reconciliation: **PASS**
- Integrated Commercial Population: **PASS**
- Classification Completeness: **PASS**
- Schema / Data Completeness: **PASS**
- Final Commercial Gate: **PASS**

### Final Status

**SALES ANALYSIS — COMPLETE & VALIDATED**

The governed sales and commercial-performance layers are ready for downstream **Inventory / Stock Analysis, SOA Analysis, Cross-Functional Analysis, and subsequent forecasting / AI decision-support stages**.